In [1]:
# ============================================================
# CELDA 0 — PANEL DE CONTROL (único lugar que se edita)
# ============================================================
from pathlib import Path

VERSION      = 'V8'                      
PROJECT_ROOT = Path(r'C:\Users\rodri\Documents\PINN\b2_proyecto')

# Etiquetas descriptivas (solo informativas, para títulos/JSON)
DATASET_LABEL = 'v6 nuevo B4-INF'
ARCH_LABEL    = 'PINNModal_v4b'
LOSS_LABEL    = 'compute_loss_v16'
CURR_LABEL    = 'curriculum v20'

# ── Rutas derivadas automáticamente de VERSION ───────────────
VTAG          = VERSION.lower()                      
DATA_DIR      = PROJECT_ROOT / 'dataset' / 'opensees'
TRIALS_ROOT   = PROJECT_ROOT / 'models' / 'trials_31'
TRIALS_DIR    = TRIALS_ROOT / VERSION                 
SCATTER_DIR   = TRIALS_DIR / 'scatter_plots'
DEFINITIVO_DIR = TRIALS_DIR / 'DEFINITIVO'

# Alias de compatibilidad: las celdas que aún usan TRIALS_DIR_V8 siguen funcionando
TRIALS_DIR_VER = TRIALS_DIR
TRIALS_DIR_V8  = TRIALS_DIR    # alias retro-compatible → apunta a la VERSION activa

for _d in (TRIALS_ROOT, TRIALS_DIR, SCATTER_DIR, DEFINITIVO_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# Helper para nombres de archivo con sufijo de versión:
#   vfile('all_results', 'json')          -> 'all_results_v8.json'
#   vfile('dashboard_31_trials', 'png')   -> 'dashboard_31_trials_v8.png'
def vfile(stem, ext):
    return f'{stem}_{VTAG}.{ext}'

# Rutas de archivos de resultados (derivadas de VERSION)
RESULTS_JSON = TRIALS_DIR / vfile('all_results', 'json')

print('=' * 60)
print('PANEL DE CONTROL NB5')
print('=' * 60)
print(f'  VERSION        : {VERSION}')
print(f'  PROJECT_ROOT   : {PROJECT_ROOT}')
print(f'  TRIALS_DIR     : {TRIALS_DIR}')
print(f'  SCATTER_DIR    : {SCATTER_DIR}')
print(f'  DEFINITIVO_DIR : {DEFINITIVO_DIR}')
print(f'  RESULTS_JSON   : {RESULTS_JSON.name}')
print(f'  DATASET        : {DATASET_LABEL}')
print(f'  ARQUITECTURA   : {ARCH_LABEL}')
print(f'  LOSS           : {LOSS_LABEL}')
print(f'  CURRICULUM     : {CURR_LABEL}')
print('=' * 60)

PANEL DE CONTROL NB5
  VERSION        : V8
  PROJECT_ROOT   : C:\Users\rodri\Documents\PINN\b2_proyecto
  TRIALS_DIR     : C:\Users\rodri\Documents\PINN\b2_proyecto\models\trials_31\V8
  SCATTER_DIR    : C:\Users\rodri\Documents\PINN\b2_proyecto\models\trials_31\V8\scatter_plots
  DEFINITIVO_DIR : C:\Users\rodri\Documents\PINN\b2_proyecto\models\trials_31\V8\DEFINITIVO
  RESULTS_JSON   : all_results_v8.json
  DATASET        : v6 nuevo B4-INF
  ARQUITECTURA   : PINNModal_v4b
  LOSS           : compute_loss_v16
  CURRICULUM     : curriculum v20


## 📌 Celda 1 — Imports, dispositivo de cómputo y configuración del experimento

### 🎯 ¿Qué hace esta celda?

Es la **celda de inicialización del notebook**. Prepara todo lo necesario para que el resto del código funcione:

1. **Importa librerías** científicas y de deep learning:
   - `numpy`, `pandas`, `matplotlib` → manejo de datos numéricos y gráficos.
   - `h5py` → lectura del dataset `Familia B` almacenado en formato HDF5.
   - `torch`, `torch.nn`, `torch.nn.functional` → construcción de la **PINN** (Physics-Informed Neural Network).
   - `sklearn` → métricas auxiliares ($R^2$, MAE) y split de datos.

2. **Fija una semilla** $\text{SEED}=2026$ para reproducibilidad estricta:
   $$\text{torch.manual\_seed}(\text{SEED}), \quad \text{np.random.seed}(\text{SEED})$$

3. **Detecta GPU vs CPU** (CUDA si está disponible).

4. **Define rutas** del proyecto y crea el directorio `TRIALS_DIR_V5/` donde se guardarán los resultados de los 31 ensayos.

5. **Define la metodología experimental** — *Repeated Hold-Out con Diseño Factorial 2×2* (Arlot & Celisse, 2010):

| Símbolo | Valor | Significado |
|---|---|---|
| $N_{\text{train}}$ | 1000 | Casos por repetición |
| $N_{\text{val}}^{\text{OOD}}$ | 200 | Validación fuera de distribución |
| $N_{\text{test}}^{\text{OOD}}$ | 200 | Test fuera de distribución |
| $K_{\text{reps}}$ | 4 | Repeticiones por trial |

### 🧪 Diseño Factorial 2×2 — esquema

```
                  ┌─── Pool A (TRAIN) ───┐    ┌──── Pool B (OOD) ────┐
                  │                       │    │                       │
Dominio físico:   │   A1 (1000)   A2 (1000)│    │  B1..B8 (200 c/u)   │
                  └───────────────────────┘    └───────────────────────┘
                                                       │
            ┌──────────────────┬──────────────────┬────┴─────────────┬───────────────┐
            │                  │                  │                  │
        Rep 1               Rep 2              Rep 3              Rep 4
     Train=A1            Train=A1           Train=A2           Train=A2
     Val=B1, Test=B2     Val=B3, Test=B4    Val=B5, Test=B6    Val=B7, Test=B8
            │__________________│__________________│__________________│
                 σ_OOD                 σ_TRAIN              σ_OOD
            (Reps 1-2)            (Reps 1-3)            (Reps 3-4)
```

- **Reps 1 vs 2** → mismo TRAIN, distinto OOD → mide **variabilidad por dominio OOD** ($\sigma_{\text{OOD}}$).
- **Reps 1 vs 3** → distinto TRAIN → mide **variabilidad por selección del entrenamiento** ($\sigma_{\text{train}}$).

### ✅ Lo que se obtiene

- `device`: `cuda` o `cpu` configurado.
- Rutas `PROJECT_ROOT`, `DATA_DIR`, `TRIALS_DIR_V5`.
- Path al HDF5 más reciente del dataset `dataset_familyB_OPENSEES_*.h5`.
- Constantes globales: `N_MAX_PISOS = 18`, `N_MODOS = 18`, `BATCH_SIZE = 64`.
- Verificación de **factibilidad** del pool: Pool A ≥ 2000, Pool B ≥ 1600 casos.

In [2]:
# ============================================================
# CELDA 1 -- IMPORTS + DEVICE + RUTAS
# ============================================================
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import h5py, json, math, shutil
from pathlib import Path
from datetime import datetime
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

SEED = 2026
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# Rutas: definidas en la CELDA 0 (panel de control).
# Disponibles aquí: PROJECT_ROOT, DATA_DIR, TRIALS_DIR, TRIALS_DIR_V8 (alias),
#                   SCATTER_DIR, DEFINITIVO_DIR, RESULTS_JSON, vfile().
print(f'TRIALS_DIR ({VERSION}): {TRIALS_DIR}')

# ── HDF5 — siempre el mas reciente ───────────────────────────────────────
H5_MODAL = sorted(DATA_DIR.glob('dataset_familyB_OPENSEES_*.h5'),
                  key=lambda p: p.stat().st_mtime)[-1]
print(f'HDF5: {H5_MODAL.name}')

# ── Arquitectura ──────────────────────────────────────────────────────────
N_MAX_PISOS  = 18
N_MODOS      = 18
BATCH_SIZE   = 64

# ── Metodología Repeated Hold-Out con Diseño Factorial 2×2 ───────────────
N_TRAIN    = 1000
N_VAL_OOD  = 200
N_TEST_OOD = 200
K_REPS     = 4

MIN_POOL_A = 2 * N_TRAIN
MIN_POOL_B = K_REPS * (N_VAL_OOD + N_TEST_OOD)

print(f'\nMetodología: Repeated Hold-Out OOD — Diseño Factorial 2×2')
print(f'  N_TRAIN   : {N_TRAIN} casos por repetición')
print(f'  N_VAL_OOD : {N_VAL_OOD} casos por repetición')
print(f'  N_TEST_OOD: {N_TEST_OOD} casos por repetición')
print(f'  K_REPS    : {K_REPS} repeticiones por trial')
print(f'  Pool A mínimo: {MIN_POOL_A} casos')
print(f'  Pool B mínimo: {MIN_POOL_B} casos')
print(f'  Trials: 31')
print(f'\nTRIALS_DIR: {TRIALS_DIR}')
print('CELDA 1 OK')

Device: cuda
GPU:  NVIDIA GeForce RTX 3070 Ti
VRAM: 8.6 GB
TRIALS_DIR (V8): C:\Users\rodri\Documents\PINN\b2_proyecto\models\trials_31\V8
HDF5: dataset_familyB_OPENSEES_v6_20260523_1226.h5

Metodología: Repeated Hold-Out OOD — Diseño Factorial 2×2
  N_TRAIN   : 1000 casos por repetición
  N_VAL_OOD : 200 casos por repetición
  N_TEST_OOD: 200 casos por repetición
  K_REPS    : 4 repeticiones por trial
  Pool A mínimo: 2000 casos
  Pool B mínimo: 1600 casos
  Trials: 31

TRIALS_DIR: C:\Users\rodri\Documents\PINN\b2_proyecto\models\trials_31\V8
CELDA 1 OK


## 📌 Celda 2 — Carga del dataset HDF5 OpenSees v4

### 🎯 ¿Qué hace esta celda?

Abre el archivo HDF5 del dataset `Familia B` (generado con OpenSees) y carga **todos los arrays** que el modelo necesita para entrenar y evaluar. El dataset contiene **~20 000 casos paramétricos** de edificios residenciales chilenos según NCh 433.

### 📦 Datos cargados (estructura del HDF5)

| Grupo HDF5 | Variable | Forma | Significado físico |
|---|---|---|---|
| `inputs/` | `X` | $(N, 25)$ | Features de entrada (geometría, materiales, suelo, zona) |
| `inputs/` | `mask_floors` | $(N, 18)$ | Máscara binaria de pisos activos por edificio |
| `inputs/` | `floor_masses` | $(N, 18)$ | Masas concentradas por piso $m_i$ [t] |
| `modal/` | `T_r` | $(N, 18)$ | Periodos modales $T_r$ [s] |
| `modal/` | `Phi_x, Phi_y, Phi_theta` | $(N, 18, 18)$ | Formas modales $\Phi$ en 3 GDL/piso |
| `modal/` | `mass_acc_x, mass_acc_y` | $(N, 18)$ | Masa modal acumulada (NCh 433 §6.3.3) |
| `modal/` | `omega_r` | $(N, 18)$ | Frecuencias angulares $\omega_r = 2\pi/T_r$ [rad/s] |
| `response/` | `Ux_m, Uy_m` | $(N, 18)$ | Desplazamientos máximos por piso [m] |
| `response/` | `drift_x, drift_y` | $(N, 18)$ | Derivas de entrepiso [adim] |
| `response/` | `Vb_x_kN, Vb_y_kN` | $(N,)$ | Cortante basal [kN] |
| `stiffness/` | `K_global, M_global` | $(N, 54, 54)$ | Matrices $\mathbf{K}, \mathbf{M}$ globales (18 pisos × 3 GDL = 54) |

### 🔧 Normalización de las formas modales $\Phi$

Las formas modales se renormalizan a **norma euclidiana unitaria por modo**:

$$
\hat{\Phi}_r = \frac{\Phi_r}{\|\Phi_r\|_2}, \qquad \|\Phi_r\|_2 = \sqrt{\sum_{i=1}^{N_p} \Phi_{r,i}^2}
$$

**Justificación ingenieril:**
- Las formas modales en dinámica son **invariantes a escala** (se pueden escalar arbitrariamente).
- Normalizar a $\|\Phi_r\| = 1$ hace que la escala sea independiente del número de pisos $N_p$.
- Esto es consistente con la base sinusoidal que aprende la PINN, que genera vectores con norma $\propto \sqrt{N_p}$.
- Sin esta normalización, los gradientes de la **MAC loss** ($L_\phi$) están sesgados por el tamaño del edificio.

### 📐 Matriz global $\mathbf{K}$ — esquema

```
          ┌──────────────────────────────────┐
          │  K_global  (54 × 54)             │
          │                                  │  Piso 1: GDL [Ux, Uy, θz]
          │  ┌─────┐                         │  Piso 2: GDL [Ux, Uy, θz]
          │  │ K_1 │                         │  ...
          │  └─────┘ ← acoplamiento piso-piso│  Piso 18: GDL [Ux, Uy, θz]
          │           ┌─────┐                │
          │           │ K_2 │                │  Total: 18 pisos × 3 GDL = 54
          │           └─────┘                │
          │                  ...             │
          └──────────────────────────────────┘
```

### ✅ Lo que se obtiene

- Todos los tensores numpy listos en RAM.
- `N_CASES`: número total de casos disponibles.
- `N_MODOS_PHYS = 18`: número de modos físicos extraídos.
- Verificación de que **no hay NaN** en las features.
- Estimación de memoria de las matrices $\mathbf{K}, \mathbf{M}$ por batch.

In [3]:
# CELDA 2 -- CARGA HDF5 OPENSEES v4
with h5py.File(H5_MODAL, 'r') as h5:
    X_raw        = h5['inputs/X'][:].astype(np.float32)
    input_feats  = [s.decode() for s in h5['inputs/input_features'][:]]
    mask_floors  = h5['inputs/mask_floors'][:].astype(np.float32)
    floor_masses = h5['inputs/floor_masses'][:].astype(np.float32)
    T_r          = h5['modal/T_r'][:].astype(np.float32)
    Phi_x        = h5['modal/Phi_x'][:].astype(np.float32)
    Phi_y        = h5['modal/Phi_y'][:].astype(np.float32)
    Phi_theta    = h5['modal/Phi_theta'][:].astype(np.float32)
    mass_acc_x   = h5['modal/mass_acc_x'][:].astype(np.float32)
    mass_acc_y   = h5['modal/mass_acc_y'][:].astype(np.float32)
    mass_part_x  = h5['modal/mass_part_x'][:].astype(np.float32)
    mass_part_y  = h5['modal/mass_part_y'][:].astype(np.float32)
    omega_phys   = h5['modal/omega_r'][:].astype(np.float32)
    Ux_m         = h5['response/Ux_m'][:].astype(np.float32)
    Uy_m         = h5['response/Uy_m'][:].astype(np.float32)
    drift_x      = h5['response/drift_x'][:].astype(np.float32)
    drift_y      = h5['response/drift_y'][:].astype(np.float32)
    Vb_x         = h5['response/Vb_x_kN'][:].astype(np.float32)
    Vb_y         = h5['response/Vb_y_kN'][:].astype(np.float32)
    N_pisos_arr  = h5['scalars/N_pisos'][:].astype(int)
    K_global     = h5['stiffness/K_global'][:].astype(np.float32)
    M_global     = h5['stiffness/M_global'][:].astype(np.float32)
    I_rot_floors = h5['stiffness/I_rot_floors'][:].astype(np.float32)

N_CASES      = X_raw.shape[0]
N_MODOS_PHYS = T_r.shape[1]  # 18 slots

# ── Normalizar Phi a norma vectorial=1 por modo ───────────────────────────
# Norma L2 del vector por caso — independiente de N_pisos
# Garantiza escala consistente con la base sinusoidal del PINN
# que genera vectores de norma proporcional a sqrt(N_pisos)
print("Normalizando Phi a norma vectorial=1 por modo...")
for r in range(Phi_x.shape[2]):
    for Phi_arr in [Phi_x, Phi_y, Phi_theta]:
        norma_r = np.sqrt((Phi_arr[:, :, r]**2).sum(axis=1, keepdims=True))
        Phi_arr[:, :, r] = np.where(norma_r > 1e-10,
                                     Phi_arr[:, :, r] / norma_r,
                                     0.0)

print(f'Casos        : {N_CASES}')
print(f'T_r          : {T_r.shape}')
print(f'Phi_x        : {Phi_x.shape}')
print(f'K_global     : {K_global.shape}')
print(f'M_global     : {M_global.shape}')
print(f'omega_phys   : {omega_phys.shape}')
print(f'NaN en X     : {np.isnan(X_raw).sum()}')
print(f'N_MODOS_PHYS : {N_MODOS_PHYS}')

kb_per_case = 54 * 54 * 4 / 1024
print(f'\nMemoria K_global por caso : {kb_per_case:.1f} KB')
print(f'Memoria batch K_global (bs=64) : {kb_per_case*64/1024:.1f} MB')

# Verificar norma vectorial — debe ser 1.0
print('\nVerificacion norma vectorial Phi (debe ser 1.0):')
for i in range(3):
    norma_y = np.sqrt((Phi_y[i,:,0]**2).sum())
    norma_t = np.sqrt((Phi_theta[i,:,1]**2).sum())
    norma_x = np.sqrt((Phi_x[i,:,2]**2).sum())
    print(f'  caso {i}: '
          f'norma Phi_y modo 0={norma_y:.4f}  '
          f'norma Phi_t modo 1={norma_t:.4f}  '
          f'norma Phi_x modo 2={norma_x:.4f}')

print('\nFeatures:')
for i, f in enumerate(input_feats):
    print(f'  [{i:02d}] {f}')

print('\nCELDA 2 OK')
# ── Advertencia features constantes ──────────────────────────────────────
_FEATURES_CONSTANTES = []
for i, feat in enumerate(input_feats):
    v = X_raw[:, i]
    if v.std() < 1e-8:
        _FEATURES_CONSTANTES.append((i, feat, float(v[0])))

if _FEATURES_CONSTANTES:
    print(f'\nAVISO — Features constantes en el HDF5 (std=0):')
    for i, feat, val in _FEATURES_CONSTANTES:
        print(f'  [{i:02d}] {feat} = {val:.0f} siempre '
              f'← se excluye de X_clean en celda 3')
    print(f'  Estas features no aportan información al modelo.')
    print(f'  Son parámetros fijos del generador Familia B v6.')
    print(f'  Se mantienen en el HDF5 por trazabilidad del dataset.')
    print(f'  Se excluyen de COLS_BASE en celda 3 → N_INPUTS={23-len(_FEATURES_CONSTANTES)}')

print('\nCELDA 2 OK')

Normalizando Phi a norma vectorial=1 por modo...
Casos        : 10000
T_r          : (10000, 18)
Phi_x        : (10000, 18, 18)
K_global     : (10000, 54, 54)
M_global     : (10000, 54, 54)
omega_phys   : (10000, 18)
NaN en X     : 0
N_MODOS_PHYS : 18

Memoria K_global por caso : 11.4 KB
Memoria batch K_global (bs=64) : 0.7 MB

Verificacion norma vectorial Phi (debe ser 1.0):
  caso 0: norma Phi_y modo 0=1.0000  norma Phi_t modo 1=1.0000  norma Phi_x modo 2=1.0000
  caso 1: norma Phi_y modo 0=1.0000  norma Phi_t modo 1=1.0000  norma Phi_x modo 2=1.0000
  caso 2: norma Phi_y modo 0=1.0000  norma Phi_t modo 1=1.0000  norma Phi_x modo 2=1.0000

Features:
  [00] N_pisos
  [01] n_unid_lado
  [02] mods_por_depto
  [03] activar_B2
  [04] activar_B3
  [05] L_mod_m
  [06] prof_depto_m
  [07] ancho_corredor_m
  [08] L_nucleo_m
  [09] B_nucleo_m
  [10] h_story_m
  [11] fc_MPa
  [12] gk_kN_m2
  [13] t_muro_nucleo_m
  [14] t_muro_borde_m
  [15] t_muro_mid_m
  [16] suelo
  [17] zona

CELDA 2 OK

AVI

## 📌 Celda 3 — Preprocesamiento global y codificación One-Hot

### 🎯 ¿Qué hace esta celda?

Transforma las features de entrada `X_raw` en un formato amigable para la red neuronal:

1. **Codifica variables categóricas** con *one-hot encoding*:
   - `suelo` ∈ {0,1,2,3} → `suelo_A, suelo_B, suelo_C, suelo_D` (tipos de suelo NCh 433 §4.2)
   - `zona` ∈ {1,2,3} → `zona_1, zona_2, zona_3` (zonas sísmicas NCh 433 §4.1)

2. **Construye las 21 columnas finales** del vector de entrada:

| Categoría | Variables |
|---|---|
| **Geometría global** | `N_pisos`, `n_unid_lado`, `activar_B2`,  |
| **Dimensiones módulo** | `L_mod_m`, `prof_depto_m`, `ancho_corredor_m` |
| **Núcleo/muros** | `L_nucleo_m`, `B_nucleo_m`, `h_story_m` |
| **Materiales** | `fc_MPa` (resistencia hormigón), `gk_kN_m2` (carga gravitacional) |
| **Espesores muro** | `t_muro_nucleo_m`, `t_muro_borde_m`, `t_muro_mid_m` |
| **Suelo (OHE)** | `suelo_A`, `suelo_B`, `suelo_C`, `suelo_D` |
| **Zona sísmica (OHE)** | `zona_1`, `zona_2`, `zona_3` |

3. **Transformación logarítmica** de las salidas con respuesta sísmica:

$$
\tilde{y}_i = \begin{cases} \log(y_i) & \text{si } y_i > 10^{-8} \\ 0 & \text{en caso contrario} \end{cases}
$$

Para: $T_r$, $U_x$, $U_y$, $\text{drift}_x$, $\text{drift}_y$, $V_{b,x}$, $V_{b,y}$.

**¿Por qué log?**
- La respuesta sísmica varía sobre **varios órdenes de magnitud** (un edificio de 6 pisos vs uno de 18).
- $V_b$ puede ir desde 100 kN a 100 000 kN → MSE en escala lineal sería dominado por edificios altos.
- Predecir $\log(y)$ corresponde a minimizar el **error relativo**, que es más útil ingenierilmente.

### ✅ Lo que se obtiene

- `X_clean` ∈ $\mathbb{R}^{N \times 23}$: matriz de features lista para la red.
- `N_INPUTS = 21`: dimensión del input.
- Arrays log-transformados: `logT_global, logUx, logUy, logDx, logDy, logVbx, logVby`.

In [4]:
# CELDA 3 -- PREPROCESAMIENTO GLOBAL + ONE-HOT
df_X = pd.DataFrame(X_raw, columns=input_feats)
suelo_map_inv = {0:'A', 1:'B', 2:'C', 3:'D'}
suelo_ohe = pd.get_dummies(
    df_X['suelo'].map(suelo_map_inv), prefix='suelo'
).astype(np.float32)
zona_ohe = pd.get_dummies(
    df_X['zona'].astype(int), prefix='zona'
).astype(np.float32)
df_X = df_X.drop(columns=['suelo', 'zona'])
df_X = pd.concat([df_X, suelo_ohe, zona_ohe], axis=1)
for col in ['suelo_A','suelo_B','suelo_C','suelo_D']:
    if col not in df_X.columns: df_X[col] = 0.0
for col in ['zona_1','zona_2','zona_3']:
    if col not in df_X.columns: df_X[col] = 0.0

COLS_BASE = [
    'N_pisos', 'n_unid_lado',
    # 'mods_por_depto',  ← ELIMINADO: constante=2, std=0
    'activar_B2',
    # 'activar_B3',      ← ELIMINADO: constante=1, std=0
    'L_mod_m', 'prof_depto_m', 'ancho_corredor_m',
    'L_nucleo_m', 'B_nucleo_m', 'h_story_m',
    'fc_MPa', 'gk_kN_m2',
    't_muro_nucleo_m', 't_muro_borde_m', 't_muro_mid_m',
    'suelo_A', 'suelo_B', 'suelo_C', 'suelo_D',
    'zona_1', 'zona_2', 'zona_3',
]

faltantes = [c for c in COLS_BASE if c not in df_X.columns]
if faltantes:
    print(f'AVISO faltantes: {faltantes}')
else:
    df_X = df_X[COLS_BASE]

X_clean  = df_X.to_numpy(dtype=np.float32)
N_INPUTS = X_clean.shape[1]

with np.errstate(divide='ignore', invalid='ignore'):
    logT_global = np.where(T_r > 1e-8, np.log(np.where(T_r > 1e-8, T_r, 1.0)), 0.0).astype(np.float32)
    logUx       = np.where(Ux_m > 1e-8,    np.log(np.where(Ux_m > 1e-8,    Ux_m,    1.0)), 0.0).astype(np.float32)
    logUy       = np.where(Uy_m > 1e-8,    np.log(np.where(Uy_m > 1e-8,    Uy_m,    1.0)), 0.0).astype(np.float32)
    logDx       = np.where(drift_x > 1e-8, np.log(np.where(drift_x > 1e-8, drift_x, 1.0)), 0.0).astype(np.float32)
    logDy       = np.where(drift_y > 1e-8, np.log(np.where(drift_y > 1e-8, drift_y, 1.0)), 0.0).astype(np.float32)
    logVbx      = np.where(Vb_x > 1e-8,    np.log(np.where(Vb_x > 1e-8,    Vb_x,    1.0)), 0.0).astype(np.float32)
    logVby      = np.where(Vb_y > 1e-8,    np.log(np.where(Vb_y > 1e-8,    Vb_y,    1.0)), 0.0).astype(np.float32)

print(f'N_INPUTS (con one-hot, sin constantes): {N_INPUTS}')
print(f'X_clean : {X_clean.shape}')
print('CELDA 3 OK')

N_INPUTS (con one-hot, sin constantes): 21
X_clean : (10000, 21)
CELDA 3 OK


## 📌 Celda 4 — Arquitectura PINN v4 (definitiva)

### 🎯 ¿Qué hace esta celda?

Define la **arquitectura de red neuronal definitiva** `PINNModal_v4` con una innovación clave: **dos encoders independientes** para evitar la *contaminación de gradientes* (Wang et al., 2022).

### 🧠 Motivación experimental

| Versión | Encoder | T1_MAPE | Sesgo |
|---|---|---|---|
| `PINNModal_v2` | Compartido para todo | ~25% | +0.878 ❌ |
| **`PINNModal_v4`** | **encoder_T1 separado** | **7.8%** | **+0.089 ✅** |
| GradientBoosting (baseline) | — | — | ≈0 |

> **Diagnóstico**: La información para predecir $T_1$ ya está en las features; el problema era que los gradientes de $V_b$ y $\text{drift}$ (que son mucho mayores) **dominaban** el encoder compartido y sesgaban la predicción de $T_1$.

### 🏗️ Arquitectura — diagrama

```
                  X ∈ ℝ²³ (input features)
                        │
                ┌───────┴────────┐
                ▼                ▼
       ┌────────────────┐ ┌─────────────────────┐
       │  encoder_T1    │ │  encoder_compartido │
       │  (hidden=128)  │ │  (hidden=256)       │
       │                │ │                     │
       │  Linear→LN→SiLU│ │  Linear→LN→SiLU     │
       │  ResBlock × 2  │ │  ResBlock × 4       │
       └───────┬────────┘ └──────────┬──────────┘
               │                     │
               ▼                     ├─────────┬──────────┬─────────┐
        ┌──────────────┐             ▼         ▼          ▼         ▼
        │  head_T1     │      ┌──────────┐┌───────── ─┐┌─────── ─┐ ┌────── ┐
        │  (1 valor)   │      │head_T_rest││head_Phi  ││head_resp│ │head_Vb│
        └──────┬───────┘      │  (17)    ││  (18×18×3)││(18×4)   │ │ (2)   │
               │              └─────┬────┘└─────┬─────┘└────┬────┘ └──┬── ─┘
               ▼                    ▼          ▼            ▼        ▼
              T₁                  T₂..T₁₈     Φx,Φy,Φθ   Ux,Uy,    Vb_x,
                                                          drift_x,  Vb_y
                                                          drift_y
```

### 🧱 Bloque residual (ResBlock)

$$
\mathrm{ResBlock}(x) = \mathrm{SiLU}\big(x + \mathrm{LN}(W_2 \cdot \mathrm{SiLU}(\mathrm{LN}(W_1 x)))\big)
$$

donde **SiLU** (Sigmoid Linear Unit, también llamada *Swish*) es: $\mathrm{SiLU}(x) = x \cdot \sigma(x)$.

### 🔗 Garantía monótona de periodos: $T_1 > T_2 > \cdots > T_{18}$

Se construye la cadena por **diferencias positivas**:

$$
\log T_r = \log T_1 - \sum_{j=1}^{r-1} \Delta_j, \qquad \Delta_j = \mathrm{softplus}(z_j) > 0
$$

Esto **garantiza por diseño** el orden físico de los periodos modales.

### 📐 Normalización de formas modales

Después del head_Phi, cada modo se normaliza por su norma $L_2$:

$$
\hat{\Phi}_r = \frac{\Phi_r \odot \mathrm{mask}}{\|\Phi_r \odot \mathrm{mask}\|_2 + \epsilon}
$$

para evitar la ambigüedad de escala intrínseca de los autovectores.

### ✅ Lo que se obtiene

- Clase `PINNModal_v4` definida.
- Función `build_model()` que instancia el modelo en `device`.

In [5]:
# CELDA 4 -- ARQUITECTURA PINN v4b
# encoder_T1 separado (hidden=128) + encoder_compartido (hidden=256)
# v4b: dx via ratio — head_resp canal 2 = ratio, dx = ratio × dy

class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.LayerNorm(dim), nn.SiLU(),
            nn.Linear(dim, dim), nn.LayerNorm(dim),
        )
        self.act = nn.SiLU()
    def forward(self, x):
        return self.act(x + self.net(x))


class PINNModal_v4(nn.Module):
    def __init__(self, n_inputs, n_modos=18,
                 n_max_pisos=18, hidden=256, hidden_T1=128):
        super().__init__()
        self.n_modos     = n_modos
        self.n_max_pisos = n_max_pisos

        # ── encoder_T1 dedicado ───────────────────────────────────────────
        self.encoder_T1 = nn.Sequential(
            nn.Linear(n_inputs, hidden_T1),
            nn.LayerNorm(hidden_T1),
            nn.SiLU(),
            ResBlock(hidden_T1),
            ResBlock(hidden_T1),
        )
        self.head_T1 = nn.Sequential(
            nn.Linear(hidden_T1, 32),
            nn.SiLU(),
            nn.Linear(32, 1)
        )

        # ── encoder compartido ────────────────────────────────────────────
        self.encoder = nn.Sequential(
            nn.Linear(n_inputs, hidden),
            nn.LayerNorm(hidden),
            nn.SiLU(),
            ResBlock(hidden), ResBlock(hidden),
            ResBlock(hidden), ResBlock(hidden),
        )
        self.head_T_rest = nn.Sequential(
            nn.Linear(hidden, 128), nn.SiLU(),
            nn.Linear(128, n_modos - 1)
        )
        self.head_Phi = nn.Sequential(
            nn.Linear(hidden, 512), nn.SiLU(),
            nn.Linear(512, n_modos * n_max_pisos * 3),
        )
        self.head_resp = nn.Sequential(
            nn.Linear(hidden, 256), nn.SiLU(),
            nn.Linear(256, n_max_pisos * 4),
        )
        self.head_Vb = nn.Sequential(
            nn.Linear(hidden, 64), nn.SiLU(),
            nn.Linear(64, 2)
        )

    def forward(self, X, mask):
        B = X.shape[0]

        # T1
        h_T1   = self.encoder_T1(X)
        T1_out = self.head_T1(h_T1)

        # T2-T18
        h        = self.encoder(X)
        T_deltas = F.softplus(self.head_T_rest(h))
        T_parts  = [T1_out]
        for r in range(self.n_modos - 1):
            T_parts.append(T_parts[-1] - T_deltas[:, r:r+1])
        logT_pred = torch.cat(T_parts, dim=1)

        # Formas modales
        Phi_raw  = self.head_Phi(h).view(B, self.n_modos, self.n_max_pisos, 3)
        mask_4d  = mask.unsqueeze(1).unsqueeze(-1)
        Phi_raw  = Phi_raw * mask_4d
        norma    = Phi_raw.norm(dim=2, keepdim=True).clamp(min=1e-8)
        Phi_norm = Phi_raw / norma
        Phi_all  = Phi_norm.permute(0, 2, 1, 3)
        Phi_x_p     = Phi_all[:, :, :, 0]
        Phi_y_p     = Phi_all[:, :, :, 1]
        Phi_theta_p = Phi_all[:, :, :, 2]

        # Respuesta por piso — canal 2 ahora es ratio (dx = ratio × dy)
        resp    = self.head_resp(h).view(B, self.n_max_pisos, 4)
        Ux_p    = resp[:, :, 0] * mask
        Uy_p    = resp[:, :, 1] * mask
        ratio_p = F.softplus(resp[:, :, 2]) * mask   # ← ratio > 0
        dy_p    = resp[:, :, 3] * mask

        # dx = expm1(log1p_ratio) * dy  (BUG3 fix: ratio in log1p space)
        dx_p = torch.expm1(ratio_p.clamp(min=0)) * dy_p

        # Corte basal
        Vb_p = self.head_Vb(h)

        # ← 10 outputs — v4b, dx via ratio
        return (logT_pred, Phi_x_p, Phi_y_p, Phi_theta_p,
                Ux_p, Uy_p, dx_p, dy_p, Vb_p, ratio_p)


def build_model():
    m = PINNModal_v4(
        N_INPUTS, N_MODOS, N_MAX_PISOS,
        hidden=256, hidden_T1=128
    ).to(device)
    n_total  = sum(p.numel() for p in m.parameters() if p.requires_grad)
    n_enc_T1 = sum(p.numel() for p in m.encoder_T1.parameters()) + \
               sum(p.numel() for p in m.head_T1.parameters())
    n_enc_sh = sum(p.numel() for p in m.encoder.parameters())
    print(f'Parámetros totales    : {n_total:,}')
    print(f'  encoder_T1+head_T1  : {n_enc_T1:,}')
    print(f'  encoder_compartido  : {n_enc_sh:,}')
    print(f'  N_INPUTS            : {N_INPUTS}  (sin features constantes)')
    return m

_ = build_model()
print('CELDA 4 OK -- PINNModal_v4b  (10 outputs, dx via ratio)')

Parámetros totales    : 1,377,064
  encoder_T1+head_T1  : 74,305
  encoder_compartido  : 536,576
  N_INPUTS            : 21  (sin features constantes)
CELDA 4 OK -- PINNModal_v4b  (10 outputs, dx via ratio)


In [6]:
# ============================================================
# CELDA PASO 1 — Validacion NCh433SpectrumLayer
# Verifica que la implementacion PyTorch diferenciable
# reproduce exactamente el espectro NCh433 de nch433.py
# ============================================================
import sys
sys.path.insert(0, r'C:\Users\rodri\Documents\PINN\b2_proyecto')
from b2core.nch433_torch import NCh433SpectrumLayer, Sa_nch433_numpy
import torch, numpy as np

layer = NCh433SpectrumLayer().to(device)
layer.eval()

test_cases = [
    (1, 'A',  6, [0.10, 0.20, 0.50, 1.00, 2.00]),
    (2, 'B', 10, [0.15, 0.30, 0.75, 1.50]),
    (3, 'C', 14, [0.40, 0.60, 0.90, 1.20, 2.50]),
    (1, 'D', 18, [0.10, 0.75, 1.00, 1.40, 2.00, 3.00]),
    (3, 'D',  8, [0.30, 0.60, 1.00, 1.80]),
]

print('Zona Suelo  N |     max|dSa|     max|dSd|  Status')
print('-'*55)
all_ok = True
for zone, soil, N, T_list in test_cases:
    T_arr  = np.array(T_list, dtype=np.float32)
    Sa_np  = np.array([Sa_nch433_numpy(t, zone, soil, N) for t in T_arr])
    Sd_np  = Sa_np * T_arr**2 / (4.0 * np.pi**2)
    soil_idx = ['A','B','C','D'].index(soil)
    x_raw = np.zeros((len(T_arr), 21), dtype=np.float32)
    x_raw[:, 0]              = N
    x_raw[:, 14 + soil_idx]  = 1.0
    x_raw[:, 18 + (zone-1)]  = 1.0
    T_diag = torch.diag(torch.tensor(T_arr)).to(device)
    x_pt   = torch.tensor(x_raw).to(device)
    with torch.no_grad():
        Sa_m, Sd_m = layer(T_diag, x_pt)
    Sa_pt = Sa_m.diagonal().cpu().numpy()
    Sd_pt = Sd_m.diagonal().cpu().numpy()
    err_Sa = np.abs(Sa_pt - Sa_np).max()
    err_Sd = np.abs(Sd_pt - Sd_np).max()
    ok = err_Sa < 1e-4 and err_Sd < 1e-6
    if not ok: all_ok = False
    print(f'   {zone}     {soil}  {N:2d} | {err_Sa:12.2e} {err_Sd:12.2e}  {"OK" if ok else "FAIL"}')

print('-'*55)
print(f'Resultado: {"ALL PASSED" if all_ok else "SOME FAILED"}')

# Curva espectral visual
import matplotlib.pyplot as plt
T_plot = np.linspace(0.01, 3.0, 300).astype(np.float32)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax_i, zone in enumerate([1, 2, 3]):
    ax = axes[ax_i]
    for soil, ls in zip(['A','B','C','D'], ['-','--',':','-.']):
        Sa_curve = np.array([Sa_nch433_numpy(t, zone, soil, 12) for t in T_plot])
        ax.plot(T_plot, Sa_curve / 9.80665, ls=ls, label=f'Suelo {soil}')
    ax.set_xlabel('T [s]'); ax.set_ylabel('Sa/g')
    ax.set_title(f'Zona {zone} (N=12)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.suptitle('Espectro NCh433 — Validacion PyTorch', fontsize=12)
plt.tight_layout(); plt.show()
print('CELDA PASO 1 OK')


Zona Suelo  N |     max|dSa|     max|dSd|  Status
-------------------------------------------------------
   1     A   6 |     1.97e-08     7.85e-10  OK
   2     B  10 |     6.27e-08     5.45e-10  OK
   3     C  14 |     9.97e-08     2.43e-09  OK
   1     D  18 |     1.15e-07     4.49e-09  OK
   3     D   8 |     2.30e-07     1.36e-08  OK
-------------------------------------------------------
Resultado: ALL PASSED
CELDA PASO 1 OK


C:\Users\rodri\AppData\Local\Temp\ipykernel_8692\3856476707.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 📌 Celda 5 — Función de pérdida PINN v16 (definitiva)

### 🎯 ¿Qué hace esta celda?

Define la **pérdida total** del PINN como una **combinación ponderada** de términos data-driven y términos físicos. La versión `v16` separa los términos para garantizar gradientes "limpios" hacia cada encoder e introduce un término de **ratio de derivas**.

> **Cambio clave v15 → v16:** se elimina `dx` (deriva en X) de $\mathcal{L}_{\text{data\_resp}}$ y se supervisa indirectamente mediante un nuevo término $\mathcal{L}_{\text{ratio}}$ que predice el cociente $\text{drift}_x/\text{drift}_y$. Motivo documentado en el código: $\text{drift}_x$ no es predecible en valor absoluto desde las features disponibles, pero el ratio relativo sí es estable.

### 🧮 Pérdida total (8 términos)

$$
\mathcal{L}_{\text{total}} = \lambda_{\text{r}}\,\mathcal{L}_{\text{data\_resp}} + \lambda_{\text{ratio}}\,\mathcal{L}_{\text{ratio}} + \lambda_{\text{T}_r}\,\mathcal{L}_{\text{data\_T\_rest}} + \lambda_{\text{T}_1}\,\mathcal{L}_{\text{data\_T}_1} + \lambda_{T_1}\,\mathcal{L}_{T_1} + \lambda_\phi\,\mathcal{L}_\phi + \lambda_{\text{eig}}\,\mathcal{L}_{\text{eig}} + \lambda_{\text{o}}\,\mathcal{L}_{\text{ortho}}
$$

### 📊 Desglose de los 8 términos

| Término | Qué supervisa | Gradiente fluye a → |
|---|---|---|
| $\mathcal{L}_{\text{data\_resp}}$ | $\mathrm{MSE}$ de $U_x, U_y, \text{drift}_y, V_{b,x}, V_{b,y}$ (**sin** $\text{drift}_x$) | encoder_compartido |
| $\mathcal{L}_{\text{ratio}}$ | $\mathrm{MSE}$ enmascarado de $\widehat{\text{ratio}}$ vs $\text{drift}_x/\text{drift}_y$ | encoder_compartido |
| $\mathcal{L}_{\text{data\_T\_rest}}$ | $\frac{1}{N_v}\sum_{r=2}^{18} \text{mask}_r (\widehat{\log T_r} - \log T_r)^2$ | encoder_compartido |
| $\mathcal{L}_{\text{data\_T}_1}$ | $\mathrm{MSE}(\widehat{\log T_1}, \log T_1)$ | **encoder_T1** |
| $\mathcal{L}_{T_1}$ (explícito) | $\mathrm{MSE}(\widehat{\log T_1}, \log T_1)$ (refuerzo) | encoder_T1 |
| $\mathcal{L}_\phi$ (MAC loss) | $\frac{1}{N_M}\sum_r (1 - \mathrm{MAC}_r)$ | encoder_compartido |
| $\mathcal{L}_{\text{eig}}$ (eigen) | residuo de $\mathbf{K}\Phi = \omega^2\mathbf{M}\Phi$ | encoder_compartido |
| $\mathcal{L}_{\text{ortho}}$ (M-orto) | $\|\Phi^\top \mathbf{M} \Phi - \mathbf{I}\|^2$ | encoder_compartido |

> Los términos $\mathcal{L}_\phi$, $\mathcal{L}_{\text{eig}}$ y $\mathcal{L}_{\text{ortho}}$ solo se computan cuando `stage_active` los activa (ver curriculum); en otro caso valen 0.

### 🔬 Pérdida física $\mathcal{L}_{\text{eig}}$ — el corazón del PINN

Fuerza el **problema generalizado de autovalores** de la dinámica modal, con $\mathbf{K}$ y $\mathbf{M}$ escaladas por $\max|\mathbf{K}|$ del caso para evitar problemas de magnitud entre edificios:

$$
\mathbf{K}\,\Phi_r = \omega_r^2 \,\mathbf{M}\,\Phi_r, \qquad \omega_r = \frac{2\pi}{T_r}
$$

El residuo normalizado, evaluado sobre los primeros $R = \min(N_{\text{modos phys}}, \cdot)$ modos:

$$
\mathcal{L}_{\text{eig}} = \mathbb{E}\!\left[\frac{\|\mathbf{K}\Phi_r - \omega_r^2 \mathbf{M}\Phi_r\|_2^2}{\|\mathbf{K}\Phi_r\|_2^2 + \|\omega_r^2 \mathbf{M}\Phi_r\|_2^2 + \epsilon}\right]
$$

El período se reconstruye desnormalizando la predicción ($T = \exp(\widehat{\log T}\cdot\sigma + \mu)$) y los modos se reensamblan en formato global de 54 GDL intercalando X/Y/θ antes de normalizar.

### 🎯 Pérdida modal MAC ($\mathcal{L}_\phi$)

**Modal Assurance Criterion** (Allemang & Brown, 1982), evaluado por modo según el patrón cíclico de dirección:

$$
\mathrm{MAC}(\Phi^{\text{pred}}_r, \Phi^{\text{true}}_r) = \frac{(\Phi^{\text{pred}\,\top}_r \Phi^{\text{true}}_r)^2}{(\Phi^{\text{pred}\,\top}_r \Phi^{\text{pred}}_r)(\Phi^{\text{true}\,\top}_r \Phi^{\text{true}}_r)}
$$

- $\mathrm{MAC} = 1$: formas modales idénticas (excepto escala).
- $\mathrm{MAC} \geq 0.9$: aceptable (Pastor et al., 2012).

> **Patrón cíclico:** los modos se ordenan alternando dirección — modo $r$ con $r\bmod 3 = 0 \to$ Y, $=1 \to$ T (torsional), $=2 \to$ X.

### 🎼 Pérdida de M-ortogonalidad ($\mathcal{L}_{\text{ortho}}$)

Propiedad clásica de los modos normalizados a masa, penalizada con MSE sobre los primeros $R$ modos:

$$
\Phi^\top \mathbf{M}\, \Phi = \mathbf{I}
$$

### ✅ Lo que se obtiene

- Función `compute_loss_v16(batch, model_out, lambdas, scalers, stage_active)` que devuelve $\mathcal{L}_{\text{total}}$ y un diccionario con cada componente (incluido `ratio`).
- Alias `compute_loss = compute_loss_v16` para compatibilidad con el resto del notebook.
- `model_out` tiene **10 salidas**: `logT, Phi_x, Phi_y, Phi_t, Ux, Uy, dx, dy, Vb, ratio`.

In [7]:
# CELDA 5 -- LOSS PINN v16
# Cambios vs v15:
#   + L_ratio = MSE(ratio_p, drift_x/drift_y) — NUEVO
#   - dx_p eliminado de L_data_resp (supervisado via ratio)
#   Motivo: drift_x no predecible en valor absoluto desde features
#   disponibles. Predecir ratio relativo es más estable.

def compute_loss_v16(batch, model_out, lambdas, scalers, stage_active):
    logT_p, Phi_x_p, Phi_y_p, Phi_t_p, \
        Ux_p, Uy_p, dx_p, dy_p, Vb_p, ratio_p = model_out  # 10 outputs

    logT_true  = batch['logT'].to(device)
    Phi_x_true = batch['Phi_x'].to(device)
    Phi_y_true = batch['Phi_y'].to(device)
    Phi_t_true = batch['Phi_theta'].to(device)
    Ux_true    = batch['logUx'].to(device)
    Uy_true    = batch['logUy'].to(device)
    dy_true    = batch['logDy'].to(device)
    Vbx_true   = batch['logVbx'].to(device)
    Vby_true   = batch['logVby'].to(device)
    ratio_true = batch['ratio_dx_dy'].to(device)
    mask       = batch['mask'].to(device)

    ld = {}

    # ── L_data_resp — Vb, Uy, Ux, dy (dx via ratio) ──────────────────────
    ld['data_resp'] = (
        F.mse_loss(Ux_p * mask, Ux_true * mask) +
        F.mse_loss(Uy_p * mask, Uy_true * mask) +
        F.mse_loss(dy_p * mask, dy_true * mask) +
        F.mse_loss(Vb_p[:, 0], Vbx_true)        +
        F.mse_loss(Vb_p[:, 1], Vby_true)
    )

    # ── L_ratio — ratio drift_x/drift_y ──────────────────────────────────
    n_activos   = mask.sum().clamp(min=1.0)
    ld['ratio'] = (
        (ratio_p - ratio_true).pow(2) * mask
    ).sum() / n_activos

    # ── L_data_T_rest — T2-T18 ───────────────────────────────────────────
    mask_T  = (batch['T_r'].to(device)[:, 1:] > 1e-6).float()
    n_valid = mask_T.sum().clamp(min=1.0)
    ld['data_T_rest'] = (
        (logT_p[:, 1:] - logT_true[:, 1:]).pow(2) * mask_T
    ).sum() / n_valid

    # ── L_data_T1 ────────────────────────────────────────────────────────
    ld['data_T1'] = F.mse_loss(logT_p[:, 0], logT_true[:, 0])

    # ── L_phi ────────────────────────────────────────────────────────────
    L_phi = torch.tensor(0.0, device=device)
    if stage_active.get('phi', False):
        for r in range(N_MODOS):
            ciclo = r % 3
            if ciclo == 0:
                Phi_p_r = Phi_y_p[:, :, r]; Phi_t_r = Phi_y_true[:, :, r]
            elif ciclo == 1:
                Phi_p_r = Phi_t_p[:, :, r]; Phi_t_r = Phi_t_true[:, :, r]
            else:
                Phi_p_r = Phi_x_p[:, :, r]; Phi_t_r = Phi_x_true[:, :, r]
            pp     = Phi_p_r * mask
            pt     = Phi_t_r * mask
            num    = (pp * pt).sum(dim=1) ** 2
            den    = ((pp**2).sum(dim=1) *
                      (pt**2).sum(dim=1)).clamp(min=1e-12)
            valido = (pt.pow(2).sum(dim=1) > 1e-10).float()
            n_val  = valido.sum().clamp(min=1.0)
            L_phi  = L_phi + (valido * (1.0 - num/den)).sum() / n_val
        L_phi = L_phi / N_MODOS
    ld['phi'] = L_phi

    # ── L_eig ────────────────────────────────────────────────────────────
    L_eig = torch.tensor(0.0, device=device)
    if stage_active.get('eig', False):
        K_b = batch['K_global'].to(device)
        M_b = batch['M_global'].to(device)
        _Bs = Phi_x_p.shape[0]
        _mf = batch['mask'].to(device).unsqueeze(-1).expand(-1,-1,3).reshape(_Bs,54)
        _m2 = _mf.unsqueeze(2) * _mf.unsqueeze(1)
        K_b = K_b * _m2;  M_b = M_b * _m2
        K_scale = K_b.abs().amax(dim=(-2,-1)).clamp(min=1.0)
        K_b = K_b / K_scale.unsqueeze(-1).unsqueeze(-1)
        M_b = M_b / K_scale.unsqueeze(-1).unsqueeze(-1)
        R   = min(N_MODOS_PHYS, Phi_x_p.shape[2])
        Phi_g = torch.zeros(Phi_x_p.shape[0], 54, R,
                            dtype=Phi_x_p.dtype, device=device)
        Phi_g[:, 0::3, :] = Phi_x_p[:, :, :R]
        Phi_g[:, 1::3, :] = Phi_y_p[:, :, :R]
        Phi_g[:, 2::3, :] = Phi_t_p[:, :, :R]
        Phi_g   = Phi_g / Phi_g.norm(dim=1, keepdim=True).clamp(min=1e-8)
        logT_std_R  = torch.tensor(scalers['logT_std'][:R],
                                   dtype=torch.float32, device=device)
        logT_mean_R = torch.tensor(scalers['logT_mean'][:R],
                                   dtype=torch.float32, device=device)
        T_pred  = torch.exp(
            logT_p[:, :R] * logT_std_R + logT_mean_R).clamp(min=1e-6)
        omega2  = (2.0 * math.pi / T_pred) ** 2
        KPhi    = torch.bmm(K_b, Phi_g)
        MPhi    = torch.bmm(M_b, Phi_g)
        o2MPhi  = omega2.unsqueeze(1) * MPhi
        residual= KPhi - o2MPhi
        denom   = (KPhi.pow(2).sum(dim=1) +
                   o2MPhi.pow(2).sum(dim=1) + 1e-12)
        L_eig   = (residual.pow(2).sum(dim=1) / denom).mean()
    ld['eig'] = L_eig

    # ── L_ortho ───────────────────────────────────────────────────────────
    L_ortho = torch.tensor(0.0, device=device)
    if stage_active.get('ortho', False):
        M_b = batch['M_global'].to(device)
        _Bs = Phi_x_p.shape[0]
        _mf = batch['mask'].to(device).unsqueeze(-1).expand(-1,-1,3).reshape(_Bs,54)
        _m2 = _mf.unsqueeze(2) * _mf.unsqueeze(1)
        M_b = M_b * _m2
        K_s = M_b.abs().amax(dim=(-2,-1)).clamp(min=1.0)
        M_b = M_b / K_s.unsqueeze(-1).unsqueeze(-1)
        R   = min(N_MODOS_PHYS, Phi_x_p.shape[2])
        Phi_g = torch.zeros(Phi_x_p.shape[0], 54, R,
                            dtype=Phi_x_p.dtype, device=device)
        Phi_g[:, 0::3, :] = Phi_x_p[:, :, :R]
        Phi_g[:, 1::3, :] = Phi_y_p[:, :, :R]
        Phi_g[:, 2::3, :] = Phi_t_p[:, :, :R]
        Phi_g   = Phi_g / Phi_g.norm(dim=1, keepdim=True).clamp(min=1e-8)
        MPhi    = torch.bmm(M_b, Phi_g)
        G       = torch.bmm(Phi_g.transpose(1, 2), MPhi)
        I_mat   = torch.eye(R, device=device).unsqueeze(0)
        L_ortho = ((G - I_mat) ** 2).mean()
    ld['ortho'] = L_ortho

    # ── L_total ───────────────────────────────────────────────────────────
    L_total = (
        lambdas.get('data_resp',   0.0) * ld['data_resp']   +
        lambdas.get('ratio',       0.0) * ld['ratio']       +
        lambdas.get('data_T_rest', 0.0) * ld['data_T_rest'] +
        lambdas.get('data_T1',     0.0) * ld['data_T1']     +
        lambdas.get('phi',         0.0) * ld['phi']         +
        lambdas.get('eig',         0.0) * ld['eig']         +
        lambdas.get('ortho',       0.0) * ld['ortho']
    )
    return L_total, {
        k: v.item() if hasattr(v, 'item') else float(v)
        for k, v in ld.items()
    }

compute_loss = compute_loss_v16

print('CELDA 5 OK -- Loss v16')
print('  L_data_resp  : MSE(Ux, Uy, dy, Vb)  ← dx via ratio')
print('  L_ratio      : MSE(ratio_p, drift_x/drift_y)')
print('  L_data_T_rest: MSE(logT[:,1:])')
print('  L_data_T1    : MSE(logT[:,0])')
print('  L_T1         : término explícito T1')
print('  L_phi        : MAC loss')
print('  L_eig        : K·Phi = w²·M·Phi')
print('  L_ortho      : Phi^T·M·Phi = I')

CELDA 5 OK -- Loss v16
  L_data_resp  : MSE(Ux, Uy, dy, Vb)  ← dx via ratio
  L_ratio      : MSE(ratio_p, drift_x/drift_y)
  L_data_T_rest: MSE(logT[:,1:])
  L_data_T1    : MSE(logT[:,0])
  L_T1         : término explícito T1
  L_phi        : MAC loss
  L_eig        : K·Phi = w²·M·Phi
  L_ortho      : Phi^T·M·Phi = I


## 📌 Celda 6 — Dataset PyTorch + funciones auxiliares

### 🎯 ¿Qué hace esta celda?

Define las **abstracciones PyTorch** necesarias para iterar sobre el dataset durante el entrenamiento.

### 🔧 Componentes definidos

#### 1. Clase `ModalDataset(Dataset)`

Hereda de `torch.utils.data.Dataset`. Para cada caso $j$, devuelve un diccionario con **todos los tensores** del problema (input normalizado, máscara, masas, periodos log-normalizados, formas modales, respuestas, $\mathbf{K}$, $\mathbf{M}$, etc.).

La normalización de X se aplica al vuelo:

$$
\tilde{x}_j = \frac{x_j - \mu_X^{\text{train}}}{\sigma_X^{\text{train}}}
$$

donde $\mu_X^{\text{train}}, \sigma_X^{\text{train}}$ se calculan **solo** sobre el conjunto de entrenamiento (para evitar **data leakage**).

#### 2. Función `compute_scalers(idx_train)`

Calcula los **estadísticos de normalización** ($\mu, \sigma$) usando *solo* los índices de entrenamiento. Para los periodos log se normaliza modo a modo:

$$
\widetilde{\log T_r} = \frac{\log T_r - \mu_{\log T_r}^{\text{train}}}{\sigma_{\log T_r}^{\text{train}}}
$$

Análogamente para $U_x, U_y, \text{drift}_x, \text{drift}_y, V_{b,x}, V_{b,y}$ (pero sobre el **máximo por caso**, no por piso).

#### 3. Función `make_dataloaders(idx_tr, idx_val, idx_test, scalers)`

Crea tres `DataLoader`s. El de entrenamiento usa **`WeightedRandomSampler`** para balancear el número de pisos:

$$
w_j = \frac{1}{\text{counts}[N_{\text{pisos},j}]}
$$

> **¿Por qué?** Si el dataset tiene muchos más edificios de 8 pisos que de 18, sin balanceo la red **sobreajusta** a los edificios bajos. El sampler ponderado fuerza una distribución uniforme sobre $N_{\text{pisos}}$.

#### 4. Función `build_model()`

Construye una instancia fresca de `PINNModal_v4` con los hiperparámetros definitivos:

$$
\text{hidden}=256, \quad \text{hidden}_{T_1}=128, \quad N_{\text{modos}}=18, \quad N_{\text{max pisos}}=18
$$

### ✅ Lo que se obtiene

- `ModalDataset` — generador de batches.
- `compute_scalers(idx)` — calcula $\mu, \sigma$ por trial.
- `make_dataloaders(...)` — crea los 3 DataLoaders.
- `build_model()` — alias para construir el modelo.

In [8]:
# CELDA 6 -- DATASET PYTORCH + FUNCIONES AUXILIARES

class ModalDataset(Dataset):
    def __init__(self, indices, scalers):
        self.idx = indices
        self.sc  = scalers
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]; sc = self.sc
        return {
            'X':           torch.tensor((X_clean[j]-sc['X_mean'])/sc['X_std'], dtype=torch.float32),
            'mask':        torch.tensor(mask_floors[j],    dtype=torch.float32),
            'masses':      torch.tensor(floor_masses[j],   dtype=torch.float32),
            'N_pisos':     torch.tensor(N_pisos_arr[j],    dtype=torch.long),
            'h_story':     torch.tensor(X_clean[j, COLS_BASE.index('h_story_m')], dtype=torch.float32),
            'logT':        torch.tensor(sc['logT_norm'][j],   dtype=torch.float32),
            'T_r':         torch.tensor(T_r[j],               dtype=torch.float32),
            'Phi_x':       torch.tensor(Phi_x[j],             dtype=torch.float32),
            'Phi_y':       torch.tensor(Phi_y[j],             dtype=torch.float32),
            'Phi_theta':   torch.tensor(Phi_theta[j],         dtype=torch.float32),
            'mass_acc_x':  torch.tensor(mass_acc_x[j],        dtype=torch.float32),
            'mass_acc_y':  torch.tensor(mass_acc_y[j],        dtype=torch.float32),
            'logUx':       torch.tensor(sc['logUx_norm'][j],  dtype=torch.float32),
            'logUy':       torch.tensor(sc['logUy_norm'][j],  dtype=torch.float32),
            'logDx':       torch.tensor(sc['logDx_norm'][j],  dtype=torch.float32),
            'logDy':       torch.tensor(sc['logDy_norm'][j],  dtype=torch.float32),
            'logVbx':      torch.tensor(sc['logVbx_norm'][j], dtype=torch.float32),
            'logVby':      torch.tensor(sc['logVby_norm'][j], dtype=torch.float32),
            'Ux_raw':      torch.tensor(Ux_m[j],              dtype=torch.float32),
            'Uy_raw':      torch.tensor(Uy_m[j],              dtype=torch.float32),
            'Vbx_raw':     torch.tensor(Vb_x[j],             dtype=torch.float32),
            'Vby_raw':     torch.tensor(Vb_y[j],             dtype=torch.float32),
            'drift_x_raw': torch.tensor(drift_x[j],          dtype=torch.float32),
            'drift_y_raw': torch.tensor(drift_y[j],          dtype=torch.float32),
            'ratio_dx_dy': torch.tensor(
                np.log1p(drift_x[j] / np.clip(drift_y[j], 1e-6, None)),
                dtype=torch.float32),
            'omega_phys':  torch.tensor(omega_phys[j],        dtype=torch.float32),
            'K_global':    torch.tensor(K_global[j],          dtype=torch.float32),
            'M_global':    torch.tensor(M_global[j],          dtype=torch.float32),
        }


def compute_scalers(idx_train):
    X_m = X_clean[idx_train].mean(axis=0)
    X_s = X_clean[idx_train].std(axis=0); X_s[X_s < 1e-8] = 1.0

    N_MODOS_LOG = logT_global.shape[1]
    logT_m = np.zeros(N_MODOS_LOG, dtype=np.float32)
    logT_s = np.ones(N_MODOS_LOG,  dtype=np.float32)
    for r in range(N_MODOS_LOG):
        vals = logT_global[idx_train, r]; vals = vals[vals != 0]
        if len(vals) > 0:
            logT_m[r] = vals.mean()
            logT_s[r] = max(vals.std(), 1e-8)

    def scaler_pisos(arr_2d):
        """Media/std sobre todos los pisos activos — corrección bug .max(axis=1)"""
        vals = []
        for i in idx_train:
            n = int(N_pisos_arr[i])
            v = arr_2d[i, :n].flatten()
            v = v[v != 0]
            if len(v) > 0:
                vals.extend(v.tolist())
        vals = np.array(vals, dtype=np.float32)
        if len(vals) == 0:
            return 0.0, 1.0
        return float(vals.mean()), max(float(vals.std()), 1e-8)

    def scaler_1d(arr):
        v = arr[idx_train]; v = v[v != 0]
        if len(v) == 0: return 0.0, 1.0
        return float(v.mean()), max(float(v.std()), 1e-8)

    Ux_m_s,  Ux_s_s  = scaler_pisos(logUx)
    Uy_m_s,  Uy_s_s  = scaler_pisos(logUy)
    Dx_m_s,  Dx_s_s  = scaler_pisos(logDx)
    Dy_m_s,  Dy_s_s  = scaler_pisos(logDy)
    Vbx_m_s, Vbx_s_s = scaler_1d(logVbx)
    Vby_m_s, Vby_s_s = scaler_1d(logVby)

    logT_norm   = (logT_global  - logT_m) / logT_s
    logUx_norm  = (logUx - Ux_m_s)  / Ux_s_s
    logUy_norm  = (logUy - Uy_m_s)  / Uy_s_s
    logDx_norm  = (logDx - Dx_m_s)  / Dx_s_s
    logDy_norm  = (logDy - Dy_m_s)  / Dy_s_s
    logVbx_norm = (logVbx - Vbx_m_s) / Vbx_s_s
    logVby_norm = (logVby - Vby_m_s) / Vby_s_s

    return {
        'X_mean': X_m, 'X_std': X_s,
        'logT_mean': logT_m, 'logT_std': logT_s, 'logT_norm': logT_norm,
        'logUx_norm': logUx_norm, 'Ux_mean': Ux_m_s, 'Ux_std': Ux_s_s,
        'logUy_norm': logUy_norm, 'Uy_mean': Uy_m_s, 'Uy_std': Uy_s_s,
        'logDx_norm': logDx_norm, 'Dx_mean': Dx_m_s, 'Dx_std': Dx_s_s,
        'logDy_norm': logDy_norm, 'Dy_mean': Dy_m_s, 'Dy_std': Dy_s_s,
        'logVbx_norm': logVbx_norm, 'Vbx_mean': Vbx_m_s, 'Vbx_std': Vbx_s_s,
        'logVby_norm': logVby_norm, 'Vby_mean': Vby_m_s, 'Vby_std': Vby_s_s,
    }


def make_dataloaders(idx_tr, idx_val, idx_test, scalers):
    N_tr    = N_pisos_arr[idx_tr]
    counts  = np.bincount(N_tr, minlength=N_MAX_PISOS+1).astype(float)
    counts[counts == 0] = 1.0
    weights = 1.0 / counts[N_tr]
    sampler = WeightedRandomSampler(weights, num_samples=len(idx_tr), replacement=True)

    dl_tr  = DataLoader(ModalDataset(idx_tr,   scalers), batch_size=BATCH_SIZE,
                        sampler=sampler, drop_last=True)
    dl_val = DataLoader(ModalDataset(idx_val,  scalers), batch_size=BATCH_SIZE, shuffle=False)
    dl_tst = DataLoader(ModalDataset(idx_test, scalers), batch_size=BATCH_SIZE, shuffle=False)
    return dl_tr, dl_val, dl_tst


def build_model():
    return PINNModal_v4(N_INPUTS, N_MODOS, N_MAX_PISOS,
                        hidden=256, hidden_T1=128).to(device)


print('CELDA 6 OK — v16 ratio + scaler_pisos corregido')
print('  scaler_pisos: sobre pisos activos (corrección bug)')
print('  + ratio_dx_dy = drift_x / clip(drift_y, 1e-6)')

CELDA 6 OK — v16 ratio + scaler_pisos corregido
  scaler_pisos: sobre pisos activos (corrección bug)
  + ratio_dx_dy = drift_x / clip(drift_y, 1e-6)


## 📌 Celda 7 — Curriculum de entrenamiento v20 (definitivo)

### 🎯 ¿Qué hace esta celda?

Define el **currículo de aprendizaje** (Bengio et al., 2009) que **activa progresivamente** los términos de la pérdida a lo largo del entrenamiento, mediante la función `get_lambdas_and_active(epoch)`. Empieza solo con términos data-driven y va incorporando los físicos (φ, eigen, ortogonalidad) por etapas.

### 🗂️ Etapas (Fase 1: `EP_TOTAL = 600` épocas)

| Épocas | Etapa | Activa además de los data |
|---|---|---|
| 0 – 149 | E0 (frozen) | solo `data_resp`, `data_T_rest`, `ratio`; **encoder_T1 congelado** |
| 150 – 199 | S1 | + `data_T1`, + `phi` ($\lambda = 0.1$) |
| 200 – 349 | S2 | + `T1`, + `eig` ($\lambda = 0.1$) |
| 350 – 499 | S3 | + `ortho` ($\lambda = 10^{-5}$) |
| 500 – 599 | S4 | igual que S3 (consolidación) |

### 🧮 Pesos $\lambda$ por etapa

En todas las etapas: `data_resp = 2.0`, `data_T_rest = 1.0`, `ratio = 1.0`. Los términos `data_T1`, `T1`, `phi`, `eig` y `ortho` se encienden según la tabla anterior. Las banderas `stage_active` (`phi`, `eig`, `ortho`) controlan qué términos físicos se computan en cada época.

> **Nota:** durante las primeras 150 épocas se desactivan los gradientes del `encoder_T1` (entrenamiento en dos velocidades). El optimizador usa `CosineAnnealingWarmRestarts` (Loshchilov & Hutter, 2017).

### ✅ Lo que se obtiene

- Función `get_lambdas_and_active(epoch)` que devuelve el diccionario de pesos $\lambda$ y los flags `stage_active` para esa época.
- Constante `EP_TOTAL = 600` (Fase 1). La Fase 2 puede usar un horizonte distinto.

In [9]:
# CELDA 7 -- CURRICULUM v20
EP_TOTAL = 600

def get_lambdas_and_active(epoch):
    if epoch < 150:
        return (
            {'data_resp':2.0,'data_T_rest':1.0,'data_T1':0.0,
             'T1':0.0,'phi':0.0,'eig':0.0,'ortho':0.0,'ratio':1.0},
            {'phi':False,'eig':False,'ortho':False}
        )
    elif epoch < 200:
        return (
            {'data_resp':2.0,'data_T_rest':1.0,'data_T1':1.0,
             'T1':0.0,'phi':0.1,'eig':0.0,'ortho':0.0,'ratio':1.0},
            {'phi':True,'eig':False,'ortho':False}
        )
    elif epoch < 350:
        return (
            {'data_resp':2.0,'data_T_rest':1.0,'data_T1':1.0,
             'T1':1.0,'phi':0.1,'eig':0.1,'ortho':0.0,'ratio':1.0},
            {'phi':True,'eig':True,'ortho':False}
        )
    elif epoch < 500:
        return (
            {'data_resp':2.0,'data_T_rest':1.0,'data_T1':1.0,
             'T1':1.0,'phi':0.1,'eig':0.1,'ortho':1e-5,'ratio':1.0},
            {'phi':True,'eig':True,'ortho':True}
        )
    else:
        return (
            {'data_resp':2.0,'data_T_rest':1.0,'data_T1':1.0,
             'T1':1.0,'phi':0.1,'eig':0.1,'ortho':1e-5,'ratio':1.0},
            {'phi':True,'eig':True,'ortho':True}
        )

STAGES = None
print(f'CELDA 7 OK -- Curriculum v20')
print(f'  EP_TOTAL : {EP_TOTAL}')

CELDA 7 OK -- Curriculum v20
  EP_TOTAL : 600


## 📌 Celda 8 — Funciones de entrenamiento y métricas OOD (v20)

### 🎯 ¿Qué hace esta celda?

Define **todas las funciones núcleo** del experimento:
1. Análisis de los dominios A vs B.
2. Verificación de **diversidad** entre conjuntos.
3. Bucle de entrenamiento con staged training.
4. Cálculo de **métricas OOD** finales.

### 🔍 1. `tabla_dominios(trial_def, idx_pool_A, idx_pool_B)`

Para cada feature, calcula la **distancia normalizada** entre las medias de Pool A y Pool B:

$$
\Delta_{\text{norm}} = \frac{|\bar{x}_A - \bar{x}_B|}{\max(\sigma_A, \sigma_B, \epsilon)}
$$

Si $\Delta_{\text{norm}} > 0.3\sigma$, la feature **difiere significativamente** entre dominios. También calcula valores exclusivos (categorías presentes solo en A o solo en B) y estadísticas de la respuesta sísmica ($T_1$, $V_{b,x}$).

### 📊 2. `verificar_diversidad(idx_A1, idx_A2, idx_pool_B)`

Calcula **distancias mínimas euclidianas** en el espacio normalizado de features:

$$
d_{\min}(\mathcal{S}) = \min_{i \neq j \in \mathcal{S}} \|\tilde{x}_i - \tilde{x}_j\|_2
$$

- $d_{\min}^{A_1}$: diversidad dentro de A1 (debería ser > 1.0).
- $d_{\min}^{A \leftrightarrow B}$: **separación** entre A y B (debería ser > 1.0).

### 🏋️ 3. `train_trial(model, dl_tr, dl_val, scalers, trial_dir, ep_total, verbose)`

Loop de entrenamiento:

```
para cada época ep en [0, ep_total):
    1. Obtener lambdas y stages activos de get_lambdas_and_active(ep)
    2. Congelar/descongelar encoder_T1 según ep ⋚ 150
    3. TRAIN:
         ▸ forward → out = model(X, mask)
         ▸ loss, ld = compute_loss(batch, out, lambdas, scalers, active)
         ▸ backward + clip_grad_norm(max_norm=1.0)
         ▸ optimizer.step()
    4. VAL:
         ▸ forward sin gradientes
         ▸ acumular val_losses
         ▸ val_total = Σ λ_k * val_loss_k
    5. scheduler.step()  (Cosine warm restarts)
    6. Si val_total < best Y ep ≥ 250:
         ▸ guardar best_model.pt
```

> **Clip de gradientes**: `clip_grad_norm_(parameters, max_norm=1.0)` evita explosiones cuando los términos físicos se activan de golpe.

### 📏 4. `compute_metrics_ood(model, dl_ood, scalers)`

Calcula sobre el conjunto **fuera de distribución** (OOD = Pool B test):

#### 4.1 MAC promedio sobre 3 direcciones
$$
\text{MAC}_{\text{avg}} = \frac{1}{3}\left(\text{MAC}_{Y_1} + \text{MAC}_{T_1} + \text{MAC}_{X_1}\right)
$$
donde el slot 0 es Y, slot 1 es T (torsional), slot 2 es X.

#### 4.2 MAPE del periodo fundamental
$$
\text{T1\_MAPE} = 100 \cdot \mathbb{E}\!\left[\frac{|T_1^{\text{pred}} - T_1^{\text{true}}|}{T_1^{\text{true}} + \epsilon}\right]
$$

#### 4.3 MAPE del corte basal (promedio X-Y)
$$
\text{Vb\_MAPE} = \tfrac{1}{2}\big(\text{Vbx\_MAPE} + \text{Vby\_MAPE}\big)
$$

#### 4.4 MAPE de la deriva máxima
$$
\delta_{\max} = \max_{i, d} \delta_{d,i}, \qquad d \in \{x,y\}, \; i \in \{1,...,N_p\}
$$

> **Desnormalización**: se aplica $T = \exp(\widetilde{\log T} \cdot \sigma + \mu)$ antes de calcular MAPE.

### ✅ Lo que se obtiene

- 4 funciones globales: `tabla_dominios`, `verificar_diversidad`, `train_trial`, `compute_metrics_ood`.
- Patrón fijo de direcciones modales: `DIR_MODO = [Y, T, X, Y, T, X, ...]`.

In [10]:
# CELDA 8 -- FUNCIONES TRAIN Y METRICAS 

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

_scaler_div = StandardScaler()
_X_norm_div = _scaler_div.fit_transform(X_clean)

DIR_MODO = []
for r in range(18):
    ciclo = r % 3
    if ciclo == 0:   DIR_MODO.append('Y')
    elif ciclo == 1: DIR_MODO.append('T')
    else:            DIR_MODO.append('X')


def tabla_dominios(trial_def, idx_pool_A, idx_pool_B):
    rng_v  = np.random.default_rng(42)
    n_mues = min(500, len(idx_pool_A), len(idx_pool_B))
    idx_Am = rng_v.choice(idx_pool_A, n_mues, replace=False)
    idx_Bm = rng_v.choice(idx_pool_B, n_mues, replace=False)

    print(f"\n{'='*65}")
    print(f"TRIAL {trial_def['id']:02d} — {trial_def['nombre']}")
    print(f"  Pool A (TRAIN): {len(idx_pool_A)} casos")
    print(f"  Pool B (OOD)  : {len(idx_pool_B)} casos")

    difs = []
    for col in COLS_BASE:
        idx_c  = COLS_BASE.index(col)
        vA     = X_clean[idx_Am, idx_c]
        vB     = X_clean[idx_Bm, idx_c]
        sigma  = max(vA.std(), vB.std(), 1e-8)
        d_norm = abs(vA.mean() - vB.mean()) / sigma
        if d_norm > 0.3:
            difs.append((col, vA, vB, d_norm))
    difs.sort(key=lambda x: x[3], reverse=True)

    print(f"\n  FEATURES QUE DIFIEREN ENTRE DOMINIOS (Δ > 0.3σ):")
    print(f"  De {len(COLS_BASE)} features: {len(difs)} difieren significativamente")
    if difs:
        print(f"  {'Feature':<22} {'Pool A':>28} {'Pool B':>28} {'Δ':>6}")
        print(f"  {'-'*88}")
        for col, vA, vB, d in difs:
            signo  = '↑' if vB.mean() > vA.mean() else '↓'
            u_A    = sorted(set(np.round(vA, 2).tolist()))
            u_B    = sorted(set(np.round(vB, 2).tolist()))
            str_A  = str([round(v,2) for v in u_A]) if len(u_A)<=8 \
                     else f"[{vA.min():.2f}→{vA.max():.2f}] media={vA.mean():.2f}"
            str_B  = str([round(v,2) for v in u_B]) if len(u_B)<=8 \
                     else f"[{vB.min():.2f}→{vB.max():.2f}] media={vB.mean():.2f}"
            print(f"  {col:<22} {str_A:>28} {str_B:>28}  {signo}{d:.2f}σ")

    print(f"\n  RESPUESTA SÍSMICA:")
    for arr, nombre in [(T_r[:,0],'T1 [s]'), (Vb_x,'Vbx [kN]')]:
        vA = arr[idx_Am]; vB = arr[idx_Bm]
        print(f"  {nombre:<12} Pool A: [{vA.min():.2f}→{vA.max():.2f}] "
              f"media={vA.mean():.2f}   "
              f"Pool B: [{vB.min():.2f}→{vB.max():.2f}] "
              f"media={vB.mean():.2f}")
    print(f"{'='*65}")


def verificar_diversidad(idx_A1, idx_A2, idx_pool_B):
    rng_v = np.random.default_rng(42)
    n_m   = 300
    mA1   = rng_v.choice(idx_A1,     min(n_m, len(idx_A1)),     replace=False)
    mA2   = rng_v.choice(idx_A2,     min(n_m, len(idx_A2)),     replace=False)
    mB    = rng_v.choice(idx_pool_B, min(n_m, len(idx_pool_B)), replace=False)

    def min_dist_interna(idx_m):
        D = pairwise_distances(_X_norm_div[idx_m])
        np.fill_diagonal(D, np.inf)
        return D[D != np.inf].min()

    min_A1 = min_dist_interna(mA1)
    min_A2 = min_dist_interna(mA2)
    min_B  = min_dist_interna(mB)
    min_AB = pairwise_distances(_X_norm_div[mA1], _X_norm_div[mB]).min()

    def est(v): return '✅' if v > 1.0 else '⚠️'

    print(f"\n  DIVERSIDAD (dist_min espacio normalizado, meta > 1.0):")
    print(f"  A1={min_A1:.3f}{est(min_A1)}  "
          f"A2={min_A2:.3f}{est(min_A2)}  "
          f"B={min_B:.3f}{est(min_B)}  "
          f"| Sep A↔B={min_AB:.3f}{est(min_AB)}")

    return {
        'min_A1': round(min_A1, 4),
        'min_A2': round(min_A2, 4),
        'min_B' : round(min_B,  4),
        'min_AB': round(min_AB, 4),
    }


def train_trial(model, dl_tr, dl_val, scalers, trial_dir,
                ep_total=600, verbose=True):
    trial_dir.mkdir(parents=True, exist_ok=True)
    history         = []
    best_path       = trial_dir / 'best_model.pt'
    best_val_global = float('inf')
    best_epoch      = 0

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = CosineAnnealingWarmRestarts(
        optimizer, T_0=200, T_mult=2, eta_min=1e-6)

    params_T1 = (list(model.encoder_T1.parameters()) +
                 list(model.head_T1.parameters()))

    etapa_actual = -1
    stage_actual = -1

    for ep in range(ep_total):
        lambdas, active = get_lambdas_and_active(ep)

        if ep < 150:
            if etapa_actual != 0:
                etapa_actual = 0
                for p in params_T1:
                    p.requires_grad = False
                if verbose:
                    print(f'\n  ── Etapa 0 (ep {ep}) encoder_T1 CONGELADO ──')
        else:
            if etapa_actual != 1:
                etapa_actual = 1
                for p in params_T1:
                    p.requires_grad = True
                if verbose:
                    print(f'\n  ── Etapa 1 (ep {ep}) encoder_T1 DESCONGELADO ──')

        stage_nuevo = {150:1, 200:2, 350:3, 500:4}.get(ep, stage_actual)
        if stage_nuevo != stage_actual and ep >= 150:
            stage_actual = stage_nuevo
            if verbose:
                print(f'\n  ── Stage {stage_actual} (ep {ep}) '
                      f'T1λ={lambdas["T1"]} '
                      f'eig={lambdas["eig"]} '
                      f'lr={optimizer.param_groups[0]["lr"]:.2e} ──')

        model.train()
        tr_losses = {}
        for batch in dl_tr:
            optimizer.zero_grad()
            out  = model(batch['X'].to(device), batch['mask'].to(device))
            loss, ld = compute_loss(batch, out, lambdas, scalers, active)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            for k, v in ld.items():
                tr_losses[k] = tr_losses.get(k, 0) + v

        model.eval()
        val_losses = {}
        with torch.no_grad():
            for batch in dl_val:
                out  = model(batch['X'].to(device), batch['mask'].to(device))
                _, ld = compute_loss(batch, out, lambdas, scalers, active)
                for k, v in ld.items():
                    val_losses[k] = val_losses.get(k, 0) + v

        val_total = sum(lambdas.get(k, 0) * v
                        for k, v in val_losses.items())
        scheduler.step()

        if val_total < best_val_global and ep >= 250:
            best_val_global = val_total
            torch.save(model.state_dict(), best_path)
            best_epoch = ep

        row = {'etapa': etapa_actual, 'stage': stage_actual,
               'epoch': ep, 'val_total': val_total,
               'lr': optimizer.param_groups[0]['lr']}
        row.update({f'tr_{k}': v/max(len(dl_tr), 1)
                    for k, v in tr_losses.items()})
        row.update({f'val_{k}': v/max(len(dl_val), 1)
                    for k, v in val_losses.items()})
        history.append(row)

        if verbose and ep % 25 == 0:
            frozen_str = 'T1_frozen' if ep < 150 else 'T1_free'
            print(f'  ep{ep:4d}/{ep_total} '
                  f'val={val_total:.5f} '
                  f'best={best_val_global:.5f} (ep{best_epoch}) '
                  f'lr={optimizer.param_groups[0]["lr"]:.2e} '
                  f'[{frozen_str}]')

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(trial_dir / 'history.csv', index=False)
    return hist_df, best_val_global, best_epoch


def compute_metrics_ood(model, dl_ood, scalers):
    model.eval()

    logT_std_t  = torch.tensor(scalers['logT_std'],  dtype=torch.float32, device=device)
    logT_mean_t = torch.tensor(scalers['logT_mean'], dtype=torch.float32, device=device)
    Vbx_std_t   = torch.tensor([scalers['Vbx_std']], dtype=torch.float32, device=device)
    Vbx_mean_t  = torch.tensor([scalers['Vbx_mean']],dtype=torch.float32, device=device)
    Vby_std_t   = torch.tensor([scalers['Vby_std']], dtype=torch.float32, device=device)
    Vby_mean_t  = torch.tensor([scalers['Vby_mean']],dtype=torch.float32, device=device)
    Dx_std_t    = torch.tensor([scalers['Dx_std']],  dtype=torch.float32, device=device)
    Dx_mean_t   = torch.tensor([scalers['Dx_mean']], dtype=torch.float32, device=device)
    Dy_std_t    = torch.tensor([scalers['Dy_std']],  dtype=torch.float32, device=device)
    Dy_mean_t   = torch.tensor([scalers['Dy_mean']], dtype=torch.float32, device=device)

    MAC_Y1_list=[]; MAC_T1_list=[]; MAC_X1_list=[]
    T1_pred_list=[]; T1_true_list=[]
    Vbx_pred_list=[]; Vbx_true_list=[]
    Vby_pred_list=[]; Vby_true_list=[]
    drift_max_pred_list=[]; drift_max_true_list=[]

    with torch.no_grad():
        for batch in dl_ood:
            X_b    = batch['X'].to(device)
            mask_b = batch['mask'].to(device)

            logT_p, Phi_x_p, Phi_y_p, Phi_t_p, \
                Ux_p, Uy_p, dx_p, dy_p, Vb_p, ratio_p = model(X_b, mask_b)   # ← 10 outputs

            for slot, MAC_list, Phi_p, key in [
                (0, MAC_Y1_list, Phi_y_p, 'Phi_y'),
                (1, MAC_T1_list, Phi_t_p, 'Phi_theta'),
                (2, MAC_X1_list, Phi_x_p, 'Phi_x'),
            ]:
                phi_pred = Phi_p[:, :, slot] * mask_b
                phi_true = batch[key][:, :, slot].to(device) * mask_b
                num   = (phi_pred * phi_true).sum(dim=1) ** 2
                den_p = (phi_pred ** 2).sum(dim=1)
                den_t = (phi_true ** 2).sum(dim=1)
                den   = (den_p * den_t).clamp(min=1e-12)
                valido= (den_t > 1e-10)
                mac_r = torch.where(valido, num/den, torch.zeros_like(num))
                MAC_list.extend(mac_r[valido].cpu().numpy().tolist())

            T_pred  = torch.exp(logT_p * logT_std_t + logT_mean_t)
            T_true  = batch['T_r'].to(device)
            mask_T1 = T_true[:, 0] > 1e-6
            if mask_T1.sum() > 0:
                T1_pred_list.extend(T_pred[mask_T1, 0].cpu().numpy().tolist())
                T1_true_list.extend(T_true[mask_T1, 0].cpu().numpy().tolist())

            Vbx_pred = torch.exp(Vb_p[:,0]*Vbx_std_t + Vbx_mean_t)
            Vby_pred = torch.exp(Vb_p[:,1]*Vby_std_t + Vby_mean_t)
            Vbx_pred_list.extend(Vbx_pred.squeeze().cpu().numpy().tolist())
            Vbx_true_list.extend(batch['Vbx_raw'].cpu().numpy().tolist())
            Vby_pred_list.extend(Vby_pred.squeeze().cpu().numpy().tolist())
            Vby_true_list.extend(batch['Vby_raw'].cpu().numpy().tolist())

            # ← dy físico desde su scaler; dx = ratio × dy en escala física
            dy_real = torch.exp(dy_p*Dy_std_t+Dy_mean_t).clamp(min=1e-10) * mask_b   # ←
            dx_real = (ratio_p * dy_real) * mask_b                                   # ←
            drift_max_pred = torch.maximum(dx_real, dy_real).max(dim=1).values

            drift_x_raw    = batch['drift_x_raw'].to(device) * mask_b
            drift_y_raw    = batch['drift_y_raw'].to(device) * mask_b
            drift_max_true = torch.maximum(drift_x_raw, drift_y_raw).max(dim=1).values
            mask_drift     = drift_max_true > 1e-8
            if mask_drift.sum() > 0:
                drift_max_pred_list.extend(drift_max_pred[mask_drift].cpu().numpy().tolist())
                drift_max_true_list.extend(drift_max_true[mask_drift].cpu().numpy().tolist())

    MAC_Y1  = float(np.mean(MAC_Y1_list))  if MAC_Y1_list  else 0.0
    MAC_T1  = float(np.mean(MAC_T1_list))  if MAC_T1_list  else 0.0
    MAC_X1  = float(np.mean(MAC_X1_list))  if MAC_X1_list  else 0.0
    MAC_avg = (MAC_Y1 + MAC_T1 + MAC_X1) / 3.0

    T1_pred_a = np.array(T1_pred_list); T1_true_a = np.array(T1_true_list)
    T1_MAPE   = float(np.mean(np.abs(T1_pred_a-T1_true_a)/(T1_true_a+1e-8))*100)

    Vbx_pred_a = np.array(Vbx_pred_list); Vbx_true_a = np.array(Vbx_true_list)
    Vby_pred_a = np.array(Vby_pred_list); Vby_true_a = np.array(Vby_true_list)
    Vbx_MAPE   = float(np.mean(np.abs(Vbx_pred_a-Vbx_true_a)/(Vbx_true_a+1e-8))*100)
    Vby_MAPE   = float(np.mean(np.abs(Vby_pred_a-Vby_true_a)/(Vby_true_a+1e-8))*100)
    Vb_MAPE    = (Vbx_MAPE + Vby_MAPE) / 2.0

    drift_pred_a = np.array(drift_max_pred_list)
    drift_true_a = np.array(drift_max_true_list)
    drift_MAPE   = float(np.mean(
        np.abs(drift_pred_a-drift_true_a)/(drift_true_a+1e-8))*100)

    return {
        'MAC_avg'    : round(MAC_avg,   6),
        'MAC_Y1'     : round(MAC_Y1,    6),
        'MAC_T1'     : round(MAC_T1,    6),
        'MAC_X1'     : round(MAC_X1,    6),
        'T1_MAPE'    : round(T1_MAPE,   4),
        'Vb_MAPE'    : round(Vb_MAPE,   4),
        'Vbx_MAPE'   : round(Vbx_MAPE,  4),
        'Vby_MAPE'   : round(Vby_MAPE,  4),
        'drift_MAPE' : round(drift_MAPE, 4),
        'n_casos_ood': len(T1_pred_list),
    }


print('CELDA 8 OK — v20')

CELDA 8 OK — v20


## 📌 Celda 9 — Definición de los 31 trials OOD

### 🎯 ¿Qué hace esta celda?

Define los **31 escenarios experimentales** que se evaluarán. Cada trial divide el dataset en dos *pools* (A y B) según un criterio físico/estructural, permitiendo medir cómo **generaliza el PINN ante un *distributional shift*** (desplazamiento de distribución).

### 🗂️ Estructura de cada trial

```python
{
    'id'    : int,                 # identificador del trial
    'nombre': str,                  # descripción corta
    'mask_A': boolean array (N_cases,),   # define Pool A (TRAIN)
    'mask_B': boolean array (N_cases,),   # define Pool B (OOD)
    'tipo'  : 'conceptual' | 'complemento' | 'aleatorio_puro'
}
```

### 📚 Los 6 grupos de trials

| Grupo | Trials | Eje de distributional shift |
|---|---|---|
| **G1 — Nº de pisos** | T01–T05 | Edificios bajos vs altos |
| **G2 — Material** | T06–T10 | $f_c$, espesor de muros, $g_k$, tipo de suelo |
| **G3 — Geometría** | T11–T15 | Dimensiones de planta y núcleo |
| **G4 — Sísmica** | T16–T20 | Zona sísmica, tipo de suelo NCh 433 |
| **G5 — Tipología** | T21–T25 | Combinaciones de tipos de edificio |
| **G6 — Combinados** | T26–T29 | Mezclas mixtas |
| **Línea base** | T30, T31 | Aleatorio vs complemento, aleatorio puro |

### 🎯 Trials destacados (ejemplos)

| ID | Nombre | Pool A | Pool B |
|---|---|---|---|
| T02 | `N_bajo_vs_alto` | $N_p \leq 10$ | $N_p \geq 13$ |
| T10 | `suelo_AB_vs_CD` | Suelos rígidos (A,B) | Suelos blandos (C,D) |
| T16 | `zona_1_3_vs_zona_2` | Zonas 1 y 3 | Zona 2 |
| T20 | `fc_25_30_vs_fc_35_40` | $f_c \in \{25, 30\}$ MPa | $f_c \in \{35, 40\}$ MPa |
| T31 | **`aleatorio_puro`** | 5000 al azar | 5000 al azar | ← **línea base** sin shift |

### 🏛️ Justificación ingenieril (norma NCh 433)

- **Trial 10 (suelo AB vs CD)**: el tipo de suelo afecta $S_a(T)$ — espectro de aceleración. Suelos blandos (C, D) amplifican períodos largos.
- **Trial 16 (zona 1_3 vs 2)**: la zona sísmica determina $A_0$ (aceleración máxima efectiva). Zona 2 cubre la mayor parte de Chile central.
- **Trial 20 (fc bajo vs alto)**: $E = 4700\sqrt{f_c}$ → cambia rigidez $\mathbf{K}$ → cambia $T_r$.

### 🔄 Trials de reemplazo (v3)

Los trials originales T05, T10, T20, T26, T28, T29 tenían pools insuficientes (< 1600 casos). Se reemplazaron por criterios ingenierilmente significativos manteniendo factibilidad.

### ✅ Verificación de factibilidad

Cada trial debe cumplir:
$$
|\text{Pool A}| \geq 2N_{\text{train}} = 2000, \qquad |\text{Pool B}| \geq K_{\text{reps}}(N_{\text{val}} + N_{\text{test}}) = 1600
$$

Si no, se reducen las repeticiones a $K_{\text{reps}} = 2$.

### ✅ Lo que se obtiene

- Lista `TRIALS_DEF` con los 31 diccionarios.
- Tabla impresa de factibilidad (Pool A, Pool B, ¿ok para 4 reps?).

In [11]:
# CELDA 9 -- DEFINICION DE LOS 31 TRIALS OOD
# v3: reemplazo de 6 trials con pools insuficientes

N_p  = N_pisos_arr
fc   = X_clean[:, COLS_BASE.index('fc_MPa')]
h_s  = X_clean[:, COLS_BASE.index('h_story_m')]
n_u  = X_clean[:, COLS_BASE.index('n_unid_lado')]
L_m  = X_clean[:, COLS_BASE.index('L_mod_m')]
t_n  = X_clean[:, COLS_BASE.index('t_muro_nucleo_m')]
t_b  = X_clean[:, COLS_BASE.index('t_muro_borde_m')]
t_m  = X_clean[:, COLS_BASE.index('t_muro_mid_m')]
gk   = X_clean[:, COLS_BASE.index('gk_kN_m2')]
Ln   = X_clean[:, COLS_BASE.index('L_nucleo_m')]
Bn   = X_clean[:, COLS_BASE.index('B_nucleo_m')]
pd_  = X_clean[:, COLS_BASE.index('prof_depto_m')]
ac   = X_clean[:, COLS_BASE.index('ancho_corredor_m')]
s_A  = X_clean[:, COLS_BASE.index('suelo_A')]
s_C  = X_clean[:, COLS_BASE.index('suelo_C')]
s_B  = X_clean[:, COLS_BASE.index('suelo_B')]
s_D  = X_clean[:, COLS_BASE.index('suelo_D')]
z_1  = X_clean[:, COLS_BASE.index('zona_1')]
z_2  = X_clean[:, COLS_BASE.index('zona_2')]
z_3  = X_clean[:, COLS_BASE.index('zona_3')]
B2   = X_clean[:, COLS_BASE.index('activar_B2')]

Ln_med = float(np.median(Ln)); Bn_med = float(np.median(Bn))
pd_med = float(np.median(pd_)); tn_med = float(np.median(t_n))
tb_med = float(np.median(t_b))

TRIALS_DEF = [
    # ── Grupo 1: N_pisos ─────────────────────────────────────────────────
    {'id':1,  'nombre':'N_par_vs_impar',
     'mask_A':N_p%2==0,             'mask_B':N_p%2!=0},
    {'id':2,  'nombre':'N_bajo_vs_alto',
     'mask_A':N_p<=10,              'mask_B':N_p>=13},
    {'id':3,  'nombre':'N_6a9_vs_14a18',
     'mask_A':N_p<=9,               'mask_B':N_p>=14},
    {'id':4,  'nombre':'N_6_7_8_vs_15_16_17_18',
     'mask_A':N_p<=8,               'mask_B':N_p>=15},
    # T05 REEMPLAZO: gk_60_65_vs_70_75
    # Original (N_par_bajo_vs_impar_alto) tenía Pool_B=2289 insuficiente
    # Nuevo: carga gravitacional baja vs alta — afecta masa y por tanto T y Vb
    {'id':5,  'nombre':'gk_60_65_vs_70_75',
     'mask_A':gk<=6.5,              'mask_B':gk>=7.0},
    # ── Grupo 2: Material ─────────────────────────────────────────────────
    {'id':6,  'nombre':'fc_25_35_vs_30_40',
     'mask_A':(fc==25)|(fc==35),    'mask_B':(fc==30)|(fc==40)},
    {'id':7,  'nombre':'t_nucleo_fino_vs_grueso',
     'mask_A':t_n<=tn_med,          'mask_B':t_n>tn_med},
    {'id':8,  'nombre':'t_borde_fino_vs_grueso',
     'mask_A':t_b<=tb_med,          'mask_B':t_b>tb_med},
    {'id':9,  'nombre':'gk_bajo_vs_alto',
     'mask_A':gk<=6.5,              'mask_B':gk>=7.0},
    # T10 REEMPLAZO: suelo_AB_vs_suelo_CD
    # Original (fc_bajo_N_bajo_vs_fc_alto_N_alto) tenía Pool_A=1888 insuficiente
    # Nuevo: suelos rígidos (A,B) vs blandos (C,D) — afecta Sa(T) en NCh433
    {'id':10, 'nombre':'suelo_AB_vs_suelo_CD',
     'mask_A':(s_A==1)|(s_B==1),    'mask_B':(s_C==1)|(s_D==1)},
    # ── Grupo 3: Geometría ────────────────────────────────────────────────
    {'id':11, 'nombre':'n_unid_bajo_vs_alto',
     'mask_A':n_u<=3,               'mask_B':n_u>=5},
    {'id':12, 'nombre':'L_mod_bajo_vs_alto',
     'mask_A':L_m<=3.4,             'mask_B':L_m>=3.6},
    {'id':13, 'nombre':'L_nucleo_corto_vs_largo',
     'mask_A':Ln<Ln_med,            'mask_B':Ln>Ln_med},
    {'id':14, 'nombre':'B_nucleo_pequeno_vs_grande',
     'mask_A':Bn<Bn_med,            'mask_B':Bn>Bn_med},
    {'id':15, 'nombre':'prof_depto_bajo_vs_alto',
     'mask_A':pd_<pd_med,           'mask_B':pd_>pd_med},
    # ── Grupo 4: Sísmica ──────────────────────────────────────────────────
    {'id':16, 'nombre':'zona_1_3_vs_zona_2',
     'mask_A':(z_1==1)|(z_3==1),    'mask_B':z_2==1},
    {'id':17, 'nombre':'suelo_AC_vs_BD',
     'mask_A':(s_A==1)|(s_C==1),    'mask_B':(s_B==1)|(s_D==1)},
    {'id':18, 'nombre':'t_nucleo_min_vs_max',
     'mask_A':t_n<=np.percentile(t_n,25), 'mask_B':t_n>=np.percentile(t_n,75)},
    {'id':19, 'nombre':'zona_1_vs_zona_3',
     'mask_A':z_1==1,               'mask_B':z_3==1},
    # T20 REEMPLAZO: fc_25_30_vs_fc_35_40
    # Original (suelo_A_vs_suelo_D) tenía Pool_B=2503 insuficiente
    # Nuevo: fc bajo vs alto — afecta E=4700√fc y por tanto rigidez y periodos
    {'id':20, 'nombre':'fc_25_30_vs_fc_35_40',
     'mask_A':(fc==25)|(fc==30),    'mask_B':(fc==35)|(fc==40)},
    # ── Grupo 5: Tipología ────────────────────────────────────────────────
    {'id':21, 'nombre':'n_unid_pequeno_vs_grande',
     'mask_A':n_u<=2,               'mask_B':n_u>=4},
    {'id':22, 'nombre':'t_borde_min_vs_max',
     'mask_A':t_b<=np.percentile(t_b,25), 'mask_B':t_b>=np.percentile(t_b,75)},
    {'id':23, 'nombre':'h_bajo_vs_h_alto',
     'mask_A':h_s<=np.percentile(h_s,33), 'mask_B':h_s>=np.percentile(h_s,67)},
    {'id':24, 'nombre':'t_mid_min_vs_max',
     'mask_A':t_m<=np.percentile(t_m,25), 'mask_B':t_m>=np.percentile(t_m,75)},
    {'id':25, 'nombre':'corredor_angosto_vs_ancho',
     'mask_A':ac<=np.percentile(ac,33),   'mask_B':ac>=np.percentile(ac,67)},
    # ── Grupo 6: Combinados ───────────────────────────────────────────────
    # T26 REEMPLAZO: t_nucleo_018_020_vs_028_030
    # Original (N_par_fc_bajo_vs_N_impar_fc_alto) tenía Pool_A=2670 insuficiente
    # Nuevo: espesor núcleo delgado vs grueso — afecta rigidez torsional directamente
    {'id':26, 'nombre':'t_nucleo_018_020_vs_028_030',
     'mask_A':t_n<=0.20,            'mask_B':t_n>=0.28},
    {'id':27, 'nombre':'n_unid_2_3_vs_4_5',
     'mask_A':(n_u==2)|(n_u==3),    'mask_B':(n_u==4)|(n_u==5)},
    # T28 REEMPLAZO: corredor_angosto_vs_ancho (percentiles)
    # Original (L_mod_extremo_bajo_vs_alto) tenía Pool_B=3429 — casi suficiente
    # Nuevo: corredor afecta distribución de masa en Y — variable no explorada en extremos
    {'id':28, 'nombre':'corredor_angosto_p33_vs_ancho_p67',
     'mask_A':ac<=np.percentile(ac,33),   'mask_B':ac>=np.percentile(ac,67)},
    # T29 REEMPLAZO: fc_alto_vs_gk_alto
    # Original (h_story_min_vs_max) tenía Pool_B=2519 insuficiente
    # Nuevo: fc alto (alta rigidez) vs gk alto (alta masa) — separación mixta
    # que no existe en ningún otro trial — afecta ratio masa/rigidez
    {'id':29, 'nombre':'fc_alto_vs_gk_alto',
     'mask_A':fc>=35,               'mask_B':gk>=7.0},
    # ── Trial 30: Aleatorio vs complemento ───────────────────────────────
    {'id':30, 'nombre':'aleatorio_vs_complemento',
     'mask_A':None, 'mask_B':None, 'tipo':'complemento'},
    # ── Trial 31: Aleatorio puro — LÍNEA BASE ────────────────────────────
    {'id':31, 'nombre':'aleatorio_puro',
     'mask_A':None, 'mask_B':None, 'tipo':'aleatorio_puro'},
]

# ── Verificar factibilidad ────────────────────────────────────────────────
print(f"{'ID':<4} {'Nombre':<45} {'Pool_A':>7} {'Pool_B':>7} {'4_reps':>7}")
print("-"*72)

n_ok = 0
for t in TRIALS_DEF:
    tipo = t.get('tipo', 'conceptual')
    if tipo == 'complemento':
        n_A, n_B = 4000, 6000
    elif tipo == 'aleatorio_puro':
        n_A, n_B = 5000, 5000
    else:
        n_A = int(t['mask_A'].sum())
        n_B = int(t['mask_B'].sum())

    ok_4 = n_A >= MIN_POOL_A and n_B >= MIN_POOL_B
    flag = '✅' if ok_4 else '⚠️'
    if ok_4: n_ok += 1
    print(f"{t['id']:<4} {t['nombre']:<45} {n_A:>7} {n_B:>7}  {flag}")

print(f"\nTrials con 4 reps completas: {n_ok}/{len(TRIALS_DEF)}")
print(f"CELDA 9 OK — {len(TRIALS_DEF)} trials definidos")

ID   Nombre                                         Pool_A  Pool_B  4_reps
------------------------------------------------------------------------
1    N_par_vs_impar                                   5367    4633  ✅
2    N_bajo_vs_alto                                   3834    4637  ✅
3    N_6a9_vs_14a18                                   3100    3914  ✅
4    N_6_7_8_vs_15_16_17_18                           2318    3121  ✅
5    gk_60_65_vs_70_75                                4969    5031  ✅
6    fc_25_35_vs_30_40                                4989    5011  ✅
7    t_nucleo_fino_vs_grueso                          6650    3350  ✅
8    t_borde_fino_vs_grueso                           6005    3995  ✅
9    gk_bajo_vs_alto                                  4969    5031  ✅
10   suelo_AB_vs_suelo_CD                             4961    5039  ✅
11   n_unid_bajo_vs_alto                              4676    2981  ✅
12   L_mod_bajo_vs_alto                               4157    3438  ✅
13   L_nucle

## 📌 Celda 10 — Loop principal: 31 Trials × 4 Repeticiones

### 🎯 ¿Qué hace esta celda?

Es el **núcleo computacional** de la investigación. Ejecuta toda la metodología **Repeated Hold-Out OOD con Diseño Factorial 2×2** sobre los 31 trials definidos.

### 🔁 Flujo del loop principal

```
┌──────────────────────────────────────────────────────────────────────┐
│  PARA CADA trial t en TRIALS_DEF (31 trials):                        │
│  ┌────────────────────────────────────────────────────────────────┐  │
│  │  1. Construir Pool A (idx_pool_A) y Pool B (idx_pool_B)        │  │
│  │  2. Verificar factibilidad → ajustar n_train, k_reps           │  │
│  │  3. Partir Pool A en A1 y A2 (sin solapamiento)                │  │
│  │  4. Partir Pool B en 8 bloques (B1..B8)                        │  │
│  │  5. tabla_dominios()      → mostrar diferencias A vs B         │  │
│  │  6. verificar_diversidad() → distancias mínimas                │  │
│  │  7. Detectar variable clave (la de mayor Δ entre A y B)        │  │
│  │  ┌──────────────────────────────────────────────────────────┐  │  │
│  │  │  PARA CADA rep r en [1..4]:                              │  │  │
│  │  │     • train = A1 si r ≤ 2  ; A2 si r ≥ 3                 │  │  │
│  │  │     • val   = bloque_{2(r-1)}                               │  │
│  │  │     • test  = bloque_{2(r-1)+1}                             │  │
│  │  │     • scalers = compute_scalers(train)                      │  │
│  │  │     • dataloaders = make_dataloaders(...)                   │  │
│  │  │     • model = build_model()                                 │  │
│  │  │     • history = train_trial(model, ..., ep_total=600)    │  │  │
│  │  │     • cargar best_model.pt                                  │  │
│  │  │     • metrics = compute_metrics_ood(model, dl_test, ...) │  │  │
│  │  │     • guardar metricas.json                                 │  │
│  │  └──────────────────────────────────────────────────────────┘  │  │
│  │  8. Resumir: media, std, var_OOD, var_train sobre 4 reps       │  │
│  │  9. Guardar resumen.json del trial                             │  │
│  └────────────────────────────────────────────────────────────────┘  │
│  → Guardar all_results_v5.json con los 31 trials                     │
└──────────────────────────────────────────────────────────────────────┘
```

### 📐 Descomposición de varianza

Por cada métrica $\mu$ (MAC_avg, T1_MAPE, Vb_MAPE, drift_MAPE):

| Componente | Cálculo | Interpretación |
|---|---|---|
| Media | $\bar\mu = \tfrac{1}{4}\sum_{r=1}^4 \mu_r$ | Valor central |
| Desv. estándar | $s_\mu = \sqrt{\tfrac{1}{4}\sum (\mu_r - \bar\mu)^2}$ | Dispersión global |
| $\sigma_{\text{OOD}}$ | $\|\mu_1 - \mu_2\|$ | Variabilidad por dominio (mismo TRAIN) |
| $\sigma_{\text{train}}$ | $\|\mu_1 - \mu_3\|$ | Variabilidad por selección del entrenamiento |

### 💾 Estructura de archivos generada

```
TRIALS_DIR_V5/
├── trial_01_N_par_vs_impar/
│   ├── rep_01/
│   │   ├── best_model.pt
│   │   ├── history.csv
│   │   └── metricas.json
│   ├── rep_02/... rep_03/... rep_04/...
│   └── resumen.json
├── trial_02_N_bajo_vs_alto/
│   └── ...
├── ...
└── all_results_v5.json     ← ARCHIVO MAESTRO con los 31 resúmenes
```

### ⏱️ Costo computacional

- 31 trials × 4 reps × 600 épocas = **74 400 épocas** totales.
- Con batch_size=64 y dataset de ~1000 casos por rep → ~15 batches/época.
- En GPU moderna (RTX 30xx), aproximadamente **8–12 horas** totales.

### ✅ Lo que se obtiene

- Diccionario `all_results` en memoria con los 31 resúmenes.
- Archivo `all_results_v5.json` con todos los resultados serializados.
- Carpetas `trial_XX_*/rep_XX/` con modelos `.pt` e historiales `.csv`.

In [12]:
# ══════════════════════════════════════════════════════════════════════
# ABLATION V6 — (A) Arquitectura SIN encoder_T1 dedicado (filas 1-2)
# Misma capacidad/hidden que v4b; T1 sale del encoder compartido.
# ══════════════════════════════════════════════════════════════════════
class PINNModal_noT1enc(nn.Module):
    """Variante de PINNModal_v4 donde T1 sale del encoder compartido."""
    def __init__(self, n_inputs, n_modos=18, n_max_pisos=18, hidden=256):
        super().__init__()
        self.n_modos     = n_modos
        self.n_max_pisos = n_max_pisos
        self.encoder = nn.Sequential(
            nn.Linear(n_inputs, hidden), nn.LayerNorm(hidden), nn.SiLU(),
            ResBlock(hidden), ResBlock(hidden),
            ResBlock(hidden), ResBlock(hidden),
        )
        self.head_T1 = nn.Sequential(
            nn.Linear(hidden, 32), nn.SiLU(), nn.Linear(32, 1))
        self.head_T_rest = nn.Sequential(
            nn.Linear(hidden, 128), nn.SiLU(), nn.Linear(128, n_modos - 1))
        self.head_Phi = nn.Sequential(
            nn.Linear(hidden, 512), nn.SiLU(),
            nn.Linear(512, n_modos * n_max_pisos * 3))
        self.head_resp = nn.Sequential(
            nn.Linear(hidden, 256), nn.SiLU(),
            nn.Linear(256, n_max_pisos * 4))
        self.head_Vb = nn.Sequential(
            nn.Linear(hidden, 64), nn.SiLU(), nn.Linear(64, 2))

    def forward(self, X, mask):
        B  = X.shape[0]
        h  = self.encoder(X)
        T1 = self.head_T1(h)
        T_deltas = F.softplus(self.head_T_rest(h))
        parts = [T1]
        for r in range(self.n_modos - 1):
            parts.append(parts[-1] - T_deltas[:, r:r+1])
        logT_pred = torch.cat(parts, dim=1)
        Phi_raw  = self.head_Phi(h).view(B, self.n_modos, self.n_max_pisos, 3)
        mask_4d  = mask.unsqueeze(1).unsqueeze(-1)
        Phi_raw  = Phi_raw * mask_4d
        Phi_norm = Phi_raw / Phi_raw.norm(dim=2, keepdim=True).clamp(min=1e-8)
        Phi_all  = Phi_norm.permute(0, 2, 1, 3)
        resp    = self.head_resp(h).view(B, self.n_max_pisos, 4)
        Ux_p    = resp[:, :, 0] * mask
        Uy_p    = resp[:, :, 1] * mask
        ratio_p = F.softplus(resp[:, :, 2]) * mask
        dy_p    = resp[:, :, 3] * mask
        dx_p    = torch.expm1(ratio_p.clamp(min=0)) * dy_p
        Vb_p    = self.head_Vb(h)
        return (logT_pred,
                Phi_all[:, :, :, 0], Phi_all[:, :, :, 1], Phi_all[:, :, :, 2],
                Ux_p, Uy_p, dx_p, dy_p, Vb_p, ratio_p)

# ── Verificación de dependencias del main ─────────────────────────────
_deps = ["build_model","compute_loss","get_lambdas_and_active","compute_scalers",
         "make_dataloaders","compute_metrics_ood","ResBlock",
         "CosineAnnealingWarmRestarts","device","SEED","N_INPUTS"]
_missing = [d for d in _deps if d not in dir()]
assert not _missing, f"Faltan definiciones del NB5: {_missing} — corre las celdas del main primero"
print("PINNModal_noT1enc definida. Dependencias del main: OK")

PINNModal_noT1enc definida. Dependencias del main: OK


In [13]:
# ══════════════════════════════════════════════════════════════════════
# ABLATION V6 — (B) Variantes + curriculum + entrenamiento (Adam+Cosine)
# ══════════════════════════════════════════════════════════════════════
EP_ABL_V6  = 600    # idéntico al main
EP_WARM_V6 = 200    # warm-up data-only para filas 1-5 (curriculum plano)
CKPT_MIN   = 250    # umbral de checkpoint best-val (idéntico al main)

# λ por término = los del main. data_T1=1.0 constante post-warmup en TODAS las filas
# (sin el parámetro muerto "T1"). Cada fila difiere en UN solo término activo.
ABL_V6_VARIANTS = [
    {"id":1, "label":"(1) Supervised NN",      "use_enc_T1":False, "staged":False,
     "lam":{"data_resp":2.0,"data_T_rest":1.0,"phi":0.0,"eig":0.0,"ortho":0.0,"ratio":1.0},
     "act":{"phi":False,"eig":False,"ortho":False}},
    {"id":2, "label":"(2) + Encoder T1",       "use_enc_T1":True,  "staged":False,
     "lam":{"data_resp":2.0,"data_T_rest":1.0,"phi":0.0,"eig":0.0,"ortho":0.0,"ratio":1.0},
     "act":{"phi":False,"eig":False,"ortho":False}},
    {"id":3, "label":"(3) + L_eig",            "use_enc_T1":True,  "staged":False,
     "lam":{"data_resp":2.0,"data_T_rest":1.0,"phi":0.0,"eig":0.1,"ortho":0.0,"ratio":1.0},
     "act":{"phi":False,"eig":True,"ortho":False}},
    {"id":4, "label":"(4) + L_phi (MAC)",      "use_enc_T1":True,  "staged":False,
     "lam":{"data_resp":2.0,"data_T_rest":1.0,"phi":0.1,"eig":0.1,"ortho":0.0,"ratio":1.0},
     "act":{"phi":True,"eig":True,"ortho":False}},
    {"id":5, "label":"(5) + L_ortho",          "use_enc_T1":True,  "staged":False,
     "lam":{"data_resp":2.0,"data_T_rest":1.0,"phi":0.1,"eig":0.1,"ortho":1e-5,"ratio":1.0},
     "act":{"phi":True,"eig":True,"ortho":True}},
    {"id":6, "label":"(6) Full PINN v4",       "use_enc_T1":True,  "staged":True,
     "lam":None, "act":None},
]

def get_lambdas_ablation_v6(epoch, cfg):
    """Fila 6: curriculum v20 del main. Filas 1-5: 2 fases (warm-up data, luego fijo)."""
    if cfg["staged"]:
        return get_lambdas_and_active(epoch)
    if epoch < EP_WARM_V6:
        lam = {"data_resp":2.0,"data_T_rest":1.0,"data_T1":0.0,
               "phi":0.0,"eig":0.0,"ortho":0.0,"ratio":1.0}
        act = {"phi":False,"eig":False,"ortho":False}
    else:
        lam = dict(cfg["lam"]); lam["data_T1"] = 1.0
        act = dict(cfg["act"])
    return lam, act

def train_ablation_v6(cfg, idx_tr, idx_val, idx_tst, rep):
    """Entrena una variante. Régimen idéntico al main. Devuelve métricas OOD en test."""
    torch.manual_seed(1000*rep + cfg["id"]); np.random.seed(1000*rep + cfg["id"])
    scalers = compute_scalers(idx_tr)                       # scalers SOLO de train
    dl_tr, dl_val, dl_tst = make_dataloaders(idx_tr, idx_val, idx_tst, scalers)

    if cfg["use_enc_T1"]:
        model = build_model().to(device)                   # PINNModal_v4b
    else:
        model = PINNModal_noT1enc(n_inputs=N_INPUTS).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    sch = CosineAnnealingWarmRestarts(opt, T_0=200, T_mult=2, eta_min=1e-6)

    has_encT1 = hasattr(model, "encoder_T1")
    if has_encT1:
        pT1 = list(model.encoder_T1.parameters()) + list(model.head_T1.parameters())

    best_val, best_sd = float("inf"), None
    for ep in range(EP_ABL_V6):
        lam, act = get_lambdas_ablation_v6(ep, cfg)
        if has_encT1:
            # entrenamiento en dos velocidades: encoder_T1 activo desde ep>=150 (staged)
            # o desde el warm-up (no-staged), igual que el espíritu del main
            unfreeze = (ep >= 150) if cfg["staged"] else (ep >= EP_WARM_V6)
            for p in pT1: p.requires_grad = unfreeze
        model.train()
        for batch in dl_tr:
            opt.zero_grad()
            out = model(batch["X"].to(device), batch["mask"].to(device))
            loss, _ = compute_loss(batch, out, lam, scalers, act)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sch.step()
        # checkpoint best-val (loss ponderada en val), idéntico al main
        model.eval(); vt, nb_ = 0.0, 0
        with torch.no_grad():
            for batch in dl_val:
                out = model(batch["X"].to(device), batch["mask"].to(device))
                l, _ = compute_loss(batch, out, lam, scalers, act)
                vt += l.item(); nb_ += 1
        if ep >= CKPT_MIN and vt/max(nb_,1) < best_val:
            best_val = vt/max(nb_,1)
            best_sd  = {k:v.cpu().clone() for k,v in model.state_dict().items()}
    if best_sd is not None:
        model.load_state_dict(best_sd)
    return compute_metrics_ood(model, dl_tst, scalers)

print("Variantes y funciones de ablación V6 listas.")

Variantes y funciones de ablación V6 listas.


In [14]:
# ══════════════════════════════════════════════════════════════════════
# ABLATION V6 — (C) Ejecución: 6 filas × 4 reps en Trial 16
# ══════════════════════════════════════════════════════════════════════
RUN_ABL_V6  = True    # <- True para ejecutar (~1.5 h)
ABL_TID     = 16       # Trial 16: zona_1_3_vs_zona_2 (mismo del ablation previo)
ABL_V6_PATH = TRIALS_DIR / vfile("ablation_v6", "json")
N_REP_ABL   = 4

if RUN_ABL_V6:
    _tdef = next(t for t in TRIALS_DEF if t["id"] == ABL_TID)
    idx_A = np.where(np.atleast_1d(_tdef["mask_A"]))[0]
    idx_B = np.where(np.atleast_1d(_tdef["mask_B"]))[0]

    # Splits reproducibles: MISMO generador que el loop principal
    rng = np.random.default_rng(SEED + ABL_TID * 100)
    A_sh = rng.permutation(idx_A)
    B_sh = rng.permutation(idx_B)
    A1 = A_sh[:N_TRAIN]; A2 = A_sh[N_TRAIN:2*N_TRAIN]
    n_blk = N_VAL_OOD + N_TEST_OOD
    # 4 reps: alterna dominios de train A1/A2 (factorial) y 4 bloques B disjuntos
    rep_splits = []
    for rep in range(N_REP_ABL):
        tr  = A1 if rep % 2 == 0 else A2
        blk = B_sh[rep*n_blk:(rep+1)*n_blk]
        val = blk[:N_VAL_OOD]; tst = blk[N_VAL_OOD:N_VAL_OOD+N_TEST_OOD]
        # ── Anti-fuga: intersección vacía verificada por código ──
        assert np.intersect1d(tr,  val).size == 0, f"fuga train/val rep{rep}"
        assert np.intersect1d(tr,  tst).size == 0, f"fuga train/test rep{rep}"
        assert np.intersect1d(val, tst).size == 0, f"fuga val/test rep{rep}"
        rep_splits.append((tr, val, tst))
    print(f"Trial {ABL_TID}: Pool A={len(idx_A)}  Pool B={len(idx_B)}  "
          f"| {N_REP_ABL} reps, anti-fuga OK\n")

    abl_results = {}
    for cfg in ABL_V6_VARIANTS:
        reps_m = {"MAC_avg":[], "T1_MAPE":[], "Vb_MAPE":[], "drift_MAPE":[]}
        for rep, (tr, val, tst) in enumerate(rep_splits):
            m = train_ablation_v6(cfg, tr, val, tst, rep)
            for k in reps_m: reps_m[k].append(float(m[k]))
        abl_results[str(cfg["id"])] = {
            "label": cfg["label"],
            "media": {k: float(np.mean(v)) for k, v in reps_m.items()},
            "std":   {k: float(np.std(v))  for k, v in reps_m.items()},
            "reps":  reps_m,
        }
        r = abl_results[str(cfg["id"])]
        print(f"{cfg['label']:22s} | MAC={r['media']['MAC_avg']:.3f}  "
              f"T1={r['media']['T1_MAPE']:.1f}%  Vb={r['media']['Vb_MAPE']:.1f}%  "
              f"drift={r['media']['drift_MAPE']:.1f}%")

    with open(ABL_V6_PATH, "w") as f:
        json.dump(abl_results, f, indent=2)
    print(f"\nGuardado: {ABL_V6_PATH.name}")
else:
    print("RUN_ABL_V6 = False  (no ejecutado)")


Trial 16: Pool A=6623  Pool B=3377  | 4 reps, anti-fuga OK

(1) Supervised NN      | MAC=0.089  T1=11.5%  Vb=11.2%  drift=16.1%
(2) + Encoder T1       | MAC=0.104  T1=8.6%  Vb=8.9%  drift=14.4%
(3) + L_eig            | MAC=0.290  T1=8.3%  Vb=9.2%  drift=13.0%
(4) + L_phi (MAC)      | MAC=0.996  T1=11.0%  Vb=9.1%  drift=13.8%
(5) + L_ortho          | MAC=0.999  T1=9.2%  Vb=8.3%  drift=12.4%
(6) Full PINN v4       | MAC=0.995  T1=8.6%  Vb=9.9%  drift=13.0%

Guardado: ablation_v6_v8.json


In [15]:
# ══════════════════════════════════════════════════════════════════════
# ABLATION V6 — (D) Candado de reproducción + tabla + LaTeX
# ══════════════════════════════════════════════════════════════════════
if ABL_V6_PATH.exists():
    with open(ABL_V6_PATH) as f:
        abl = json.load(f)

    # ── Candado: fila 6 vs Tabla 8 (Trial 16 del loop principal) ──────
    with open(TRIALS_DIR / vfile("all_results", "json")) as f:
        _ar = json.load(f)
    _main16 = next(v for v in _ar.values() if v.get("trial_id") == ABL_TID)
    f6 = abl["6"]["media"]
    print("=== CANDADO DE REPRODUCCIÓN (fila 6 vs Tabla 8, Trial 16) ===")
    _pass = True
    for k in ["MAC_avg","T1_MAPE","Vb_MAPE","drift_MAPE"]:
        mu  = _main16["media"][k]; sd = _main16["std"][k]
        val = f6[k]; ok = abs(val - mu) <= max(sd, 1e-9)
        _pass = _pass and ok
        print(f"  {k:12s}: ablación={val:8.3f}  main={mu:7.3f}±{sd:.3f}  "
              f"{'OK' if ok else 'FUERA DE ±1sigma'}")
    print(f"\n  Candado: {'PASA -> tabla interpretable' if _pass else 'NO PASA -> revisar pipeline'}\n")

    # ── Tabla agregada ────────────────────────────────────────────────
    rows = []
    for vid in ["1","2","3","4","5","6"]:
        r = abl[vid]
        rows.append({
            "Variante":  r["label"],
            "MAC_avg":   f"{r['media']['MAC_avg']:.3f}±{r['std']['MAC_avg']:.3f}",
            "T1_MAPE":   f"{r['media']['T1_MAPE']:.1f}±{r['std']['T1_MAPE']:.1f}",
            "Vb_MAPE":   f"{r['media']['Vb_MAPE']:.1f}±{r['std']['Vb_MAPE']:.1f}",
            "drift_MAPE":f"{r['media']['drift_MAPE']:.1f}±{r['std']['drift_MAPE']:.1f}",
        })
    df_abl = pd.DataFrame(rows)
    print(df_abl.to_string(index=False))

    csv_path = TRIALS_DIR / vfile("ablation_v6", "csv")
    df_abl.to_csv(csv_path, index=False)
    print(f"\nCSV: {csv_path.name}")

    # ── LaTeX (compatible con Tabla 8) ────────────────────────────────
    print("\n% ---- LaTeX ----")
    print("\\begin{tabular}{lcccc}")
    print("\\hline")
    print("Variante & MAC & $T_1$ MAPE (\\%) & $V_b$ MAPE (\\%) & drift MAPE (\\%) \\\\")
    print("\\hline")
    for _, row in df_abl.iterrows():
        print(f"{row['Variante']} & {row['MAC_avg']} & {row['T1_MAPE']} & "
              f"{row['Vb_MAPE']} & {row['drift_MAPE']} \\\\")
    print("\\hline")
    print("\\end{tabular}")
else:
    print("Sin resultados. Corre la celda ablv6-run con RUN_ABL_V6=True.")

=== CANDADO DE REPRODUCCIÓN (fila 6 vs Tabla 8, Trial 16) ===
  MAC_avg     : ablación=   0.995  main=  0.997±0.001  FUERA DE ±1sigma
  T1_MAPE     : ablación=   8.620  main=  9.713±1.031  FUERA DE ±1sigma
  Vb_MAPE     : ablación=   9.949  main=  9.951±2.351  OK
  drift_MAPE  : ablación=  13.000  main= 13.333±1.511  OK

  Candado: NO PASA -> revisar pipeline

         Variante     MAC_avg  T1_MAPE  Vb_MAPE drift_MAPE
(1) Supervised NN 0.089±0.009 11.5±2.2 11.2±1.0   16.1±2.0
 (2) + Encoder T1 0.104±0.016  8.6±2.2  8.9±2.2   14.4±2.3
      (3) + L_eig 0.290±0.029  8.3±0.9  9.2±1.5   13.0±1.0
(4) + L_phi (MAC) 0.996±0.002 11.0±2.6  9.1±1.3   13.8±1.4
    (5) + L_ortho 0.999±0.001  9.2±1.2  8.3±1.9   12.4±1.9
 (6) Full PINN v4 0.995±0.002  8.6±1.8  9.9±1.5   13.0±2.0

CSV: ablation_v6_v8.csv

% ---- LaTeX ----
\begin{tabular}{lcccc}
\hline
Variante & MAC & $T_1$ MAPE (\%) & $V_b$ MAPE (\%) & drift MAPE (\%) \\
\hline
(1) Supervised NN & 0.089±0.009 & 11.5±2.2 & 11.2±1.0 & 16.1±2.0 \\
(2)

In [ ]:
# CELDA 10 -- LOOP PRINCIPAL: 31 TRIALS × 4 REPETICIONES
# Metodología: Repeated Hold-Out OOD con Diseño Factorial 2×2
# v7: PINNModal_v4b + compute_loss_v16 + curriculum v20
#     21 features + scaler_pisos corregido

import numpy as np

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer,)):    return int(obj)
        if isinstance(obj, (np.floating,)):   return float(obj)
        if isinstance(obj, np.ndarray):       return obj.tolist()
        if isinstance(obj, (np.bool_, bool)): return bool(obj)
        return super().default(obj)

print(f"Carpeta resultados: {TRIALS_DIR_V8}")

TRIALS_A_CORRER = list(range(1, 32))
all_results      = {}
rng_global       = np.random.default_rng(SEED)

for trial_def in TRIALS_DEF:
    tid    = trial_def['id']
    nombre = trial_def['nombre']
    tipo   = trial_def.get('tipo', 'conceptual')

    if tid not in TRIALS_A_CORRER:
        continue

    print(f"\n{'='*70}")
    print(f"TRIAL {tid:02d}/{len(TRIALS_DEF)} -- {nombre}")
    print(f"{'='*70}")
    t_start = datetime.now()

    # ── 1. Construir Pool A y Pool B ─────────────────────────────────────
    if tipo == 'complemento':
        idx_all    = np.arange(N_CASES)
        idx_pool_A = rng_global.choice(idx_all, size=4000, replace=False)
        idx_pool_B = np.setdiff1d(idx_all, idx_pool_A)
    elif tipo == 'aleatorio_puro':
        idx_all    = np.arange(N_CASES)
        idx_pool_A = rng_global.choice(idx_all, size=5000, replace=False)
        idx_pool_B = rng_global.choice(idx_all, size=5000, replace=False)
    else:
        idx_pool_A = np.where(trial_def['mask_A'])[0]
        idx_pool_B = np.where(trial_def['mask_B'])[0]

    print(f"  Pool A disponible: {len(idx_pool_A)} casos")
    print(f"  Pool B disponible: {len(idx_pool_B)} casos")

    # ── 2. Verificar factibilidad ─────────────────────────────────────────
    n_B_necesario = K_REPS * (N_VAL_OOD + N_TEST_OOD)
    n_A_necesario = 2 * N_TRAIN

    n_train_real = N_TRAIN if len(idx_pool_A) >= n_A_necesario \
                  else len(idx_pool_A) // 2
    if len(idx_pool_A) < n_A_necesario:
        print(f"  ⚠️  Pool A < {n_A_necesario} → usando n_train={n_train_real}")

    k_reps_trial = K_REPS if len(idx_pool_B) >= n_B_necesario else 2
    if len(idx_pool_B) < n_B_necesario:
        print(f"  ⚠️  Pool B < {n_B_necesario} → reduciendo a 2 repeticiones")

    # ── 3. Preparar A1 y A2 ───────────────────────────────────────────────
    rng_trial      = np.random.default_rng(SEED + tid * 100)
    idx_A_shuffled = rng_trial.permutation(idx_pool_A)

    if len(idx_pool_A) >= 2 * n_train_real:
        idx_A1 = idx_A_shuffled[:n_train_real]
        idx_A2 = idx_A_shuffled[n_train_real:2*n_train_real]
    else:
        mid    = len(idx_A_shuffled) // 2
        idx_A1 = idx_A_shuffled[:mid]
        idx_A2 = idx_A_shuffled[mid:]

    # ── 4. Preparar bloques B ─────────────────────────────────────────────
    idx_B_shuffled = rng_trial.permutation(idx_pool_B)
    n_por_bloque   = N_VAL_OOD + N_TEST_OOD
    bloques_B      = []
    for b in range(k_reps_trial * 2):
        inicio = b * n_por_bloque
        fin    = inicio + n_por_bloque
        bloques_B.append(idx_B_shuffled[inicio:min(fin, len(idx_B_shuffled))])

    # ── 5. Tabla de dominios ──────────────────────────────────────────────
    tabla_dominios(trial_def, idx_pool_A, idx_pool_B)

    # ── 6. Verificación de diversidad ────────────────────────────────────
    div_stats = verificar_diversidad(idx_A1, idx_A2, idx_pool_B)

    # ── 7. Detectar variable clave ────────────────────────────────────────
    rng_vk    = np.random.default_rng(42)
    n_mues    = min(300, len(idx_pool_A), len(idx_pool_B))
    idx_Am_vk = rng_vk.choice(idx_pool_A, n_mues, replace=False)
    idx_Bm_vk = rng_vk.choice(idx_pool_B, n_mues, replace=False)

    var_clave = 'N_pisos'
    max_diff  = 0
    for col in COLS_BASE:
        idx_c  = COLS_BASE.index(col)
        vA     = X_clean[idx_Am_vk, idx_c]
        vB     = X_clean[idx_Bm_vk, idx_c]
        sigma  = max(vA.std(), vB.std(), 1e-8)
        d_norm = abs(vA.mean() - vB.mean()) / sigma
        if d_norm > max_diff:
            max_diff  = d_norm
            var_clave = col

    if max_diff < 0.3:
        var_clave = 'N_pisos'

    print(f"\n  A1: {len(idx_A1)} casos  A2: {len(idx_A2)} casos")
    print(f"  Bloques B: {len(bloques_B)} bloques de ~{n_por_bloque} casos")
    print(f"  Variable clave: {var_clave} (Δ={max_diff:.2f}σ)")
    print(f"  Repeticiones: {k_reps_trial}")

    # ── 8. Carpeta del trial — dentro de V8/ ─────────────────────────────
    trial_dir = TRIALS_DIR_V8 / f'trial_{tid:02d}_{nombre}'
    trial_dir.mkdir(parents=True, exist_ok=True)

    # ── 9. Función para formatear variable clave ──────────────────────────
    idx_vc = COLS_BASE.index(var_clave)

    def fmt_var(idx_arr):
        vals = X_clean[idx_arr, idx_vc]
        u    = np.unique(np.round(vals, 2))
        if len(u) <= 8:
            return str([round(float(v), 2) for v in u])
        return f"[{vals.min():.2f}->{vals.max():.2f}] media={vals.mean():.2f}"

    # ── 10. Loop de repeticiones ──────────────────────────────────────────
    rep_results = []

    for rep in range(1, k_reps_trial + 1):
        print(f"\n  {'─'*50}")
        print(f"  Rep {rep}/{k_reps_trial}")

        idx_tr       = idx_A1 if rep <= 2 else idx_A2
        bloque_val   = bloques_B[(rep-1)*2]
        bloque_test  = bloques_B[(rep-1)*2 + 1]
        idx_val_ood  = bloque_val[:N_VAL_OOD]
        idx_test_ood = bloque_test[:N_TEST_OOD]

        print(f"  Train={'A1' if rep<=2 else 'A2'}: {len(idx_tr)} casos"
              f"  |  {var_clave}: {fmt_var(idx_tr)}")
        print(f"  Val OOD : {len(idx_val_ood)} casos"
              f"  |  {var_clave}: {fmt_var(idx_val_ood)}")
        print(f"  Test OOD: {len(idx_test_ood)} casos"
              f"  |  {var_clave}: {fmt_var(idx_test_ood)}")

        contam = len(set(idx_tr.tolist()) & set(idx_test_ood.tolist()))
        if contam > 0:
            print(f"  ⚠️  Contaminación train/test: {contam} casos")

        scalers = compute_scalers(idx_tr)
        dl_tr, dl_val, dl_ood = make_dataloaders(
            idx_tr, idx_val_ood, idx_test_ood, scalers)
        model = build_model()

        rep_dir = trial_dir / f'rep_{rep:02d}'
        rep_dir.mkdir(parents=True, exist_ok=True)

        t_rep_start = datetime.now()
        hist_df, best_val, best_epoch = train_trial(
            model, dl_tr, dl_val, scalers, rep_dir,
            ep_total=600, verbose=True)
        t_rep_min = (datetime.now() - t_rep_start).total_seconds() / 60

        model.load_state_dict(torch.load(
            rep_dir / 'best_model.pt', map_location=device))
        metrics = compute_metrics_ood(model, dl_ood, scalers)

        rep_info = {
            'rep'            : rep,
            'train_set'      : 'A1' if rep <= 2 else 'A2',
            'n_train'        : len(idx_tr),
            'n_val_ood'      : len(idx_val_ood),
            'n_test_ood'     : len(idx_test_ood),
            'var_clave'      : var_clave,
            'var_clave_train': fmt_var(idx_tr),
            'var_clave_val'  : fmt_var(idx_val_ood),
            'var_clave_test' : fmt_var(idx_test_ood),
            'best_epoch'     : int(best_epoch),
            'best_val'       : float(best_val),
            'tiempo_min'     : round(t_rep_min, 2),
            'metricas'       : metrics,
        }
        with open(rep_dir / 'metricas.json', 'w', encoding='utf-8') as f:
            json.dump(rep_info, f, indent=4, ensure_ascii=False, cls=NpEncoder)

        rep_results.append(rep_info)

        print(f"\n  Rep {rep} completada en {t_rep_min:.1f} min")
        print(f"    MAC_avg    : {metrics['MAC_avg']:.4f}")
        print(f"    T1_MAPE    : {metrics['T1_MAPE']:.2f}%")
        print(f"    Vb_MAPE    : {metrics['Vb_MAPE']:.2f}%")
        print(f"    drift_MAPE : {metrics['drift_MAPE']:.2f}%")

    # ── 11. Resumen del trial ─────────────────────────────────────────────
    def stats(key):
        vals = [r['metricas'][key] for r in rep_results
                if r['metricas'][key] is not None]
        if not vals:
            return {'media': 0.0, 'std': 0.0, 'vals': []}
        return {
            'media': round(float(np.mean(vals)), 6),
            'std'  : round(float(np.std(vals)),  6),
            'vals' : [round(v, 6) for v in vals],
        }

    def var_ood(key):
        if len(rep_results) >= 2:
            return round(abs(rep_results[0]['metricas'][key] -
                             rep_results[1]['metricas'][key]), 6)
        return None

    def var_train(key):
        if len(rep_results) >= 3:
            return round(abs(rep_results[0]['metricas'][key] -
                             rep_results[2]['metricas'][key]), 6)
        return None

    metricas_clave = ['MAC_avg', 'T1_MAPE', 'Vb_MAPE', 'drift_MAPE']

    resumen = {
        'trial_id'        : tid,
        'nombre'          : nombre,
        'tipo'            : tipo,
        'k_reps'          : k_reps_trial,
        'n_train'         : n_train_real,
        'n_val_ood'       : N_VAL_OOD,
        'n_test_ood'      : N_TEST_OOD,
        'var_clave'       : var_clave,
        'dataset'         : 'v6',
        'arquitectura'    : 'PINNModal_v4b',
        'loss'            : 'compute_loss_v16',
        'curriculum'      : 'v20_staged_training',
        'n_inputs'        : 21,
        'ep_total'        : 600,
        'tiempo_total_min': round((datetime.now()-t_start).total_seconds()/60, 2),
        'media'           : {k: stats(k)['media'] for k in metricas_clave},
        'std'             : {k: stats(k)['std']   for k in metricas_clave},
        'vals'            : {k: stats(k)['vals']  for k in metricas_clave},
        'var_ood'         : {k: var_ood(k)        for k in metricas_clave},
        'var_train'       : {k: var_train(k)      for k in metricas_clave},
        'best_epochs'     : [r['best_epoch'] for r in rep_results],
        'best_epoch_mean' : int(np.mean([r['best_epoch'] for r in rep_results])),
        'diversidad'      : div_stats,
        'reps'            : rep_results,
    }

    with open(trial_dir / 'resumen.json', 'w', encoding='utf-8') as f:
        json.dump(resumen, f, indent=4, ensure_ascii=False, cls=NpEncoder)

    all_results[tid] = resumen

    # ── 12. Print resumen ─────────────────────────────────────────────────
    t_total = resumen['tiempo_total_min']
    print(f"\n{'='*70}")
    print(f"TRIAL {tid:02d} — {nombre} — COMPLETADO en {t_total:.1f} min")
    print(f"{'='*70}")
    print(f"  {'Métrica':<14} {'Media':>8} {'±Std':>8} "
          f"{'Var_OOD':>10} {'Var_Train':>10}")
    print(f"  {'-'*52}")
    for k in metricas_clave:
        unidad  = '' if k == 'MAC_avg' else '%'
        v_ood   = resumen['var_ood'][k]
        v_train = resumen['var_train'][k]
        print(f"  {k:<14} {resumen['media'][k]:>8.4f} "
              f"{resumen['std'][k]:>8.4f} "
              f"{str(round(v_ood,4)) if v_ood is not None else 'N/A':>10} "
              f"{str(round(v_train,4)) if v_train is not None else 'N/A':>10}"
              f" {unidad}")
    print(f"  Épocas best: {resumen['best_epochs']}  "
          f"→ media={resumen['best_epoch_mean']}")

# ── 13. Guardar all_results en V8/ ───────────────────────────────────────
all_results_path = TRIALS_DIR_V8 / 'all_results_v8.json'
with open(all_results_path, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, indent=4, ensure_ascii=False, cls=NpEncoder)

print(f"\n{'='*70}")
print(f"TODOS LOS TRIALS COMPLETADOS: {len(all_results)}")
print(f"Dataset usado    : HDF5 v6 (masa efectiva >= 90%)")
print(f"Arquitectura     : PINNModal_v4b — encoder_T1 separado, dx via ratio")
print(f"Loss             : compute_loss_v16")
print(f"Curriculum       : v20 staged training 600 ep")
print(f"N_INPUTS         : 21 (sin mods_por_depto ni activar_B3)")
print(f"Guardado en      : {all_results_path}")
print(f"{'='*70}")

## ~~Celda Ablation V4 — DEPRECATED~~

> **Reemplazado por Ablation V5** (ver celdas abajo).  
> Código original conservado comentado en la siguiente celda.


In [ ]:
# DEPRECATED - ablation V4, reemplazado por Ablation V5
if False:
    # ═══════════════════════════════════════════════════════════════════════════
    # CELDA ABLATION — ABLATION STUDY (Structures paper)
    # 6 variantes × Trial-16 × 2 reps × 600 épocas
    #
    # Filas de la tabla:
    #   (1) Supervised NN            — sin encoder_T1 dedicado, sin física
    #   (2) + Encoder T1             — encoder dedicado, sin física
    #   (3) + L_eig                  — residual de autovalores
    #   (4) + L_MAC                  — similitud MAC de formas modales
    #   (5) + L_ortho                — ortonormalidad M-ponderada
    #   (6) Full PINN_Modal_v4       — curriculum staged v20 completo
    # ═══════════════════════════════════════════════════════════════════════════
    
    import pandas as pd
    from pathlib import Path
    
    # ─────────────────────────────────────────────────────────────────────────
    # A) MODELO SIN ENCODER DEDICADO PARA T1
    # ─────────────────────────────────────────────────────────────────────────
    class PINNModal_noT1enc(nn.Module):
        """Variante de PINNModal_v4 donde T1 sale del encoder compartido."""
        def __init__(self, n_inputs, n_modos=18, n_max_pisos=18, hidden=256):
            super().__init__()
            self.n_modos     = n_modos
            self.n_max_pisos = n_max_pisos
            self.encoder = nn.Sequential(
                nn.Linear(n_inputs, hidden), nn.LayerNorm(hidden), nn.SiLU(),
                ResBlock(hidden), ResBlock(hidden),
                ResBlock(hidden), ResBlock(hidden),
            )
            self.head_T1 = nn.Sequential(
                nn.Linear(hidden, 32), nn.SiLU(), nn.Linear(32, 1))
            self.head_T_rest = nn.Sequential(
                nn.Linear(hidden, 128), nn.SiLU(), nn.Linear(128, n_modos - 1))
            self.head_Phi = nn.Sequential(
                nn.Linear(hidden, 512), nn.SiLU(),
                nn.Linear(512, n_modos * n_max_pisos * 3))
            self.head_resp = nn.Sequential(
                nn.Linear(hidden, 256), nn.SiLU(),
                nn.Linear(256, n_max_pisos * 4))
            self.head_Vb = nn.Sequential(
                nn.Linear(hidden, 64), nn.SiLU(), nn.Linear(64, 2))
    
        def forward(self, X, mask):
            B  = X.shape[0]
            h  = self.encoder(X)
            T1 = self.head_T1(h)
            T_deltas = F.softplus(self.head_T_rest(h))
            parts = [T1]
            for r in range(self.n_modos - 1):
                parts.append(parts[-1] - T_deltas[:, r:r+1])
            logT_pred = torch.cat(parts, dim=1)
    
            Phi_raw  = self.head_Phi(h).view(B, self.n_modos, self.n_max_pisos, 3)
            mask_4d  = mask.unsqueeze(1).unsqueeze(-1)
            Phi_raw  = Phi_raw * mask_4d
            Phi_norm = Phi_raw / Phi_raw.norm(dim=2, keepdim=True).clamp(min=1e-8)
            Phi_all  = Phi_norm.permute(0, 2, 1, 3)
    
            resp    = self.head_resp(h).view(B, self.n_max_pisos, 4)
            Ux_p    = resp[:, :, 0] * mask
            Uy_p    = resp[:, :, 1] * mask
            ratio_p = F.softplus(resp[:, :, 2]) * mask
            dy_p    = resp[:, :, 3] * mask
            dx_p    = torch.expm1(ratio_p.clamp(min=0)) * dy_p
            Vb_p    = self.head_Vb(h)
    
            return (logT_pred,
                    Phi_all[:, :, :, 0], Phi_all[:, :, :, 1], Phi_all[:, :, :, 2],
                    Ux_p, Uy_p, dx_p, dy_p, Vb_p, ratio_p)
    
    
    # ─────────────────────────────────────────────────────────────────────────
    # B) CONFIGURACIÓN DE LAS 6 VARIANTES
    # ─────────────────────────────────────────────────────────────────────────
    EP_ABL  = 600   # épocas totales (igual que main)
    EP_WARM = 200   # warm-up data-only para variantes 1-5
    
    ABLATION_VARIANTS = [
        {
            'id': 1, 'label': '(1) Supervised NN',
            'use_enc_T1': False, 'staged': False,
            'lambdas': {'data_resp':2.0,'data_T_rest':1.0,'T1':0.0,
                        'phi':0.0,'eig':0.0,'ortho':0.0,'ratio':1.0},
            'active':  {'phi':False,'eig':False,'ortho':False},
        },
        {
            'id': 2, 'label': '(2) + Encoder T1',
            'use_enc_T1': True, 'staged': False,
            'lambdas': {'data_resp':2.0,'data_T_rest':1.0,'T1':0.0,
                        'phi':0.0,'eig':0.0,'ortho':0.0,'ratio':1.0},
            'active':  {'phi':False,'eig':False,'ortho':False},
        },
        {
            'id': 3, 'label': '(3) + L_eig',
            'use_enc_T1': True, 'staged': False,
            'lambdas': {'data_resp':2.0,'data_T_rest':1.0,'T1':1.0,
                        'phi':0.0,'eig':0.1,'ortho':0.0,'ratio':1.0},
            'active':  {'phi':False,'eig':True,'ortho':False},
        },
        {
            'id': 4, 'label': '(4) + L_MAC',
            'use_enc_T1': True, 'staged': False,
            'lambdas': {'data_resp':2.0,'data_T_rest':1.0,'T1':1.0,
                        'phi':0.1,'eig':0.1,'ortho':0.0,'ratio':1.0},
            'active':  {'phi':True,'eig':True,'ortho':False},
        },
        {
            'id': 5, 'label': '(5) + L_ortho',
            'use_enc_T1': True, 'staged': False,
            'lambdas': {'data_resp':2.0,'data_T_rest':1.0,'T1':1.0,
                        'phi':0.1,'eig':0.1,'ortho':1e-5,'ratio':1.0},
            'active':  {'phi':True,'eig':True,'ortho':True},
        },
        {
            'id': 6, 'label': '(6) Full PINN_Modal_v4',
            'use_enc_T1': True, 'staged': True,
            'lambdas': None, 'active': None,
        },
    ]
    
    
    # ─────────────────────────────────────────────────────────────────────────
    # C) CURRICULUM PARA ABLACIÓN
    # ─────────────────────────────────────────────────────────────────────────
    def get_lambdas_ablation(epoch, cfg):
        """2-phase para variantes 1-5; curriculum v20 original para variante 6."""
        if cfg['staged']:
            return get_lambdas_and_active(epoch)
        if epoch < EP_WARM:
            lam = {'data_resp':2.0,'data_T_rest':1.0,'data_T1':0.0,
                   'T1':0.0,'phi':0.0,'eig':0.0,'ortho':0.0,'ratio':1.0}
            act = {'phi':False,'eig':False,'ortho':False}
        else:
            lam = dict(cfg['lambdas'])
            lam['data_T1'] = 1.0
            act = cfg['active']
        return lam, act
    
    
    # ─────────────────────────────────────────────────────────────────────────
    # D) FUNCIÓN DE ENTRENAMIENTO PARA ABLACIÓN
    # ─────────────────────────────────────────────────────────────────────────
    def train_ablation_variant(model, dl_tr, dl_val, scalers,
                                save_dir, cfg, ep_total=EP_ABL, verbose=True):
        save_dir.mkdir(parents=True, exist_ok=True)
        best_path = save_dir / 'best.pt'
        best_val  = float('inf')
    
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        scheduler = CosineAnnealingWarmRestarts(
            optimizer, T_0=200, T_mult=2, eta_min=1e-6)
    
        has_enc_T1 = hasattr(model, 'encoder_T1')
        if has_enc_T1:
            params_T1 = (list(model.encoder_T1.parameters()) +
                         list(model.head_T1.parameters()))
            for p in params_T1:
                p.requires_grad = False
    
        stage_prev = -1
        for ep in range(ep_total):
            lambdas, active = get_lambdas_ablation(ep, cfg)
    
            if has_enc_T1 and ep == EP_WARM:
                for p in params_T1:
                    p.requires_grad = True
                if verbose:
                    print(f'    ep {ep}: encoder_T1 descongelado')
    
            if cfg['staged'] and verbose:
                st = {150:1, 200:2, 350:3, 500:4}.get(ep, stage_prev)
                if st != stage_prev:
                    stage_prev = st
                    print(f'    ep {ep}: Stage {st}  '
                          f'eig={lambdas["eig"]}  ortho={lambdas["ortho"]}')
    
            model.train()
            for batch in dl_tr:
                optimizer.zero_grad()
                out = model(batch['X'].to(device), batch['mask'].to(device))
                loss, _ = compute_loss(batch, out, lambdas, scalers, active)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step(ep)
    
            if ep >= EP_WARM:
                model.eval()
                vl = 0.0; nb = 0
                with torch.no_grad():
                    for batch in dl_val:
                        out  = model(batch['X'].to(device), batch['mask'].to(device))
                        loss, _ = compute_loss(batch, out, lambdas, scalers, active)
                        vl += loss.item(); nb += 1
                vl /= max(nb, 1)
                if vl < best_val:
                    best_val = vl
                    torch.save(model.state_dict(), best_path)
    
        if best_path.exists():
            model.load_state_dict(
                torch.load(best_path, map_location=device, weights_only=True))
        return model
    
    
    # ─────────────────────────────────────────────────────────────────────────
    # E) RUNNER PRINCIPAL
    # ─────────────────────────────────────────────────────────────────────────
    def run_ablation_study(trial_id=16, n_reps=2, ep_total=EP_ABL, base_dir=None):
        """Corre las 6 variantes sobre Trial-{trial_id} con n_reps repeticiones."""
        if base_dir is None:
            base_dir = TRIALS_DIR / f'ablation_t{trial_id:02d}'
        base_dir.mkdir(parents=True, exist_ok=True)
    
        td    = next(t for t in TRIALS_DEF if t['id'] == trial_id)
        idx_A = np.where(td['mask_A'])[0]
        idx_B = np.where(td['mask_B'])[0]
    
        print(f"\nABLATION STUDY — Trial {trial_id} ({td['nombre']})")
        print(f"  Pool A (TRAIN): {len(idx_A)}  Pool B (OOD): {len(idx_B)}")
        print(f"  {n_reps} reps x {ep_total} epocas x 6 variantes\n")
    
        records = []
        for cfg in ABLATION_VARIANTS:
            vid = cfg['id']
            print(f"{'─'*60}")
            print(f"Variante {vid}: {cfg['label']}")
    
            rep_metrics = []
            for rep in range(n_reps):
                rng   = np.random.default_rng(seed=rep * 100 + trial_id)
                idx_s = idx_A.copy(); rng.shuffle(idx_s)
                n_tr  = int(len(idx_s) * 0.80)
                n_val = int(len(idx_s) * 0.10)
                idx_tr  = idx_s[:n_tr]
                idx_val = idx_s[n_tr:n_tr + n_val]
    
                scalers = compute_scalers(idx_tr)
                dl_tr, dl_val, dl_tst = make_dataloaders(
                    idx_tr, idx_val, idx_B, scalers)
    
                if cfg['use_enc_T1']:
                    model = PINNModal_v4(N_INPUTS, N_MODOS, N_MAX_PISOS,
                                          hidden=256, hidden_T1=128).to(device)
                else:
                    model = PINNModal_noT1enc(N_INPUTS, N_MODOS, N_MAX_PISOS,
                                               hidden=256).to(device)
    
                save_dir = base_dir / f'v{vid}' / f'rep{rep}'
                print(f'  rep {rep+1}/{n_reps}... ', end='', flush=True)
                model = train_ablation_variant(
                    model, dl_tr, dl_val, scalers,
                    save_dir, cfg, ep_total=ep_total, verbose=False)
    
                m = compute_metrics_ood(model, dl_tst, scalers)
                rep_metrics.append(m)
                print(f'T1={m["T1_MAPE"]:.1f}%  Vb={m["Vb_MAPE"]:.1f}%  '
                      f'Drift={m["drift_MAPE"]:.1f}%  MAC={m["MAC_avg"]:.3f}')
    
            keys = list(rep_metrics[0].keys())
            avg  = {k: float(np.mean([r[k] for r in rep_metrics])) for k in keys}
            std  = {k: float(np.std( [r[k] for r in rep_metrics])) for k in keys}
            records.append({
                'id':    vid,
                'label': cfg['label'],
                **{f'{k}_mean': avg[k] for k in keys},
                **{f'{k}_std':  std[k] for k in keys},
            })
    
        df = pd.DataFrame(records)
        df.to_csv(base_dir / 'ablation_results.csv', index=False)
        print(f'\nGuardado: {base_dir / "ablation_results.csv"}')
        return df
    
    
    # ─────────────────────────────────────────────────────────────────────────
    # F) TABLA FORMATEADA
    # ─────────────────────────────────────────────────────────────────────────
    def display_ablation_table(df, n_reps, trial_id=16):
        W = 74
        print('\n' + '=' * W)
        print(f'  TABLE — Ablation Study  |  Trial {trial_id}  |  '
              f'{n_reps} reps  |  {EP_ABL} ep')
        print('=' * W)
        print(f'  {"Model Variant":<32} {"T1 MAPE":>10} {"Vb MAPE":>10}'
              f'  {"Drift MAPE":>11} {"MAC1":>8}')
        print('  ' + '-' * (W - 2))
        for _, row in df.iterrows():
            t1  = f'{row["T1_MAPE_mean"]:5.1f}+/-{row["T1_MAPE_std"]:.1f}'
            vb  = f'{row["Vb_MAPE_mean"]:5.1f}+/-{row["Vb_MAPE_std"]:.1f}'
            dr  = f'{row["drift_MAPE_mean"]:5.1f}+/-{row["drift_MAPE_std"]:.1f}'
            mac = f'{row["MAC_avg_mean"]:.3f}+/-{row["MAC_avg_std"]:.3f}'
            print(f'  {row["label"]:<32} {t1:>10} {vb:>10}  {dr:>11} {mac:>12}')
        print('=' * W)
        print('  (1)-(5): two-phase curriculum (200 ep warm-up + 400 ep physics).')
        print('  (6)    : staged curriculum v20 (4 phases) as in main experiments.')
        print()
    
    
    # ─────────────────────────────────────────────────────────────────────────
    # G) EJECUCIÓN — comentar si ya fue corrido
    # ─────────────────────────────────────────────────────────────────────────
    _ABL_N_REPS   = 2
    _ABL_TRIAL_ID = 16
    
    ablation_df = run_ablation_study(
        trial_id = _ABL_TRIAL_ID,
        n_reps   = _ABL_N_REPS,
        ep_total = EP_ABL,
    )
    display_ablation_table(ablation_df, n_reps=_ABL_N_REPS, trial_id=_ABL_TRIAL_ID)


## Celda Ablation V5 — 6 variantes (granular)

### Objetivo
Tabla de ablación para el paper: cada fila agrega exactamente un componente
de la arquitectura PINN v5, más PINN v4 como baseline de comparación.

| # | Variante | Componente agregado |
|---|---|---|
| 1 | Solo T supervisado | baseline — solo predice T, sin φ ni física |
| 2 | + L_φ (MAC) | supervisión de formas modales |
| 3 | + L_Vb (NCh433) | pérdida física de corte basal vía NCh433 |
| 4 | + L_drift | pérdida física de derivas |
| 5 | Full PINN v5 | + ortogonalidad modal |
| 6 | PINN v4 (baseline) | arquitectura anterior con heads directos |

**Configuración**: Trial 16 × 2 reps × 600 épocas. `RUN_ABLATION_V5 = True` para ejecutar.


In [16]:
# ══════════════════════════════════════════════════════════════════════
# CELDA ABLATION V5 — SETUP
# ══════════════════════════════════════════════════════════════════════
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from b2core.pinn_modal_v5 import PINNModal_v5
from b2core.train_v5 import (
    load_dataset as load_dataset_v5,
    compute_scalers_v5, ModalDatasetV5, make_dataloaders_v5,
    compute_loss_v5, train_one_epoch as train_epoch_v5,
    eval_epoch as eval_epoch_v5, LAMBDAS_DEFAULT as LAMBDAS_V5,
)

EP_ABL_V5        = 600
N_REP_V5         = 2
RUN_ABLATION_V5  = False # <- cambiar a True para correr

_L0 = {"T1": 1.0, "T_rest": 1.0, "phi": 0.0, "Vb": 0.0, "drift": 0.0, "ortho": 0.0}

ABLATION_V5_VARIANTS = [
    {"id": 1, "label": "(1) Solo T supervisado",
     "model": "v5",
     "lambdas": {**_L0},
     "stage_active": {"phi": False, "ortho": False}},

    {"id": 2, "label": "(2) + L_phi (MAC)",
     "model": "v5",
     "lambdas": {**_L0, "phi": 0.5},
     "stage_active": {"phi": True, "ortho": False}},

    {"id": 3, "label": "(3) + L_Vb (NCh433)",
     "model": "v5",
     "lambdas": {**_L0, "phi": 0.5, "Vb": 1.0},
     "stage_active": {"phi": True, "ortho": False}},

    {"id": 4, "label": "(4) + L_drift",
     "model": "v5",
     "lambdas": {**_L0, "phi": 0.5, "Vb": 1.0, "drift": 0.5},
     "stage_active": {"phi": True, "ortho": False}},

    {"id": 5, "label": "(5) Full PINN v5",
     "model": "v5",
     "lambdas": {**_L0, "phi": 0.5, "Vb": 1.0, "drift": 0.5, "ortho": 1e-5},
     "stage_active": {"phi": True, "ortho": True}},

    {"id": 6, "label": "(6) PINN v4 (baseline)",
     "model": "v4",
     "lambdas": None,
     "stage_active": None},
]

if RUN_ABLATION_V5:
    _data_v5 = load_dataset_v5(str(H5_MODAL))
    print("Dataset v5 cargado:", _data_v5["X_clean"].shape)
print("CELDA ABL-V5-SETUP OK")


CELDA ABL-V5-SETUP OK


In [26]:
# ══════════════════════════════════════════════════════════════════════
# CELDA ABLATION V5 — FUNCIONES DE TRAINING Y EVALUACIÓN
# ══════════════════════════════════════════════════════════════════════
import torch, numpy as np


def run_ablation_trial_v5(variant, idx_tr, idx_val, idx_tst, rep, data_v5):
    """Train one ablation variant; return OOD MAPE dict."""
    torch.manual_seed(rep * 1000 + variant["id"])
    np.random.seed(rep * 1000 + variant["id"])

    if variant["model"] == "v5":
        scalers = compute_scalers_v5(data_v5, idx_tr)
        dl_tr, dl_val, dl_tst = make_dataloaders_v5(
            data_v5, idx_tr, idx_val, idx_tst, scalers)
        model = PINNModal_v5(n_inputs=N_INPUTS).to(device)
        opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
        sch   = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, patience=20, factor=0.5)
        best_val, best_sd = float("inf"), None
        for ep in range(1, EP_ABL_V5 + 1):
            # Curriculum: warm-up T only → phase in phi → phase in ortho
            lam = dict(variant["lambdas"])
            sa  = dict(variant["stage_active"])
            if ep < 100:
                lam["Vb"] = 0.0; lam["drift"] = 0.0
                sa["phi"] = False; sa["ortho"] = False
            elif ep < 300:
                lam["ortho"] = 0.0
                sa["ortho"] = False
            train_epoch_v5(model, dl_tr, opt, lam, device, sa)
            val_loss = eval_epoch_v5(model, dl_val, lam, device, sa)
            sch.step(val_loss)
            if val_loss < best_val:
                best_val = val_loss
                best_sd  = {k: v.cpu().clone()
                            for k, v in model.state_dict().items()}
        model.load_state_dict(best_sd)
        return _eval_mape_v5(model, dl_tst)

    elif variant["model"] == "v4":
        scalers_v4 = compute_scalers(idx_tr)
        dl_tr4, dl_val4, dl_tst4 = make_dataloaders(
            idx_tr, idx_val, idx_tst, scalers_v4)
        model = build_model()
        opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
        sch   = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, patience=20, factor=0.5)
        best_val, best_sd = float("inf"), None
        for ep in range(1, EP_ABL_V5 + 1):
            lam, sa = get_lambdas_and_active(ep)
            for batch in dl_tr4:
                out = model(batch["X"].to(device), batch["mask"].to(device))
                loss, _ = compute_loss(batch, out, lam, scalers_v4, sa)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                opt.step(); opt.zero_grad()
            val_loss = _eval_v4_loss(model, dl_val4, scalers_v4)
            sch.step(val_loss)
            if val_loss < best_val:
                best_val = val_loss
                best_sd  = {k: v.cpu().clone()
                            for k, v in model.state_dict().items()}
        model.load_state_dict(best_sd)
        return _eval_mape_v4(model, dl_tst4, scalers_v4)


@torch.no_grad()
def _eval_mape_v5(model, dl):
    model.eval()
    Vbx_p, Vby_p, Vbx_t, Vby_t = [], [], [], []
    dx_p, dy_p, dx_t, dy_t = [], [], [], []
    for batch in dl:
        _, _, _, _, vbx, vby, dxb, dyb = model(
            batch["X"].to(device), batch["x_raw"].to(device),
            batch["mask"].to(device), batch["floor_masses"].to(device))
        Vbx_p.append(vbx.cpu().numpy())
        Vby_p.append(vby.cpu().numpy())
        Vbx_t.append(batch["Vbx_raw"].numpy())
        Vby_t.append(batch["Vby_raw"].numpy())
        for i, n in enumerate(batch["N_pisos"].numpy()):
            dx_p.append(dxb[i, :n].max().item())
            dy_p.append(dyb[i, :n].max().item())
            dx_t.append(batch["drift_x_raw"][i, :n].max().item())
            dy_t.append(batch["drift_y_raw"][i, :n].max().item())
    def mape(p, t):
        p = np.concatenate([np.atleast_1d(x) for x in p])
        t = np.concatenate([np.atleast_1d(x) for x in t])
        return 100. * float(np.mean(np.abs(p - t) / (np.abs(t) + 1e-3)))
    return {"Vbx": mape(Vbx_p, Vbx_t), "Vby": mape(Vby_p, Vby_t),
            "drift_x": mape(dx_p, dx_t), "drift_y": mape(dy_p, dy_t)}


@torch.no_grad()
def _eval_v4_loss(model, dl, scalers_v4):
    model.eval(); total = 0.; n = 0
    for batch in dl:
        out = model(batch["X"].to(device), batch["mask"].to(device))
        lam, sa = get_lambdas_and_active(EP_ABL_V5)
        loss, _ = compute_loss(batch, out, lam, scalers_v4, sa)
        total += loss.item(); n += 1
    return total / max(n, 1)


@torch.no_grad()
def _eval_mape_v4(model, dl, scalers_v4):
    model.eval()
    Vbx_p, Vby_p, Vbx_t, Vby_t = [], [], [], []
    dx_p, dy_p, dx_t, dy_t = [], [], [], []
    Vbx_std = scalers_v4["Vbx_std"]; Vbx_mean = scalers_v4["Vbx_mean"]
    Vby_std = scalers_v4["Vby_std"]; Vby_mean = scalers_v4["Vby_mean"]
    Dx_std  = scalers_v4["Dx_std"];  Dx_mean  = scalers_v4["Dx_mean"]
    Dy_std  = scalers_v4["Dy_std"];  Dy_mean  = scalers_v4["Dy_mean"]
    for batch in dl:
        _, _, _, _, _, _, dx, dy, Vb, _ = model(
            batch["X"].to(device), batch["mask"].to(device))
        vbx = np.exp(Vb[:, 0].cpu().numpy() * Vbx_std + Vbx_mean)
        vby = np.exp(Vb[:, 1].cpu().numpy() * Vby_std + Vby_mean)
        Vbx_p.extend(vbx); Vby_p.extend(vby)
        Vbx_t.extend(batch["Vbx_raw"].numpy())
        Vby_t.extend(batch["Vby_raw"].numpy())
        for i, n in enumerate(batch["N_pisos"].numpy()):
            dx_p.append(np.exp(dx[i, :n].max().item() * Dx_std + Dx_mean))
            dy_p.append(np.exp(dy[i, :n].max().item() * Dy_std + Dy_mean))
            dx_t.append(batch["drift_x_raw"][i, :n].max().item())
            dy_t.append(batch["drift_y_raw"][i, :n].max().item())
    mape = lambda p, t: 100. * float(
        np.mean(np.abs(np.array(p) - np.array(t))
                / (np.abs(np.array(t)) + 1e-3)))
    return {"Vbx": mape(Vbx_p, Vbx_t), "Vby": mape(Vby_p, Vby_t),
            "drift_x": mape(dx_p, dx_t), "drift_y": mape(dy_p, dy_t)}


print("CELDA ABL-V5-TRAIN OK")


CELDA ABL-V5-TRAIN OK


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELDA ABLATION V5 — LOOP DE EJECUCIÓN
# RUN_ABLATION_V5 = True para correr
# ══════════════════════════════════════════════════════════════════════
import pandas as pd

ABL_V5_RESULTS_PATH = TRIALS_DIR / vfile("ablation_v5", "json")

if RUN_ABLATION_V5:
    # ── Construir índices del trial 16 (igual que el loop principal) ──
    _tdef = next(t for t in TRIALS_DEF if t["id"] == 16)
    _tid  = _tdef["id"]
    _tipo = _tdef.get("tipo", "conceptual")
    if _tipo == "complemento":
        _idx_all   = np.arange(N_CASES)
        _rng_g     = np.random.default_rng(SEED)
        _idx_pool_A = _rng_g.choice(_idx_all, size=4000, replace=False)
        _idx_pool_B = np.setdiff1d(_idx_all, _idx_pool_A)
    else:
        _idx_pool_A = np.where(_tdef["mask_A"])[0]
        _idx_pool_B = np.where(_tdef["mask_B"])[0]
    _rng_t = np.random.default_rng(SEED + _tid * 100)
    _A_sh  = _rng_t.permutation(_idx_pool_A)
    _idx_A1 = _A_sh[:N_TRAIN];  _idx_A2 = _A_sh[N_TRAIN:2*N_TRAIN]
    _B_sh   = _rng_t.permutation(_idx_pool_B)
    _bloque = _B_sh[:N_VAL_OOD + N_TEST_OOD]
    _idx_val  = _bloque[:N_VAL_OOD]
    _idx_tst  = _bloque[N_VAL_OOD:N_VAL_OOD + N_TEST_OOD]
    print(f"Trial 16: train={len(_idx_A1)}  val={len(_idx_val)}  test={len(_idx_tst)}")

    rows = []
    for var in ABLATION_V5_VARIANTS:
        for rep in range(N_REP_V5):
            idx_tr = _idx_A1 if rep == 0 else _idx_A2
            print(f"  {var['label']}  rep {rep+1}/{N_REP_V5} ...", end=" ", flush=True)
            data_arg = _data_v5 if var["model"] == "v5" else None
            res = run_ablation_trial_v5(
                var, idx_tr, _idx_val, _idx_tst, rep, data_arg)
            rows.append({"variant": var["label"], "rep": rep+1, **res})
            print(f"Vbx={res['Vbx']:.1f}%  Vby={res['Vby']:.1f}%  "
                  f"drift_x={res['drift_x']:.1f}%  drift_y={res['drift_y']:.1f}%")
    df_abl = pd.DataFrame(rows)
    summary = df_abl.groupby("variant")[["Vbx","Vby","drift_x","drift_y"]].mean()
    print("\n=== Ablation V5 — MAPE medio (%) ===")
    print(summary.to_string())
    df_abl.to_json(ABL_V5_RESULTS_PATH)
    print(f"\nGuardado: {ABL_V5_RESULTS_PATH.name}")
else:
    print("RUN_ABLATION_V5 = False  (no ejecutado)")
    if ABL_V5_RESULTS_PATH.exists():
        df_abl = pd.read_json(ABL_V5_RESULTS_PATH)
        summary = df_abl.groupby("variant")[["Vbx","Vby","drift_x","drift_y"]].mean()
        print("=== Resultado previo ===")
        print(summary.to_string())


## Celda Ablation V5 — Full Data (protocolo ablation v4)

Mismo protocolo que el ablation v4 original:
- **Train**: 80% de Pool A
- **Val**: 10% de Pool A
- **Test**: TODO Pool B

Guarda resultados en **época 600** (comparable con ablation v4) y **época 1000** (¿converge más?).

`RUN_ABLATION_V5_FULL = True` para ejecutar.


In [17]:
# ══════════════════════════════════════════════════════════════════════
# CELDA ABLATION V5 FULL — SETUP Y FUNCIÓN DE TRAINING
# ══════════════════════════════════════════════════════════════════════
RUN_ABLATION_V5_FULL = True   # <- cambiar a True para correr
EP_CHECKPOINTS       = [600, 1000]

if RUN_ABLATION_V5_FULL and "_data_v5" not in dir():
    _data_v5 = load_dataset_v5(str(H5_MODAL))
    print("Dataset v5 cargado:", _data_v5["X_clean"].shape)


def run_ablation_trial_v5_full(variant, idx_tr, idx_val, idx_tst, rep,
                               data_v5, ep_checkpoints=EP_CHECKPOINTS):
    """
    Entrena una variante y guarda métricas OOD en cada checkpoint de época.
    Retorna dict: {600: {...}, 1000: {...}}
    """
    import torch, numpy as np
    torch.manual_seed(rep * 1000 + variant["id"])
    np.random.seed(rep * 1000 + variant["id"])
    ep_max = max(ep_checkpoints)
    results = {}

    if variant["model"] == "v5":
        scalers = compute_scalers_v5(data_v5, idx_tr)
        dl_tr, dl_val, dl_tst = make_dataloaders_v5(
            data_v5, idx_tr, idx_val, idx_tst, scalers)
        model = PINNModal_v5(n_inputs=N_INPUTS).to(device)
        opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
        sch   = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, patience=20, factor=0.5)
        best_val, best_sd = float("inf"), None
        for ep in range(1, ep_max + 1):
            lam = dict(variant["lambdas"])
            sa  = dict(variant["stage_active"])
            if ep < 100:
                lam["Vb"] = 0.0; lam["drift"] = 0.0
                sa["phi"] = False; sa["ortho"] = False
            elif ep < 300:
                lam["ortho"] = 0.0; sa["ortho"] = False
            train_epoch_v5(model, dl_tr, opt, lam, device, sa)
            val_loss = eval_epoch_v5(model, dl_val, lam, device, sa)
            sch.step(val_loss)
            if val_loss < best_val:
                best_val = val_loss
                best_sd  = {k: v.cpu().clone()
                            for k, v in model.state_dict().items()}
            if ep in ep_checkpoints:
                model.load_state_dict(best_sd)
                results[ep] = _eval_mape_v5(model, dl_tst)
                print(f"    ep{ep}: Vbx={results[ep]['Vbx']:.1f}%"
                      f"  Vby={results[ep]['Vby']:.1f}%"
                      f"  drift_x={results[ep]['drift_x']:.1f}%"
                      f"  drift_y={results[ep]['drift_y']:.1f}%")
        return results

    elif variant["model"] == "v4":
        scalers_v4 = compute_scalers(idx_tr)
        dl_tr4, dl_val4, dl_tst4 = make_dataloaders(
            idx_tr, idx_val, idx_tst, scalers_v4)
        model = build_model()
        opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
        sch   = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, patience=20, factor=0.5)
        best_val, best_sd = float("inf"), None
        for ep in range(1, ep_max + 1):
            lam, sa = get_lambdas_and_active(ep)
            for batch in dl_tr4:
                out = model(batch["X"].to(device), batch["mask"].to(device))
                loss, _ = compute_loss(batch, out, lam, scalers_v4, sa)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                opt.step(); opt.zero_grad()
            val_loss = _eval_v4_loss(model, dl_val4, scalers_v4)
            sch.step(val_loss)
            if val_loss < best_val:
                best_val = val_loss
                best_sd  = {k: v.cpu().clone()
                            for k, v in model.state_dict().items()}
            if ep in ep_checkpoints:
                model.load_state_dict(best_sd)
                results[ep] = _eval_mape_v4(model, dl_tst4, scalers_v4)
                print(f"    ep{ep}: Vbx={results[ep]['Vbx']:.1f}%"
                      f"  Vby={results[ep]['Vby']:.1f}%"
                      f"  drift_x={results[ep]['drift_x']:.1f}%"
                      f"  drift_y={results[ep]['drift_y']:.1f}%")
        return results


print("CELDA ABL-V5-FULL-SETUP OK")


Loading C:\Users\rodri\Documents\PINN\b2_proyecto\dataset\opensees\dataset_familyB_OPENSEES_v6_20260523_1226.h5...
  N_cases=10000, X_clean=(10000, 21)
Dataset v5 cargado: (10000, 21)
CELDA ABL-V5-FULL-SETUP OK


In [45]:
# ══════════════════════════════════════════════════════════════════════
# CELDA ABLATION V5 FULL — LOOP
# Protocolo: 80% Pool A train / 10% Pool A val / TODO Pool B test
# ══════════════════════════════════════════════════════════════════════
import pandas as pd

ABL_V5_FULL_PATH = TRIALS_DIR / vfile("ablation_v5_full", "json")

if RUN_ABLATION_V5_FULL:
    # ── Construir índices — protocolo ablation v4 ──────────────────
    _tdef   = next(t for t in TRIALS_DEF if t["id"] == 16)
    _tid    = _tdef["id"]
    _tipo   = _tdef.get("tipo", "conceptual")
    if _tipo == "complemento":
        _rng_g  = np.random.default_rng(SEED)
        _idx_A  = _rng_g.choice(np.arange(N_CASES), size=4000, replace=False)
        _idx_B  = np.setdiff1d(np.arange(N_CASES), _idx_A)
    else:
        _idx_A  = np.where(_tdef["mask_A"])[0]
        _idx_B  = np.where(_tdef["mask_B"])[0]   # TODO Pool B como test

    _rng_t  = np.random.default_rng(SEED + _tid * 100)
    _A_sh   = _rng_t.permutation(_idx_A)
    _n_tr   = int(len(_A_sh) * 0.80)
    _n_val  = int(len(_A_sh) * 0.10)
    _idx_tr_full  = _A_sh[:_n_tr]
    _idx_val_full = _A_sh[_n_tr:_n_tr + _n_val]
    _idx_tst_full = _idx_B   # TODO Pool B
    print(f"Full-data split — train={len(_idx_tr_full)}"
          f"  val={len(_idx_val_full)}  test={len(_idx_tst_full)}")

    rows_full = []
    for var in ABLATION_V5_VARIANTS:
        for rep in range(N_REP_V5):
            print(f"\n  {var['label']}  rep {rep+1}/{N_REP_V5}")
            data_arg = _data_v5 if var["model"] == "v5" else None
            res = run_ablation_trial_v5_full(
                var, _idx_tr_full, _idx_val_full, _idx_tst_full,
                rep, data_arg)
            for ep, metrics in res.items():
                rows_full.append({
                    "variant": var["label"], "rep": rep + 1,
                    "epoch": ep, **metrics})

    df_full = pd.DataFrame(rows_full)
    print("\n=== Ablation V5 Full — MAPE medio (%) ===")
    for ep in EP_CHECKPOINTS:
        sub = df_full[df_full["epoch"] == ep]
        summary = sub.groupby("variant")[["Vbx","Vby","drift_x","drift_y"]].mean()
        print(f"\n--- Época {ep} ---")
        print(summary.to_string())
    df_full.to_json(ABL_V5_FULL_PATH)
    print(f"\nGuardado: {ABL_V5_FULL_PATH.name}")
else:
    print("RUN_ABLATION_V5_FULL = False  (no ejecutado)")
    if ABL_V5_FULL_PATH.exists():
        df_full = pd.read_json(ABL_V5_FULL_PATH)
        print("=== Resultado previo ===")
        for ep in EP_CHECKPOINTS:
            sub = df_full[df_full["epoch"] == ep]
            summary = sub.groupby("variant")[["Vbx","Vby","drift_x","drift_y"]].mean()
            print(f"\n--- Época {ep} ---")
            print(summary.to_string())


    ep1000: Vbx=11.6%  Vby=9.5%  drift_x=10.3%  drift_y=5.2%

  (6) PINN v4 (baseline)  rep 2/2
    ep600: Vbx=11.1%  Vby=9.4%  drift_x=10.4%  drift_y=4.3%
    ep1000: Vbx=11.2%  Vby=9.4%  drift_x=10.4%  drift_y=4.3%

=== Ablation V5 Full — MAPE medio (%) ===

--- Época 600 ---
                              Vbx        Vby     drift_x    drift_y
variant                                                            
(1) Solo T supervisado  39.246823  46.376660  163.255721  41.898961
(2) + L_phi (MAC)       48.011929  36.633581  105.147054  27.108523
(3) + L_Vb (NCh433)     40.341215  40.316229   88.815568  55.663474
(4) + L_drift           55.589750  38.962507   70.323065  28.451008
(5) Full PINN v5        38.046540  33.543602   99.694484  35.625979
(6) PINN v4 (baseline)  11.384487   9.457493   10.389577   4.761613

--- Época 1000 ---
                              Vbx        Vby     drift_x    drift_y
variant                                                            
(1) Solo T supervisad

## Optuna — Búsqueda de λ óptimos (PINN v4)

Busca los pesos de pérdida óptimos manteniendo el mismo curriculum staging.
Usa **Trial 16** como proxy (misma configuración que el ablation v4).

- **N_OPTUNA_TRIALS**: 50 trials TPE
- **EP_OPTUNA**: 400 épocas por trial (más rápido que 600)
- **Objetivo**: minimizar MAPE medio (Vbx + Vby + drift_x + drift_y) / 4

Pon `RUN_OPTUNA = True` para ejecutar.


In [20]:
# ══════════════════════════════════════════════════════════════════════
# CELDA OPTUNA — Búsqueda de λ óptimos para PINN v4
# ══════════════════════════════════════════════════════════════════════
import optuna
import torch, numpy as np, json
optuna.logging.set_verbosity(optuna.logging.WARNING)

RUN_OPTUNA       = True   # <- cambiar a True para correr
N_OPTUNA_TRIALS  = 50
EP_OPTUNA        = 600     # épocas por trial
OPTUNA_PATH      = TRIALS_DIR / vfile("optuna_lambdas", "json")

# ── Índices Trial 16 (proxy de optimización) ──────────────────────
_tdef16   = next(t for t in TRIALS_DEF if t["id"] == 16)
_idx16_A  = np.where(_tdef16["mask_A"])[0]
_idx16_B  = np.where(_tdef16["mask_B"])[0]
_rng16    = np.random.default_rng(SEED + 16 * 100)
_A16_sh   = _rng16.permutation(_idx16_A)
_idx16_tr  = _A16_sh[:N_TRAIN]
_B16_sh    = _rng16.permutation(_idx16_B)
_idx16_val = _B16_sh[:N_VAL_OOD]
_idx16_tst = _B16_sh[N_VAL_OOD:N_VAL_OOD + N_TEST_OOD]


def _optuna_objective(trial):
    # ── Samplear lambdas ──────────────────────────────────────────
    L = {
        "data_resp":   trial.suggest_float("data_resp",   0.5,   5.0),
        "data_T_rest": trial.suggest_float("data_T_rest", 0.1,   3.0),
        "data_T1":     trial.suggest_float("data_T1",     0.1,   3.0),
        "T1":          trial.suggest_float("T1",          0.1,   3.0),
        "phi":         trial.suggest_float("phi",         0.01,  1.0),
        "eig":         trial.suggest_float("eig",         0.001, 0.5),
        "ortho":       trial.suggest_float("ortho",       0.0,   1e-3),
        "ratio":       trial.suggest_float("ratio",       0.1,   3.0),
    }

    def _get_lam(ep):
        """Mismo staging que v20 pero con magnitudes de Optuna."""
        lam = {"data_resp": L["data_resp"], "data_T_rest": L["data_T_rest"],
               "data_T1": 0.0, "T1": 0.0, "phi": 0.0,
               "eig": 0.0, "ortho": 0.0, "ratio": L["ratio"]}
        sa  = {"phi": False, "eig": False, "ortho": False}
        if ep >= 150:
            lam["data_T1"] = L["data_T1"]; lam["phi"] = L["phi"]; sa["phi"] = True
        if ep >= 200:
            lam["T1"] = L["T1"]; lam["eig"] = L["eig"]; sa["eig"] = True
        if ep >= 350:
            lam["ortho"] = L["ortho"]; sa["ortho"] = (L["ortho"] > 0)
        return lam, sa

    # ── Entrenar ─────────────────────────────────────────────────
    torch.manual_seed(trial.number)
    np.random.seed(trial.number)
    scalers = compute_scalers(_idx16_tr)
    dl_tr, dl_val, dl_tst = make_dataloaders(
        _idx16_tr, _idx16_val, _idx16_tst, scalers)
    model = build_model().to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
    sch   = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, patience=20, factor=0.5)
    best_val, best_sd = float("inf"), None

    for ep in range(EP_OPTUNA):
        lam, sa = _get_lam(ep)
        for p in model.encoder_T1.parameters():
            p.requires_grad = (ep >= 150)
        for batch in dl_tr:
            out  = model(batch["X"].to(device), batch["mask"].to(device))
            loss, _ = compute_loss(batch, out, lam, scalers, sa)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad()
        # Val loss (last-stage lambdas)
        model.eval()
        with torch.no_grad():
            lam_v, sa_v = _get_lam(EP_OPTUNA - 1)
            val_total = 0.; n_b = 0
            for batch in dl_val:
                out = model(batch["X"].to(device), batch["mask"].to(device))
                l, _ = compute_loss(batch, out, lam_v, scalers, sa_v)
                val_total += l.item(); n_b += 1
        model.train()
        val_total /= max(n_b, 1)
        sch.step(val_total)
        if ep >= 200 and val_total < best_val:
            best_val = val_total
            best_sd  = {k: v.cpu().clone()
                        for k, v in model.state_dict().items()}

    if best_sd is not None:
        model.load_state_dict(best_sd)
    metrics = _eval_mape_v4(model, dl_tst, scalers)
    return (metrics["Vbx"] + metrics["Vby"]
            + metrics["drift_x"] + metrics["drift_y"]) / 4.0


if RUN_OPTUNA:
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42))

    def _cb(s, t):
        if t.number % 5 == 0 or t.value == s.best_value:
            print(f"  trial {t.number:3d}: {t.value:.2f}%"
                  f"  (mejor={s.best_value:.2f}%)")

    study.optimize(_optuna_objective, n_trials=N_OPTUNA_TRIALS,
                   callbacks=[_cb])

    best = study.best_trial
    print(f"\n=== OPTUNA BEST (trial {best.number}) ===")
    print(f"Objetivo: {best.value:.2f}%")
    print("\nLambda óptimos:")
    for k, v in best.params.items():
        print(f"  {k:15s} = {v:.6f}")
    print(f"\nLambdas actuales (referencia):")
    print("  data_resp=2.0  data_T_rest=1.0  data_T1=1.0")
    print("  T1=1.0  phi=0.1  eig=0.1  ortho=1e-5  ratio=1.0")

    result = {"best_value": best.value, "best_params": best.params,
              "n_trials": N_OPTUNA_TRIALS, "ep_optuna": EP_OPTUNA,
              "baseline": {"data_resp":2.0,"data_T_rest":1.0,
                           "data_T1":1.0,"T1":1.0,"phi":0.1,
                           "eig":0.1,"ortho":1e-5,"ratio":1.0}}
    with open(OPTUNA_PATH, "w") as f:
        json.dump(result, f, indent=2)
    print(f"\nGuardado: {OPTUNA_PATH.name}")
else:
    print("RUN_OPTUNA = False  (no ejecutado)")
    if OPTUNA_PATH.exists():
        with open(OPTUNA_PATH) as f:
            res = json.load(f)
        print(f"Resultado previo — objetivo: {res['best_value']:.2f}%")
        print("\nLambda óptimos:")
        for k, v in res["best_params"].items():
            print(f"  {k:15s} = {v:.6f}")
        print("\nLambdas actuales (referencia):")
        for k, v in res["baseline"].items():
            print(f"  {k:15s} = {v}")
    else:
        print("Sin resultados previos.")


C:\Users\rodri\anaconda3\envs\PRUEBAS\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  trial   0: 12.07%  (mejor=12.07%)
  trial   1: 9.56%  (mejor=9.56%)
  trial   2: 7.28%  (mejor=7.28%)
  trial   5: 9.90%  (mejor=7.28%)
  trial  10: 11.06%  (mejor=7.28%)
  trial  14: 7.09%  (mejor=7.09%)
  trial  15: 7.65%  (mejor=7.09%)
  trial  20: 6.99%  (mejor=6.99%)
  trial  25: 9.96%  (mejor=6.99%)
  trial  29: 6.94%  (mejor=6.94%)
  trial  30: 10.68%  (mejor=6.94%)
  trial  35: 8.79%  (mejor=6.94%)
  trial  40: 12.79%  (mejor=6.94%)
  trial  45: 11.29%  (mejor=6.94%)

=== OPTUNA BEST (trial 29) ===
Objetivo: 6.94%

Lambda óptimos:
  data_resp       = 2.072998
  data_T_rest     = 2.761737
  data_T1         = 0.540002
  T1              = 1.207179
  phi             = 0.128216
  eig             = 0.078613
  ortho           = 0.000165
  ratio           = 1.828585

Lambdas actuales (referencia):
  data_resp=2.0  data_T_rest=1.0  data_T1=1.0
  T1=1.0  phi=0.1  eig=0.1  ortho=1e-5  ratio=1.0

Guardado: optuna_lambdas_v8.json


In [27]:
# ══════════════════════════════════════════════════════════════════════
# CELDA OPTUNA VALIDACIÓN — Trial 16 con λ óptimos vs actuales
# ══════════════════════════════════════════════════════════════════════
RUN_OPTUNA_VAL = True   # <- cambiar a True para correr
N_REP_VAL      = 4      # mismas reps que el loop principal
OPTUNA_PATH = TRIALS_DIR / vfile("optuna_lambdas", "json")

if RUN_OPTUNA_VAL:
    # ── Cargar λ óptimos ──────────────────────────────────────────
    with open(OPTUNA_PATH) as f:
        _opt = json.load(f)
    L_OPT = _opt["best_params"]
    print("Lambda óptimos cargados. Objetivo Optuna:", f"{_opt['best_value']:.2f}%")

    def get_lambdas_optuna_final(ep):
        """Mismo curriculum staging v20, magnitudes de Optuna."""
        lam = {"data_resp": L_OPT["data_resp"],
               "data_T_rest": L_OPT["data_T_rest"],
               "data_T1": 0.0, "T1": 0.0, "phi": 0.0,
               "eig": 0.0, "ortho": 0.0,
               "ratio": L_OPT["ratio"]}
        sa = {"phi": False, "eig": False, "ortho": False}
        if ep >= 150:
            lam["data_T1"] = L_OPT["data_T1"]
            lam["phi"]     = L_OPT["phi"]; sa["phi"] = True
        if ep >= 200:
            lam["T1"]  = L_OPT["T1"]
            lam["eig"] = L_OPT["eig"]; sa["eig"] = True
        if ep >= 350:
            lam["ortho"] = L_OPT["ortho"]
            sa["ortho"]  = (L_OPT["ortho"] > 0)
        return lam, sa

    # ── Índices Trial 16 ──────────────────────────────────────────
    _tdef16  = next(t for t in TRIALS_DEF if t["id"] == 16)
    _idx16_A = np.where(_tdef16["mask_A"])[0]
    _idx16_B = np.where(_tdef16["mask_B"])[0]
    _rng16   = np.random.default_rng(SEED + 16 * 100)
    _A16_sh  = _rng16.permutation(_idx16_A)
    _B16_sh  = _rng16.permutation(_idx16_B)

    vbx_r, vby_r, dx_r, dy_r = [], [], [], []
    for rep in range(N_REP_VAL):
        idx_tr  = _A16_sh[rep*N_TRAIN:(rep+1)*N_TRAIN]
        idx_val = _B16_sh[:N_VAL_OOD]
        idx_tst = _B16_sh[N_VAL_OOD:N_VAL_OOD + N_TEST_OOD]
        torch.manual_seed(rep); np.random.seed(rep)
        scalers = compute_scalers(idx_tr)
        dl_tr, dl_val, dl_tst = make_dataloaders(
            idx_tr, idx_val, idx_tst, scalers)
        model = build_model().to(device)
        opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
        sch   = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, patience=20, factor=0.5)
        best_val, best_sd = float("inf"), None
        for ep in range(EP_TOTAL):
            lam, sa = get_lambdas_optuna_final(ep)
            for p in model.encoder_T1.parameters():
                p.requires_grad = (ep >= 150)
            for batch in dl_tr:
                out = model(batch["X"].to(device), batch["mask"].to(device))
                loss, _ = compute_loss(batch, out, lam, scalers, sa)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); opt.zero_grad()
            with torch.no_grad():
                model.eval()
                vt = 0.; nb_ = 0
                for batch in dl_val:
                    out = model(batch["X"].to(device), batch["mask"].to(device))
                    l, _ = compute_loss(batch, out, lam, scalers, sa)
                    vt += l.item(); nb_ += 1
                model.train()
            sch.step(vt / max(nb_, 1))
            if ep >= 250 and vt / max(nb_, 1) < best_val:
                best_val = vt / max(nb_, 1)
                best_sd  = {k: v.cpu().clone()
                            for k, v in model.state_dict().items()}
        model.load_state_dict(best_sd)
        m = _eval_mape_v4(model, dl_tst, scalers)
        vbx_r.append(m["Vbx"]); vby_r.append(m["Vby"])
        dx_r.append(m["drift_x"]); dy_r.append(m["drift_y"])
        print(f"  rep {rep+1}/{N_REP_VAL}: "
              f"Vbx={m['Vbx']:.1f}%  Vby={m['Vby']:.1f}%  "
              f"drift_x={m['drift_x']:.1f}%  drift_y={m['drift_y']:.1f}%")

    import numpy as _np
    print("\n=== Trial 16 — Comparación ===")
    print(f"{"Metrica":12s}  {"Actual (v8)":>12s}  {"Optuna L":>12s}  {"Delta":>8s}")
    print("-" * 50)
    # Resultados actuales de all_results_v8.json trial 16
    with open(TRIALS_DIR / vfile("all_results", "json")) as f:
        _ar = json.load(f)
    _t16 = next(v for v in _ar.values() if v["trial_id"] == 16)
    cur_vb    = _t16["media"]["Vb_MAPE"]
    cur_drift = _t16["media"]["drift_MAPE"]
    new_vb    = (_np.mean(vbx_r) + _np.mean(vby_r)) / 2
    new_drift = (_np.mean(dx_r)  + _np.mean(dy_r))  / 2
    print(f"  Vb_MAPE   {cur_vb:>12.2f}%  {new_vb:>12.2f}%  {new_vb-cur_vb:>+8.2f}%")
    print(f"  drift_MAPE{cur_drift:>12.2f}%  {new_drift:>12.2f}%  {new_drift-cur_drift:>+8.2f}%")
    print(f"  Vbx       {_t16["media"].get("Vbx_MAPE", cur_vb):>12.2f}%  "
          f"{_np.mean(vbx_r):>12.2f}%  {_np.mean(vbx_r)-_t16["media"].get("Vbx_MAPE",cur_vb):>+8.2f}%")
    print(f"  Vby       {_t16["media"].get("Vby_MAPE", cur_vb):>12.2f}%  "
          f"{_np.mean(vby_r):>12.2f}%  {_np.mean(vby_r)-_t16["media"].get("Vby_MAPE",cur_vb):>+8.2f}%")
else:
    print("RUN_OPTUNA_VAL = False  (no ejecutado)")


Lambda óptimos cargados. Objetivo Optuna: 6.94%
  rep 1/4: Vbx=16.2%  Vby=14.6%  drift_x=9.8%  drift_y=6.9%
  rep 2/4: Vbx=11.3%  Vby=10.3%  drift_x=9.8%  drift_y=6.0%
  rep 3/4: Vbx=8.0%  Vby=7.7%  drift_x=9.8%  drift_y=5.5%
  rep 4/4: Vbx=9.3%  Vby=8.7%  drift_x=9.7%  drift_y=5.5%

=== Trial 16 — Comparación ===
Metrica        Actual (v8)      Optuna L     Delta
--------------------------------------------------
  Vb_MAPE           9.95%         10.78%     +0.83%
  drift_MAPE       13.33%          7.89%     -5.44%
  Vbx               9.95%         11.23%     +1.28%
  Vby               9.95%         10.32%     +0.37%


## Resultados Optuna — Análisis

### Configuración
- **50 trials** con sampler TPE (seed=42)
- **600 épocas** por trial
- Trial proxy: **Trial 16** (zona_1_3 vs zona_2)
- Objetivo: minimizar MAPE medio (Vbx + Vby + drift_x + drift_y) / 4

### Lambda óptimos encontrados (trial 29, objetivo = 6.94%)

| Lambda | Actual | Optuna | Cambio |
|---|---|---|---|
| `data_resp` | 2.0 | 2.07 | ≈ igual |
| `data_T_rest` | 1.0 | **2.76** | ↑ 2.76× — modos 2-18 necesitan más supervisión |
| `data_T1` | 1.0 | **0.54** | ↓ encoder_T1 ya tiene suficiente señal |
| `T1` | 1.0 | 1.21 | ≈ igual |
| `phi` | 0.1 | 0.128 | ≈ igual |
| `eig` | 0.1 | **0.079** | ↓ ligeramente sobreweighted |
| `ortho` | 1e-5 | **1.65e-4** | ↑ 16× más fuerte |
| `ratio` | 1.0 | **1.83** | ↑ derivas necesitan más peso |

### Validación en Trial 16 (4 reps × 600 épocas)

| Métrica | Actual (v8) | Optuna λ | Δ |
|---|---|---|---|
| Vb_MAPE | 9.95% | 10.78% | **+0.83% peor** |
| drift_MAPE | 13.33% | **7.89%** | **−5.44% mejor** |

**Detalle por rep:**

| Rep | Vbx | Vby | drift_x | drift_y |
|---|---|---|---|---|
| 1 | 16.2% | 14.6% | 9.8% | 6.9% |
| 2 | 11.3% | 10.3% | 9.8% | 6.0% |
| 3 | 8.0% | 7.7% | 9.8% | 5.5% |
| 4 | 9.3% | 8.7% | 9.7% | 5.5% |

### Conclusiones

1. **drift mejora significativamente (−5.44%)** — el rebalanceo de λ beneficia la predicción de derivas
2. **Vb empeora ligeramente (+0.83%)** — trade-off entre Vb y drift
3. **Alta varianza en rep 1** (Vbx=16.2%) vs reps 3-4 (~8-9%) — los nuevos λ son más sensibles a la semilla inicial
4. **L_ortho NO debe eliminarse** — Optuna la subió 16×, confirmando que el problema era que estaba subponderada, no que fuera perjudicial
5. **data_T_rest es el cambio más relevante** — los modos superiores (T2-T18) estaban recibiendo insuficiente supervisión

### Decisión

Los nuevos λ de Optuna representan una mejora en drift pero con trade-off en Vb y mayor varianza.
Para confirmar si conviene aplicar al modelo final se requiere validación en los 31 trials completos.


In [19]:
# ============================================================
# CELDA 10.1 — DASHBOARD 31 TRIALS V8 — LÍNEAS + XY
# Apunta directamente a all_results_v8.json
# Guarda PNG (500 DPI) + PDF (vectorial)
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import json
from pathlib import Path

# Rutas desde el panel (CELDA 0): TRIALS_DIR, vfile()
# ── Cargar resultados desde V8/ ───────────────────────────────────────────
if 'all_results' not in globals() or not all_results:
    with open(TRIALS_DIR / vfile('all_results','json'), encoding='utf-8') as f:
        all_results = {int(k): v for k, v in json.load(f).items()}

# ── Config ────────────────────────────────────────────────────────────────
CAP_DRIFT = 100.0

def safe_val(r, key, cap=None):
    v = r['media'].get(key)
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    return min(v, cap) if cap else v

def safe_std(r, key):
    v = r['std'].get(key)
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return 0.0
    return v

def safe_rep_val(rep, key, cap=None):
    v = rep['metricas'].get(key)
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    return min(v, cap) if cap else v

tids_ordenados = sorted(all_results.keys())
nombres_cortos = [f"T{t:02d}\n{all_results[t]['nombre'][:18]}"
                  for t in tids_ordenados]

metricas_config = [
    ('MAC_avg',    'MAC_avg',    [0, 1.05],  0.90, 0.70, '',   None),
    ('T1_MAPE',    'T1_MAPE',    [0, 130],   20,   40,   '%',  None),
    ('Vb_MAPE',    'Vb_MAPE',    [0, 80],    15,   30,   '%',  None),
    ('drift_MAPE', 'drift_MAPE', [0, 100],   20,   40,   '%',  CAP_DRIFT),
]

COLORES_REP = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
LABELS_REP  = ['Rep 1 (A1)', 'Rep 2 (A1)', 'Rep 3 (A2)', 'Rep 4 (A2)']

fig, axes = plt.subplots(4, 1, figsize=(28, 22), sharex=True)
fig.suptitle(
    'Comparación 31 Trials OOD — Repeated Hold-Out Diseño Factorial 2×2\n'
    'Dataset v6 · PINNModal_v4b · compute_loss_v16 · curriculum v20\n'
    'Cada punto = 1 repetición  |  Línea = evolución dentro del trial',
    fontsize=13, fontweight='bold', y=0.98)

x_pos = np.arange(len(tids_ordenados))

for ax_idx, (key, titulo, ylim, umbral_ok, umbral_lim, unidad, cap) \
        in enumerate(metricas_config):

    ax = axes[ax_idx]

    for i, tid in enumerate(tids_ordenados):
        tipo = all_results[tid].get('tipo', 'conceptual')
        mac  = all_results[tid]['media']['MAC_avg']
        if tipo == 'aleatorio_puro':
            ax.axvspan(i-0.4, i+0.4, alpha=0.10, color='steelblue')
        elif tipo == 'complemento':
            ax.axvspan(i-0.4, i+0.4, alpha=0.10, color='purple')
        elif mac is not None and mac < 0.70:
            ax.axvspan(i-0.4, i+0.4, alpha=0.08, color='red')

    medias = [safe_val(all_results[t], key, cap) for t in tids_ordenados]
    medias_plot = [m if m is not None else np.nan for m in medias]
    ax.plot(x_pos, medias_plot, color='black', lw=1.5, ls='--',
            alpha=0.4, zorder=2, label='Media trial')

    for rep_idx in range(4):
        vals_rep = []
        x_rep    = []
        for i, tid in enumerate(tids_ordenados):
            reps = all_results[tid].get('reps', [])
            if rep_idx < len(reps):
                v = safe_rep_val(reps[rep_idx], key, cap)
                if v is not None:
                    vals_rep.append(v)
                    x_rep.append(i)
        ax.plot(x_rep, vals_rep,
                color=COLORES_REP[rep_idx], lw=1.0, alpha=0.6,
                marker='o', ms=4, zorder=3,
                label=LABELS_REP[rep_idx])

    stds    = [safe_std(all_results[t], key) for t in tids_ordenados]
    med_arr = np.array(medias_plot, dtype=float)
    std_arr = np.array(stds, dtype=float)
    ax.fill_between(x_pos,
                    np.clip(med_arr - std_arr, ylim[0], ylim[1]),
                    np.clip(med_arr + std_arr, ylim[0], ylim[1]),
                    alpha=0.12, color='gray', label='±std')

    if key == 'MAC_avg':
        ax.axhline(umbral_ok,  color='green',  ls='--', lw=1.2, alpha=0.7,
                   label=f'MAC={umbral_ok} (aceptable)')
        ax.axhline(umbral_lim, color='orange', ls='--', lw=1.0, alpha=0.7,
                   label=f'MAC={umbral_lim} (límite)')
    else:
        ax.axhline(umbral_ok,  color='green',  ls='--', lw=1.2, alpha=0.7,
                   label=f'{umbral_ok}{unidad} (aceptable)')
        ax.axhline(umbral_lim, color='orange', ls='--', lw=1.0, alpha=0.7,
                   label=f'{umbral_lim}{unidad} (límite)')

    for g in [5, 10, 15, 20, 25, 30]:
        if g < len(tids_ordenados):
            ax.axvline(g - 0.5, color='gray', ls=':', lw=0.8, alpha=0.5)

    ax.set_ylabel(f'{titulo} {unidad}', fontsize=10)
    ax.set_ylim(ylim)
    ax.grid(alpha=0.25, axis='y')
    ax.grid(alpha=0.10, axis='x')
    ax.legend(fontsize=7, loc='upper right', ncol=4, framealpha=0.85)

axes[-1].set_xticks(x_pos)
axes[-1].set_xticklabels(nombres_cortos, fontsize=6.5,
                          rotation=45, ha='right')
axes[-1].set_xlabel('Trial', fontsize=10)

leyenda = [
    mpatches.Patch(color='steelblue', alpha=0.3,
                   label='T31 — Aleatorio puro (línea base)'),
    mpatches.Patch(color='purple',    alpha=0.3,
                   label='T30 — Aleatorio vs complemento'),
    mpatches.Patch(color='red',       alpha=0.2,
                   label='MAC<0.70 (fallo total)'),
    mpatches.Patch(color='white',     alpha=0,
                   label='G1:N_pisos  G2:Material  G3:Geometría  '
                         'G4:Sísmica  G5:Tipología  G6:Combinados'),
]
fig.legend(handles=leyenda, loc='lower center', ncol=4,
           fontsize=8, framealpha=0.9, bbox_to_anchor=(0.5, -0.01))

plt.tight_layout(rect=[0, 0.03, 1, 0.96])

out_png = TRIALS_DIR / vfile('dashboard_31_trials', 'png')
out_pdf = TRIALS_DIR / vfile('dashboard_31_trials', 'pdf')
plt.savefig(out_png, dpi=500, bbox_inches='tight', facecolor='white')
plt.savefig(out_pdf,           bbox_inches='tight', facecolor='white')
plt.show()
print(f'Dashboard PNG: {out_png}')
print(f'Dashboard PDF: {out_pdf}')

# ── Gráfico XY pares de métricas ──────────────────────────────────────────
UMBRAL = {
    'MAC_avg'   : 0.90,
    'T1_MAPE'   : 20.0,
    'Vb_MAPE'   : 15.0,
    'drift_MAPE': 20.0,
}

datos = {}
for tid in tids_ordenados:
    r = all_results[tid]
    datos[tid] = {
        'nombre'    : r['nombre'][:22],
        'tipo'      : r.get('tipo', 'conceptual'),
        'MAC'       : r['media']['MAC_avg']     or 0.0,
        'T1'        : r['media']['T1_MAPE']     or 999.0,
        'Vb'        : r['media']['Vb_MAPE']     or 999.0,
        'drift'     : min(r['media']['drift_MAPE'] or 999.0, CAP_DRIFT),
        'MAC_std'   : r['std']['MAC_avg']       or 0.0,
        'T1_std'    : r['std']['T1_MAPE']       or 0.0,
        'Vb_std'    : r['std']['Vb_MAPE']       or 0.0,
        'drift_std' : r['std']['drift_MAPE']    or 0.0,
    }

def cumple(d):
    return (d['MAC']   >= UMBRAL['MAC_avg']    and
            d['T1']    <= UMBRAL['T1_MAPE']    and
            d['Vb']    <= UMBRAL['Vb_MAPE']    and
            d['drift'] <= UMBRAL['drift_MAPE'])

COLOR_OK   = '#2ecc71'
COLOR_FAIL = '#e74c3c'
COLOR_EDGE = '#2c3e50'

TIPO_MARKER = {
    'conceptual'    : 'o',
    'complemento'   : 's',
    'aleatorio_puro': '^',
}

PARES = [
    ('MAC', 'T1',    'MAC_avg [-]',  'T1_MAPE [%]'),
    ('MAC', 'Vb',    'MAC_avg [-]',  'Vb_MAPE [%]'),
    ('MAC', 'drift', 'MAC_avg [-]',  'drift_MAPE [%]'),
    ('T1',  'Vb',    'T1_MAPE [%]', 'Vb_MAPE [%]'),
    ('T1',  'drift', 'T1_MAPE [%]', 'drift_MAPE [%]'),
    ('Vb',  'drift', 'Vb_MAPE [%]', 'drift_MAPE [%]'),
]

UMBRAL_KEY = {
    'MAC'  : UMBRAL['MAC_avg'],
    'T1'   : UMBRAL['T1_MAPE'],
    'Vb'   : UMBRAL['Vb_MAPE'],
    'drift': UMBRAL['drift_MAPE'],
}

fig2, axes2 = plt.subplots(2, 3, figsize=(20, 13))
axes2 = axes2.flatten()
fig2.suptitle(
    'Comparativa 31 Trials — Pares de métricas OOD\n'
    'PINNModal_v4b · compute_loss_v16 · curriculum v20 · HDF5 v6\n'
    'Verde = cumple todos los umbrales  |  Rojo = falla ≥1 umbral',
    fontsize=13, fontweight='bold', y=1.01)

for ax_idx, (mx, my, xlabel, ylabel) in enumerate(PARES):
    ax   = axes2[ax_idx]
    n_ok = 0
    for tid in tids_ordenados:
        d      = datos[tid]
        xval   = d[mx]
        yval   = d[my]
        ok     = cumple(d)
        color  = COLOR_OK if ok else COLOR_FAIL
        marker = TIPO_MARKER.get(d['tipo'], 'o')
        n_ok  += int(ok)

        ax.scatter(xval, yval, s=120, color=color, marker=marker,
                   edgecolors=COLOR_EDGE, linewidths=0.8,
                   zorder=4, alpha=0.90)
        ax.errorbar(xval, yval,
                    xerr=d.get(f'{mx}_std', 0),
                    yerr=d.get(f'{my}_std', 0),
                    fmt='none', ecolor=color, alpha=0.4,
                    elinewidth=1.0, capsize=2, zorder=3)
        ax.annotate(f'T{tid:02d}', xy=(xval, yval),
                    xytext=(3, 3), textcoords='offset points',
                    fontsize=6.5, color=COLOR_EDGE, zorder=5)

    ax.axvline(UMBRAL_KEY[mx], color='gray', ls='--', lw=1.2, alpha=0.7)
    ax.axhline(UMBRAL_KEY[my], color='gray', ls=':',  lw=1.2, alpha=0.7)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(f'{xlabel.split("[")[0].strip()} vs '
                 f'{ylabel.split("[")[0].strip()}  '
                 f'({n_ok}/{len(tids_ordenados)} OK)',
                 fontsize=10, fontweight='bold')
    ax.grid(alpha=0.25)
    ax.tick_params(labelsize=8)

leyenda2 = [
    mpatches.Patch(color=COLOR_OK,   label='Cumple todos los umbrales'),
    mpatches.Patch(color=COLOR_FAIL, label='Falla ≥1 umbral'),
    plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='gray',
               markersize=9, label='Trial conceptual'),
    plt.Line2D([0],[0], marker='s', color='w', markerfacecolor='gray',
               markersize=9, label='Trial complemento'),
    plt.Line2D([0],[0], marker='^', color='w', markerfacecolor='gray',
               markersize=9, label='Trial aleatorio puro'),
    plt.Line2D([0],[0], color='gray', ls='--', lw=1.2, label='Umbral eje X'),
    plt.Line2D([0],[0], color='gray', ls=':',  lw=1.2, label='Umbral eje Y'),
]
fig2.legend(handles=leyenda2, loc='lower center', ncol=4,
            fontsize=9, frameon=True, bbox_to_anchor=(0.5, -0.04))

plt.tight_layout(rect=[0, 0.03, 1, 1])

out2_png = TRIALS_DIR / vfile('comparativa_XY_31trials', 'png')
out2_pdf = TRIALS_DIR / vfile('comparativa_XY_31trials', 'pdf')
plt.savefig(out2_png, dpi=500, bbox_inches='tight', facecolor='white')
plt.savefig(out2_pdf,           bbox_inches='tight', facecolor='white')
plt.show()
print(f'Comparativa XY PNG: {out2_png}')
print(f'Comparativa XY PDF: {out2_pdf}')

# ── Tabla resumen ─────────────────────────────────────────────────────────
print(f'\n{"="*100}')
print(f'RESUMEN — 31 TRIALS V8')
print(f'{"="*100}')
print(f'{"T":>3} {"Nombre":<35} {"MAC_avg":>14} {"T1_MAPE":>14} '
      f'{"Vb_MAPE":>12} {"drift_MAPE":>12} {"Estado"}')
print(f'{"-"*100}')

for tid in tids_ordenados:
    r      = all_results[tid]
    mac    = r['media']['MAC_avg']    or 0.0
    t1     = r['media']['T1_MAPE']    or 999.0
    vb     = r['media']['Vb_MAPE']    or 999.0
    dr     = r['media']['drift_MAPE'] or 999.0
    dr_cap = min(dr, CAP_DRIFT)

    if mac < 0.20:
        estado = '❌ fallo total'
    elif mac >= 0.90 and vb < 15 and dr_cap < 20:
        estado = '✅ excelente'
    elif mac >= 0.90 and vb < 30 and dr_cap < 40:
        estado = '✅ aceptable'
    else:
        estado = '⚠️  revisar'

    print(f'  {tid:>2} {r["nombre"]:<35} '
          f'{mac:.4f}±{r["std"]["MAC_avg"]:.4f}  '
          f'{t1:>6.1f}%±{r["std"]["T1_MAPE"]:>5.1f}%  '
          f'{vb:>5.1f}%±{r["std"]["Vb_MAPE"]:>4.1f}%  '
          f'{dr_cap:>5.1f}%±{r["std"]["drift_MAPE"]:>4.1f}%  '
          f'{estado}')

print(f'{"="*100}')

C:\Users\rodri\AppData\Local\Temp\ipykernel_36592\3883255054.py:152: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Dashboard PNG: C:\Users\rodri\Documents\PINN\b2_proyecto\models\trials_31\V8\dashboard_31_trials_v8.png
Dashboard PDF: C:\Users\rodri\Documents\PINN\b2_proyecto\models\trials_31\V8\dashboard_31_trials_v8.pdf
Comparativa XY PNG: C:\Users\rodri\Documents\PINN\b2_proyecto\models\trials_31\V8\comparativa_XY_31trials_v8.png
Comparativa XY PDF: C:\Users\rodri\Documents\PINN\b2_proyecto\models\trials_31\V8\comparativa_XY_31trials_v8.pdf

RESUMEN — 31 TRIALS V8
  T Nombre                                     MAC_avg        T1_MAPE      Vb_MAPE   drift_MAPE Estado
----------------------------------------------------------------------------------------------------
   1 N_par_vs_impar                      0.9532±0.0225     6.0%±  1.4%    4.4%± 0.1%    7.6%± 0.1%  ✅ excelente
   2 N_bajo_vs_alto                      0.1989±0.0331    35.7%±  1.6%   21.9%± 0.4%   34.7%± 1.7%  ❌ fallo total
   3 N_6a9_vs_14a18                      0.0867±0.0263    52.5%±  1.3%   28.1%± 1.2%   46.1%± 5.2%  ❌ fallo tota

C:\Users\rodri\AppData\Local\Temp\ipykernel_36592\3883255054.py:276: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# ============================================================
# CELDA 10.3 — SCATTER PREDICHO VS REAL — 31 TRIALS
# v8: PINNModal_v4b + compute_loss_v16 + curriculum v20
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import torch
import json
from pathlib import Path
from datetime import datetime

# Rutas desde el panel (CELDA 0): TRIALS_DIR, SCATTER_DIR, vfile()

# ── Cargar all_results si no está en memoria ──────────────────────────────
if 'all_results' not in globals() or not all_results:
    with open(TRIALS_DIR / vfile('all_results','json'), encoding='utf-8') as f:
        all_results = {int(k): v for k, v in json.load(f).items()}
    print(f"all_results cargado: {len(all_results)} trials")

# ── Funciones de métricas ─────────────────────────────────────────────────
def calc_metrics(real, pred):
    real  = np.array(real, dtype=float)
    pred  = np.array(pred, dtype=float)
    mse   = float(np.mean((real - pred)**2))
    rmse  = float(np.sqrt(mse))
    mae   = float(np.mean(np.abs(real - pred)))
    mape  = float(np.mean(np.abs((real - pred) /
                  np.clip(np.abs(real), 1e-8, None))) * 100)
    ss_res = np.sum((real - pred)**2)
    ss_tot = np.sum((real - real.mean())**2)
    r2    = float(1 - ss_res / max(ss_tot, 1e-10))
    num_r = np.sum((real - real.mean()) * (pred - pred.mean()))
    den_r = np.sqrt(np.sum((real - real.mean())**2) *
                    np.sum((pred - pred.mean())**2))
    r_p   = float(num_r / max(den_r, 1e-10))
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae,
            'MAPE': mape, 'R2': r2, 'R': r_p}

def scatter_panel(ax, real, pred, titulo, unidad, color, metricas):
    ax.scatter(real, pred, alpha=0.35, s=10,
               color=color, edgecolors='none', zorder=3)
    lim_min = min(real.min(), pred.min()) * 0.95
    lim_max = max(real.max(), pred.max()) * 1.05
    lim     = [lim_min, lim_max]
    ax.plot(lim, lim, 'k--', lw=1.5, alpha=0.7, label='Ideal y=x', zorder=4)
    x_band = np.linspace(lim_min, lim_max, 100)
    ax.fill_between(x_band, x_band * 0.90, x_band * 1.10,
                    alpha=0.08, color='green', label='±10%')
    txt = (f"MAPE={metricas['MAPE']:.1f}%\n"
           f"R²={metricas['R2']:.4f}\n"
           f"R={metricas['R']:.4f}\n"
           f"RMSE={metricas['RMSE']:.4g}\n"
           f"MAE={metricas['MAE']:.4g}")
    ax.text(0.03, 0.97, txt, transform=ax.transAxes,
            fontsize=7, va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      alpha=0.8, edgecolor='gray'))
    ax.set_xlabel(f'Real [{unidad}]', fontsize=9)
    ax.set_ylabel(f'Predicho [{unidad}]', fontsize=9)
    ax.set_title(titulo, fontsize=9, fontweight='bold')
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.grid(alpha=0.2)
    ax.legend(fontsize=7, loc='lower right')

def mac_panel(ax, mac_Y1, mac_T1, mac_X1):
    bins = np.linspace(0, 1, 26)
    ax.hist(mac_Y1, bins=bins, alpha=0.6, color='#3498db',
            label=f'MAC_Y1 μ={np.mean(mac_Y1):.3f}', density=True)
    ax.hist(mac_T1, bins=bins, alpha=0.6, color='#e74c3c',
            label=f'MAC_T1 μ={np.mean(mac_T1):.3f}', density=True)
    ax.hist(mac_X1, bins=bins, alpha=0.6, color='#2ecc71',
            label=f'MAC_X1 μ={np.mean(mac_X1):.3f}', density=True)
    ax.axvline(0.90, color='green',  ls='--', lw=1.5,
               label='0.90 aceptable (Pastor et al., 2012)')
    ax.axvline(0.80, color='orange', ls='--', lw=1.2,
               label='0.80 límite')
    mac_avg = np.mean(mac_Y1 + mac_T1 + mac_X1)
    ax.text(0.03, 0.97,
            f"MAC_avg={mac_avg:.4f}\n"
            f"Ref: Allemang & Brown (1982)\n"
            f"     Pastor et al. (2012)",
            transform=ax.transAxes, fontsize=7, va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      alpha=0.8, edgecolor='gray'))
    ax.set_xlabel('MAC [-]', fontsize=9)
    ax.set_ylabel('Densidad', fontsize=9)
    ax.set_title('MAC — Calidad formas modales\n'
                 'Modal Assurance Criterion', fontsize=9, fontweight='bold')
    ax.set_xlim(0, 1.05)
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(alpha=0.2)

# ── Función principal por trial ───────────────────────────────────────────
def generar_scatter_trial(tid, trial_info, trial_dir,
                          scalers_fn, make_dl_fn, build_model_fn, device):
    nombre = trial_info['nombre']
    reps   = trial_info.get('reps', [])

    mac_Y1_all = []; mac_T1_all = []; mac_X1_all = []
    T1_pred_all = []; T1_real_all = []
    Vbx_pred_all = []; Vbx_real_all = []
    drift_pred_all = []; drift_real_all = []

    trial_def = next(t for t in TRIALS_DEF if t['id'] == tid)

    tipo = trial_def.get('tipo', 'conceptual')
    if tipo == 'complemento':
        rng_global = np.random.default_rng(SEED)
        idx_all    = np.arange(N_CASES)
        idx_pool_A = rng_global.choice(idx_all, size=4000, replace=False)
        idx_pool_B = np.setdiff1d(idx_all, idx_pool_A)
    elif tipo == 'aleatorio_puro':
        rng_global = np.random.default_rng(SEED)
        idx_all    = np.arange(N_CASES)
        idx_pool_A = rng_global.choice(idx_all, size=5000, replace=False)
        idx_pool_B = rng_global.choice(idx_all, size=5000, replace=False)
    else:
        idx_pool_A = np.where(np.atleast_1d(trial_def['mask_A']))[0]
        idx_pool_B = np.where(np.atleast_1d(trial_def['mask_B']))[0]

    rng_trial      = np.random.default_rng(SEED + tid * 100)
    idx_A_shuffled = rng_trial.permutation(idx_pool_A)
    idx_A1         = idx_A_shuffled[:N_TRAIN]
    idx_A2         = idx_A_shuffled[N_TRAIN:2*N_TRAIN]
    idx_B_shuffled = rng_trial.permutation(idx_pool_B)
    n_por_bloque   = N_VAL_OOD + N_TEST_OOD

    for rep_idx in range(1, len(reps) + 1):
        rep_dir   = trial_dir / f'rep_{rep_idx:02d}'
        best_path = rep_dir / 'best_model.pt'
        if not best_path.exists():
            print(f"    ⚠️  Rep {rep_idx}: sin best_model.pt")
            continue

        idx_tr       = idx_A1 if rep_idx <= 2 else idx_A2
        bloque_val   = idx_B_shuffled[
            (rep_idx-1)*2*n_por_bloque:
            (rep_idx-1)*2*n_por_bloque + n_por_bloque]
        bloque_test  = idx_B_shuffled[
            (rep_idx-1)*2*n_por_bloque + n_por_bloque:
            rep_idx*2*n_por_bloque]
        idx_val_ood  = bloque_val[:N_VAL_OOD]
        idx_test_ood = bloque_test[:N_TEST_OOD]

        scalers       = scalers_fn(idx_tr)
        _, _, dl_test = make_dl_fn(idx_tr, idx_val_ood,
                                   idx_test_ood, scalers)

        logT_std_t  = torch.tensor(scalers['logT_std'],
                                   dtype=torch.float32, device=device)
        logT_mean_t = torch.tensor(scalers['logT_mean'],
                                   dtype=torch.float32, device=device)
        Vbx_std_t   = torch.tensor([scalers['Vbx_std']],
                                   dtype=torch.float32, device=device)
        Vbx_mean_t  = torch.tensor([scalers['Vbx_mean']],
                                   dtype=torch.float32, device=device)
        Dy_std_t    = torch.tensor([scalers['Dy_std']],
                                   dtype=torch.float32, device=device)
        Dy_mean_t   = torch.tensor([scalers['Dy_mean']],
                                   dtype=torch.float32, device=device)

        model = build_model_fn()
        model.load_state_dict(torch.load(best_path, map_location=device))
        model.eval()

        with torch.no_grad():
            for batch in dl_test:
                X_b    = batch['X'].to(device)
                mask_b = batch['mask'].to(device)

                logT_p, Phi_x_p, Phi_y_p, Phi_t_p, \
                    Ux_p, Uy_p, dx_p, dy_p, Vb_p, ratio_p = model(X_b, mask_b)

                for slot, MAC_list, Phi_p, key in [
                    (0, mac_Y1_all, Phi_y_p, 'Phi_y'),
                    (1, mac_T1_all, Phi_t_p, 'Phi_theta'),
                    (2, mac_X1_all, Phi_x_p, 'Phi_x'),
                ]:
                    phi_pred = Phi_p[:, :, slot] * mask_b
                    phi_true = batch[key][:, :, slot].to(device) * mask_b
                    num   = (phi_pred * phi_true).sum(dim=1) ** 2
                    den_p = (phi_pred ** 2).sum(dim=1)
                    den_t = (phi_true ** 2).sum(dim=1)
                    den   = (den_p * den_t).clamp(min=1e-12)
                    valido = (den_t > 1e-10)
                    mac_r  = torch.where(valido, num/den,
                                         torch.zeros_like(num))
                    MAC_list.extend(mac_r[valido].cpu().numpy().tolist())

                T_pred  = torch.exp(logT_p * logT_std_t + logT_mean_t)
                T_true  = batch['T_r'].to(device)
                mask_T1 = T_true[:, 0] > 1e-6
                if mask_T1.sum() > 0:
                    T1_pred_all.extend(
                        T_pred[mask_T1, 0].cpu().numpy().tolist())
                    T1_real_all.extend(
                        T_true[mask_T1, 0].cpu().numpy().tolist())

                Vbx_pred = torch.exp(Vb_p[:, 0] * Vbx_std_t + Vbx_mean_t)
                Vbx_pred_all.extend(
                    Vbx_pred.squeeze().cpu().numpy().tolist())
                Vbx_real_all.extend(batch['Vbx_raw'].numpy().tolist())

                dy_real    = torch.exp(
                    dy_p * Dy_std_t + Dy_mean_t).clamp(min=1e-10) * mask_b
                dx_real    = ratio_p * dy_real
                drift_pred = torch.maximum(
                    dx_real, dy_real).max(dim=1).values
                drift_x_raw = batch['drift_x_raw'].to(device) * mask_b
                drift_y_raw = batch['drift_y_raw'].to(device) * mask_b
                drift_true  = torch.maximum(
                    drift_x_raw, drift_y_raw).max(dim=1).values
                mask_drift  = drift_true > 1e-8
                if mask_drift.sum() > 0:
                    drift_pred_all.extend(
                        drift_pred[mask_drift].cpu().numpy().tolist())
                    drift_real_all.extend(
                        drift_true[mask_drift].cpu().numpy().tolist())

        del model
        torch.cuda.empty_cache()

    T1_pred_a    = np.array(T1_pred_all)
    T1_real_a    = np.array(T1_real_all)
    Vbx_pred_a   = np.array(Vbx_pred_all)
    Vbx_real_a   = np.array(Vbx_real_all)
    drift_pred_a = np.array(drift_pred_all)
    drift_real_a = np.array(drift_real_all)

    if len(T1_real_a) == 0:
        print(f"    ⚠️  Trial {tid}: sin datos — saltando")
        return None

    m_T1    = calc_metrics(T1_real_a,    T1_pred_a)
    m_Vbx   = calc_metrics(Vbx_real_a,  Vbx_pred_a)
    m_drift = calc_metrics(drift_real_a, drift_pred_a)
    mac_avg = float(np.mean(mac_Y1_all + mac_T1_all + mac_X1_all))

    mac_ok = trial_info['media']['MAC_avg']
    vb_ok  = trial_info['media']['Vb_MAPE']
    dr_ok  = trial_info['media']['drift_MAPE']

    if mac_ok < 0.20:
        title_color = 'red'
    elif mac_ok >= 0.99 and vb_ok < 15 and dr_ok < 20:
        title_color = 'green'
    else:
        title_color = 'black'

    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    fig.suptitle(
        f"Trial {tid:02d} — {nombre}\n"
        f"MAC_avg={mac_ok:.4f}  "
        f"T1_MAPE={trial_info['media']['T1_MAPE']:.1f}%  "
        f"Vb_MAPE={vb_ok:.1f}%  drift_MAPE={dr_ok:.1f}%  "
        f"| {len(T1_real_a)} casos (4 reps agregadas)\n"
        f"PINNModal_v4b | compute_loss_v16 | curriculum v20",
        fontsize=10, fontweight='bold', color=title_color
    )

    mac_panel(axes[0], mac_Y1_all, mac_T1_all, mac_X1_all)
    scatter_panel(axes[1], T1_real_a,    T1_pred_a,
                  'T1 — Periodo fundamental', 's',  '#3498db', m_T1)
    scatter_panel(axes[2], Vbx_real_a,   Vbx_pred_a,
                  'Vb_x — Cortante basal',   'kN', '#e74c3c', m_Vbx)
    scatter_panel(axes[3], drift_real_a, drift_pred_a,
                  'drift_max — Deriva máxima', '-', '#2ecc71', m_drift)

    plt.tight_layout()
    out_path = SCATTER_DIR / f'scatter_T{tid:02d}_{nombre}.png'
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.close()

    return {
        'tid'    : tid,
        'nombre' : nombre,
        'n_casos': len(T1_real_a),
        'mac_avg': round(mac_avg, 4),
        'T1'     : m_T1,
        'Vbx'    : m_Vbx,
        'drift'  : m_drift,
    }

# ── Loop principal ────────────────────────────────────────────────────────
print(f"{'='*70}")
print(f"GENERANDO SCATTER PLOTS — 31 TRIALS {VERSION}")
print(f"Guardando en: {SCATTER_DIR}")
print(f"{'='*70}")

t_inicio      = datetime.now()
resultados_sc = []

for tid in sorted(all_results.keys()):
    nombre    = all_results[tid]['nombre']
    trial_dir = TRIALS_DIR / f'trial_{tid:02d}_{nombre}'

    if not trial_dir.exists():
        print(f"  T{tid:02d}: carpeta no encontrada — saltando")
        continue

    print(f"  T{tid:02d} {nombre[:35]}...", end=' ', flush=True)
    t0 = datetime.now()

    res = generar_scatter_trial(
        tid            = tid,
        trial_info     = all_results[tid],
        trial_dir      = trial_dir,
        scalers_fn     = compute_scalers,
        make_dl_fn     = make_dataloaders,
        build_model_fn = build_model,
        device         = device,
    )

    t_el = (datetime.now() - t0).total_seconds()
    if res:
        resultados_sc.append(res)
        print(f"✅ {t_el:.0f}s | "
              f"MAC={res['mac_avg']:.3f} "
              f"T1={res['T1']['MAPE']:.1f}% "
              f"Vb={res['Vbx']['MAPE']:.1f}% "
              f"drift={res['drift']['MAPE']:.1f}%")
    else:
        print(f"❌ {t_el:.0f}s")

t_total = (datetime.now() - t_inicio).total_seconds() / 60

with open(SCATTER_DIR / vfile('metricas_scatter','json'),
          'w', encoding='utf-8') as f:
    json.dump(resultados_sc, f, indent=2, ensure_ascii=False)

print(f"\n{'='*70}")
print(f"COMPLETADO en {t_total:.1f} min")
print(f"  Figuras  : {SCATTER_DIR}")
print(f"  Métricas : {vfile('metricas_scatter','json')}")
print(f"  Refs MAC : Allemang & Brown (1982), Pastor et al. (2012)")
print(f"{'='*70}")

In [ ]:
# CELDA 11 -- COMPARACION FINAL 31 TRIALS + DASHBOARD
# v8: PINNModal_v4b + compute_loss_v16 + ratio activo + curriculum v20

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import pandas as pd
import numpy as np
import json
from pathlib import Path

# Rutas desde el panel (CELDA 0): TRIALS_DIR, vfile()
CAP_DRIFT = 100.0

# ── Cargar resultados desde Version/ ───────────────────────────────────────────
if 'all_results' not in globals() or not all_results:
    with open(TRIALS_DIR / vfile('all_results','json'), encoding='utf-8') as f:
        all_results = {int(k): v for k, v in json.load(f).items()}

# ── Construir DataFrame ───────────────────────────────────────────────────
rows = []
for tid, r in sorted(all_results.items()):
    drift_media = r['media']['drift_MAPE']
    drift_std   = r['std']['drift_MAPE']
    drift_cap   = min(drift_media, CAP_DRIFT) if drift_media else 0.0
    rows.append({
        'Trial'          : tid,
        'Nombre'         : r['nombre'],
        'Tipo'           : r.get('tipo', 'conceptual'),
        'K_reps'         : r['k_reps'],
        'MAC_media'      : r['media']['MAC_avg'],
        'MAC_std'        : r['std']['MAC_avg'],
        'MAC_var_ood'    : r['var_ood']['MAC_avg'],
        'MAC_var_train'  : r['var_train']['MAC_avg'],
        'T1_media'       : r['media']['T1_MAPE'],
        'T1_std'         : r['std']['T1_MAPE'],
        'T1_var_ood'     : r['var_ood']['T1_MAPE'],
        'T1_var_train'   : r['var_train']['T1_MAPE'],
        'Vb_media'       : r['media']['Vb_MAPE'],
        'Vb_std'         : r['std']['Vb_MAPE'],
        'Vb_var_ood'     : r['var_ood']['Vb_MAPE'],
        'Vb_var_train'   : r['var_train']['Vb_MAPE'],
        'drift_media_raw': drift_media,
        'drift_media'    : drift_cap,
        'drift_std'      : drift_std,
        'drift_var_ood'  : r['var_ood']['drift_MAPE'],
        'drift_var_train': r['var_train']['drift_MAPE'],
        'drift_capped'   : drift_media > CAP_DRIFT,
        'Tiempo_min'     : r['tiempo_total_min'],
    })

df_comp = pd.DataFrame(rows).set_index('Trial')

# ── Tabla de resultados ───────────────────────────────────────────────────
print('=' * 115)
print('RESULTADOS — 31 TRIALS OOD — Repeated Hold-Out Diseño Factorial 2×2')
print('Dataset v6 nuevo (B4-INF dividido) — PINNModal_v4b — compute_loss_v16 — curriculum v20')
print('Métricas NCh433: MAC_avg [0-1], T1_MAPE [%], Vb_MAPE [%], drift_MAPE [%]')
print('Formato: media ± std  |  * drift truncado a 100%')
print('=' * 115)
print(f"\n{'T':>3} {'Nombre':<38} {'MAC_avg':>14} {'T1_MAPE':>14} "
      f"{'Vb_MAPE':>14} {'drift_MAPE':>15} {'K':>3}")
print("-"*115)

for tid, row in df_comp.iterrows():
    mac_str   = f"{row['MAC_media']:.4f}±{row['MAC_std']:.4f}"
    t1_str    = f"{row['T1_media']:.2f}%±{row['T1_std']:.2f}%"
    vb_str    = f"{row['Vb_media']:.2f}%±{row['Vb_std']:.2f}%"
    drift_str = f"{row['drift_media']:.2f}%±{row['drift_std']:.2f}%"
    if row['drift_capped']:
        drift_str += '*'
    print(f"  {tid:>2} {row['Nombre']:<38} {mac_str:>14} {t1_str:>14} "
          f"{vb_str:>14} {drift_str:>15} {int(row['K_reps']):>3}")

print('=' * 115)

# ── Guardar CSV en Version/ ────────────────────────────────────────────────────
csv_path = TRIALS_DIR / vfile('comparacion_31_trials','csv')
df_comp.to_csv(csv_path, encoding='utf-8')
print(f"\nGuardado: {csv_path}")

# ── Tabla descomposición de varianza ──────────────────────────────────────
print(f"\n{'='*115}")
print('DESCOMPOSICIÓN DE VARIANZA — |Rep1-Rep2|=var_OOD  |Rep1-Rep3|=var_Train')
print(f"{'='*115}")
print(f"\n{'T':>3} {'Nombre':<38} {'MAC_vOOD':>10} {'MAC_vTR':>10} "
      f"{'T1_vOOD':>10} {'T1_vTR':>10} {'Vb_vOOD':>10} {'Vb_vTR':>10}")
print("-"*115)

for tid, row in df_comp.iterrows():
    def fmt(v):
        return f"{v:.4f}" if v is not None else "N/A"
    print(f"  {tid:>2} {row['Nombre']:<38} "
          f"{fmt(row['MAC_var_ood']):>10} {fmt(row['MAC_var_train']):>10} "
          f"{fmt(row['T1_var_ood']):>10} {fmt(row['T1_var_train']):>10} "
          f"{fmt(row['Vb_var_ood']):>10} {fmt(row['Vb_var_train']):>10}")

print(f"{'='*115}")

# ── Dashboard visual — 4 paneles ──────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(26, 20))
fig.suptitle(
    'Dashboard — 31 Trials OOD\nPINN Modal Espectral NCh433 — Familia B\n'
    'Repeated Hold-Out con Diseño Factorial 2×2\n'
    'Dataset v6 nuevo (B4-INF dividido) | PINNModal_v4b | compute_loss_v16 | curriculum v20\n'
    '* drift_MAPE truncado a 100%',
    fontsize=13, fontweight='bold', y=0.99)

trials_ids     = df_comp.index.tolist()
nombres_cortos = [f"T{t:02d} {df_comp.loc[t,'Nombre'][:22]}"
                  for t in trials_ids]

def color_trial(tid, row):
    if row['Tipo'] == 'aleatorio_puro': return '#3498db'
    if row['Tipo'] == 'complemento':   return '#9b59b6'
    if row['MAC_media'] < 0.20:        return '#e74c3c'
    if row['Vb_media']  > 30:          return '#e67e22'
    if row['drift_media'] > 40:        return '#f39c12'
    return '#2ecc71'

colores = [color_trial(t, df_comp.loc[t]) for t in trials_ids]

config_paneles = [
    (axes[0,0], 'MAC_media',   'MAC_std',
     [0, 1.05], 0.90, 0.70,
     'MAC_avg — Calidad formas modales\n(modos Y, T, X)',
     'MAC_avg [0-1]', 'MAC=', ''),
    (axes[0,1], 'T1_media',    'T1_std',
     [0, 130],  20,   40,
     'T1_MAPE — Error relativo periodo fundamental\n(Sa(T1) en NCh433)',
     'T1_MAPE [%]', '', '%'),
    (axes[1,0], 'Vb_media',    'Vb_std',
     [0, 80],   15,   30,
     'Vb_MAPE — Error relativo corte basal\n((Vbx_MAPE + Vby_MAPE)/2)',
     'Vb_MAPE [%]', '', '%'),
    (axes[1,1], 'drift_media', 'drift_std',
     [0, 100],  20,   40,
     'drift_MAPE — Error relativo deriva máxima\n'
     '(NCh433 Art. 8.3.3) * truncado a 100%',
     'drift_MAPE [%]', '', '%'),
]

for ax, col_m, col_s, xlim, u_ok, u_lim, titulo, xlabel, prefijo, sufijo \
        in config_paneles:

    medias = df_comp[col_m].values
    stds   = df_comp[col_s].values

    ax.barh(nombres_cortos, medias, xerr=stds,
            color=colores, edgecolor='white', lw=0.5,
            error_kw=dict(ecolor='gray', capsize=3, lw=1.2))

    ax.axvline(u_ok,  color='green',  ls='--', lw=1.5,
               label=f'{prefijo}{u_ok}{sufijo} (aceptable)')
    ax.axvline(u_lim, color='orange', ls='--', lw=1.2,
               label=f'{prefijo}{u_lim}{sufijo} (límite)')

    if col_m == 'drift_media':
        for i, (tid, row) in enumerate(df_comp.iterrows()):
            if row['drift_capped']:
                ax.text(97, i, '*', ha='center', va='center',
                        fontsize=9, color='red', fontweight='bold')

    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_title(titulo, fontsize=10)
    ax.set_xlim(xlim)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25, axis='x')
    ax.tick_params(labelsize=7)

leyenda = [
    Patch(color='#2ecc71', label='Generaliza correctamente'),
    Patch(color='#e74c3c', label='Fallo total (MAC < 0.20)'),
    Patch(color='#e67e22', label='Vb_MAPE alto (> 30%)'),
    Patch(color='#f39c12', label='drift_MAPE alto (> 40%)'),
    Patch(color='#9b59b6', label='T30 — aleatorio vs complemento'),
    Patch(color='#3498db', label='T31 — aleatorio puro (línea base)'),
]
fig.legend(handles=leyenda, loc='lower center', ncol=3,
           fontsize=9, framealpha=0.9, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout(rect=[0, 0.04, 1, 0.96])

out_path = TRIALS_DIR / vfile('dashboard_31_trials','png')
plt.savefig(out_path, dpi=130, bbox_inches='tight')
plt.close()
print(f"\nDashboard guardado: {out_path}")

# ── Estadísticas globales ─────────────────────────────────────────────────
print(f"\n{'='*70}")
print('ESTADÍSTICAS GLOBALES — Dataset v6 nuevo — PINNModal_v4b — loss v16')
print(f"{'='*70}")
n_mac_ok   = (df_comp['MAC_media'] >= 0.90).sum()
n_t1_ok    = (df_comp['T1_media']  <= 20).sum()
n_vb_ok    = (df_comp['Vb_media']  <= 15).sum()
n_drift_ok = (df_comp['drift_media'] <= 20).sum()
n_todos    = ((df_comp['MAC_media'] >= 0.90) &
              (df_comp['T1_media']  <= 20)   &
              (df_comp['Vb_media']  <= 15)   &
              (df_comp['drift_media'] <= 20)).sum()

print(f"  Trials con MAC_avg    >= 0.90 : {n_mac_ok}/31")
print(f"  Trials con T1_MAPE    <= 20%  : {n_t1_ok}/31")
print(f"  Trials con Vb_MAPE    <= 15%  : {n_vb_ok}/31")
print(f"  Trials con drift_MAPE <= 20%  : {n_drift_ok}/31")
print(f"  Trials que cumplen TODO       : {n_todos}/31")

if 31 in all_results:
    print(f"\n  Trial 31 (linea base sin distributional shift):")
    r31 = all_results[31]
    print(f"    MAC_avg    : {r31['media']['MAC_avg']:.4f} "
          f"+/- {r31['std']['MAC_avg']:.4f}")
    print(f"    T1_MAPE    : {r31['media']['T1_MAPE']:.2f}% "
          f"+/- {r31['std']['T1_MAPE']:.2f}%")
    print(f"    Vb_MAPE    : {r31['media']['Vb_MAPE']:.2f}% "
          f"+/- {r31['std']['Vb_MAPE']:.2f}%")
    print(f"    drift_MAPE : {r31['media']['drift_MAPE']:.2f}% "
          f"+/- {r31['std']['drift_MAPE']:.2f}%")

print(f"{'='*70}")
print('\nSeleccion del trial ganador: Trial 16 — zona_1_3_vs_zona_2')
print('Criterio: relevancia física NCh433 — zona sísmica determina A0')
print(f'Ver tabla completa en: {csv_path}')

In [ ]:
# CELDA 12 -- GUARDAR MODELO DEFINITIVO
# v7: PINNModal_v4b + compute_loss_v16 + curriculum v20

import shutil
import json
import numpy as np
from datetime import datetime
from pathlib import Path

# Rutas desde el panel (CELDA 0): TRIALS_DIR, DEFINITIVO_DIR, vfile()

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer,)):    return int(obj)
        if isinstance(obj, (np.floating,)):   return float(obj)
        if isinstance(obj, np.ndarray):       return obj.tolist()
        if isinstance(obj, (np.bool_, bool)): return bool(obj)
        return super().default(obj)

# ── CONFIGURAR AQUÍ ───────────────────────────────────────────────────────
TRIAL_GANADOR = 16   # ← Trial 16 — zona_1_3_vs_zona_2
REP_GANADORA  = None # ← None = automático (rep con mejor balance)
# ─────────────────────────────────────────────────────────────────────────

if TRIAL_GANADOR is None:
    print("Definir TRIAL_GANADOR antes de correr esta celda")
else:
    # ── Cargar resultados desde V7/ ───────────────────────────────────────
    with open(TRIALS_DIR / vfile('all_results','json'), encoding='utf-8') as f:
        all_results = {int(k): v for k, v in json.load(f).items()}

    r      = all_results[TRIAL_GANADOR]
    nombre = r['nombre']

    print(f"{'='*70}")
    print(f"MODELO DEFINITIVO — Trial {TRIAL_GANADOR:02d} — {nombre}")
    print(f"Dataset: v6 — PINNModal_v4b — curriculum v20")
    print(f"{'='*70}")
    print(f"\nMetricas Fase 1 (media +/- std sobre {r['k_reps']} reps):")
    for k in ['MAC_avg', 'T1_MAPE', 'Vb_MAPE', 'drift_MAPE']:
        unidad = '' if k == 'MAC_avg' else '%'
        print(f"  {k:<14}: {r['media'][k]:.4f} +/- {r['std'][k]:.4f} {unidad}")

    # ── Seleccionar rep ganadora — mejor balance ──────────────────────────
    if REP_GANADORA is None:
        scores = []
        for rep in r['reps']:
            m   = rep['metricas']
            mac = m['MAC_avg']
            t1  = m['T1_MAPE']
            vb  = m['Vb_MAPE']
            dr  = m['drift_MAPE']
            score = mac - (t1/100)*0.3 - (vb/100)*0.4 - (dr/100)*0.3
            scores.append(score)
        REP_GANADORA = scores.index(max(scores)) + 1
        print(f"\nRep ganadora (automático — mejor balance):")
        print(f"  Scores: {[round(s,4) for s in scores]}")
        print(f"  → Rep {REP_GANADORA} seleccionada")
    else:
        print(f"\nRep ganadora (manual): Rep {REP_GANADORA}")

    rep_info = r['reps'][REP_GANADORA - 1]
    print(f"\nMetricas de Rep {REP_GANADORA}:")
    for k in ['MAC_avg', 'T1_MAPE', 'Vb_MAPE', 'drift_MAPE']:
        unidad = '' if k == 'MAC_avg' else '%'
        print(f"  {k:<14}: {rep_info['metricas'][k]:.4f} {unidad}")
    print(f"  best_epoch      : {rep_info['best_epoch']}")
    print(f"  best_epoch_mean : {r['best_epoch_mean']}")

    # DEFINITIVO_DIR viene del panel (CELDA 0)
    trial_dir = TRIALS_DIR / f'trial_{TRIAL_GANADOR:02d}_{nombre}'
    rep_dir   = trial_dir      / f'rep_{REP_GANADORA:02d}'

    # ── Copiar archivos ───────────────────────────────────────────────────
    shutil.copy(rep_dir   / 'best_model.pt',
                DEFINITIVO_DIR / 'model_definitivo.pt')
    shutil.copy(rep_dir   / 'history.csv',
                DEFINITIVO_DIR / 'history_rep_ganadora.csv')
    shutil.copy(trial_dir / 'resumen.json',
                DEFINITIVO_DIR / 'resumen_trial_ganador.json')

    for csv_name in [vfile('comparacion_31_trials','csv')]:
        if (TRIALS_DIR / csv_name).exists():
            shutil.copy(TRIALS_DIR / csv_name,
                        DEFINITIVO_DIR / csv_name)

    # ── Guardar info completa ─────────────────────────────────────────────
    stamp = datetime.now().strftime('%Y%m%d_%H%M')
    info  = {
        'created'              : stamp,
        'trial_ganador'        : TRIAL_GANADOR,
        'nombre'               : nombre,
        'rep_ganadora'         : REP_GANADORA,
        'criterio_seleccion'   : (
            'Trial 16 — zona_1_3_vs_zona_2 — ganador por relevancia física NCh433. '
            'La zona sísmica determina A0 (0.20g/0.30g/0.40g) — variable de mayor '
            'impacto en el espectro de diseño. Generalizar entre zonas es el test '
            'OOD más exigente desde el punto de vista normativo chileno.'
        ),
        'arquitectura'         : 'PINNModal_v4b — encoder_T1 separado, dx via ratio sigmoid',
        'loss'                 : 'compute_loss_v16',
        'curriculum'           : 'v20_staged_training_600ep',
        'n_inputs'             : 21,
        'features_eliminadas'  : ['mods_por_depto', 'activar_B3'],
        'razon_eliminacion'    : 'Constantes en dataset v6 (std=0) — cero información',
        'metricas_media'       : r['media'],
        'metricas_std'         : r['std'],
        'metricas_var_ood'     : r['var_ood'],
        'metricas_var_train'   : r['var_train'],
        'metricas_rep_ganadora': rep_info['metricas'],
        'best_epoch_mean'      : r['best_epoch_mean'],
        'best_epochs_reps'     : r['best_epochs'],
        'n_train_fase1'        : r['n_train'],
        'nota_fase2'           : (
            f"Fase 2: reentrenar con Pool A completo del Trial {TRIAL_GANADOR} "
            f"({nombre}) durante 1000 epocas con curriculum v20"
        ),
        'n_modos'              : 18,
        'n_max_pisos'          : 18,
        'hidden'               : 256,
        'hidden_T1'            : 128,
        'ep_total_fase1'       : 600,
        'ep_total_fase2'       : 1000,
        'dataset'              : 'dataset_familyB_OPENSEES_v6_20260518_0811',
        'norma'                : 'NCh433 Mod.Of2012 + DS61',
        'familia'              : 'Familia B',
        'metodologia'          : 'Repeated Hold-Out OOD — Diseno Factorial 2x2',
        'referencia_metodologia': 'Arlot & Celisse (2010), Statistics Surveys, 4, 40-79',
        'referencias_arquitectura': [
            'Wang et al. (2022) — gradient pathologies PINNs',
            'Bengio et al. (2009) — Curriculum Learning, ICML',
            'Howard & Ruder (2018) — ULMFiT, ACL',
            'Loshchilov & Hutter (2017) — CosineAnnealingWarmRestarts',
            'Raissi et al. (2019) — Journal Comp. Physics 378',
        ],
        'total_trials'         : len(all_results),
        'k_reps'               : r['k_reps'],
    }

    with open(DEFINITIVO_DIR / f'info_definitivo_{stamp}.json',
              'w', encoding='utf-8') as f:
        json.dump(info, f, indent=4, ensure_ascii=False, cls=NpEncoder)

    # ── Print resumen ─────────────────────────────────────────────────────
    print(f"\n{'='*70}")
    print(f"MODELO DEFINITIVO GUARDADO")
    print(f"{'='*70}")
    print(f"Directorio: {DEFINITIVO_DIR}")
    for p in sorted(DEFINITIVO_DIR.iterdir()):
        print(f"  {p.name}")

    print(f"\n{'='*70}")
    print(f"FASE 2 — Reentrenamiento con datos completos")
    print(f"{'='*70}")
    print(f"  Trial         : {TRIAL_GANADOR:02d} — {nombre}")
    print(f"  Dataset       : v6 (masa efectiva >= 90%)")
    print(f"  Pool A Fase 1 : {r['n_train']} casos")
    print(f"  Pool A Fase 2 : todos los disponibles (~6556 casos)")
    print(f"  Épocas Fase 2 : 1000 (curriculum v20)")
    print(f"  N_INPUTS      : 21 (sin mods_por_depto ni activar_B3)")
    print(f"\n  Ejecutar Celda 13 para lanzar Fase 2")

In [12]:
# ============================================================
# CELDA 13 — FASE 2: REENTRENAMIENTO CON DATOS COMPLETOS
# Trial 16 — zona_1_3_vs_zona_2
# Arquitectura definitiva: PINNModal_v4b
# Loss definitiva:         compute_loss_v16
# Curriculum definitivo:   v20 staged training 600 épocas
# Dataset:                 v6 nuevo — B4-INF dividido por hall
# ============================================================

from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# DEFINITIVO_DIR y TRIALS_DIR vienen del panel (CELDA 0)

TRIAL_FASE2  = 16
NOMBRE_FASE2 = 'zona_1_3_vs_zona_2'
EP_FASE2     = 600
SEED_FASE2   = 2026

variables_necesarias = ['X_clean', 'N_pisos_arr', 'COLS_BASE',
                        'T_r', 'Vb_x', 'Vb_y', 'drift_x', 'drift_y',
                        'build_model', 'compute_scalers',
                        'make_dataloaders', 'compute_metrics_ood',
                        'compute_loss', 'TRIALS_DEF',
                        'N_MODOS', 'N_MODOS_PHYS', 'N_MAX_PISOS',
                        'get_lambdas_and_active']
faltantes = [v for v in variables_necesarias if v not in globals()]
if faltantes:
    print(f"⚠️  Faltan: {faltantes}")
    raise SystemExit

# ── Datos ──────────────────────────────────────────────────────────────────
trial_def  = next(t for t in TRIALS_DEF if t['id'] == TRIAL_FASE2)
idx_pool_A = np.where(trial_def['mask_A'])[0]
idx_pool_B = np.where(trial_def['mask_B'])[0]

rng_b       = np.random.default_rng(SEED_FASE2 + TRIAL_FASE2 * 200)
idx_B_shuf  = rng_b.permutation(idx_pool_B)
idx_val_f2  = idx_B_shuf[:400]
idx_test_f2 = idx_B_shuf[400:]

scalers_f2 = compute_scalers(idx_pool_A)
dl_tr_f2, dl_val_f2, dl_test_f2 = make_dataloaders(
    idx_pool_A, idx_val_f2, idx_test_f2, scalers_f2)

print(f"\n{'='*70}")
print(f"FASE 2 — Trial {TRIAL_FASE2:02d} — {NOMBRE_FASE2}")
print(f"{'='*70}")
print(f"  Pool A (TRAIN): {len(idx_pool_A)} casos")
print(f"  Pool B (OOD)  : {len(idx_pool_B)} casos")
print(f"  Val           : {len(idx_val_f2)} casos")
print(f"  Test          : {len(idx_test_f2)} casos")
print(f"  Épocas        : {EP_FASE2}")
print(f"  Arquitectura  : PINNModal_v4b")
print(f"  Loss          : compute_loss_v16")
print(f"  Curriculum    : v20 staged training")
print(f"  HDF5          : {H5_MODAL.name}")
print(f"  Guardado en   : {DEFINITIVO_DIR}")

# ── Entrenamiento ──────────────────────────────────────────────────────────
print(f"\n{'─'*60}")
print(f"Iniciando entrenamiento Fase 2...")
print(f"{'─'*60}")

model_f2  = build_model()
best_path = DEFINITIVO_DIR / 'model_fase2_definitivo.pt'

optimizer = torch.optim.Adam(model_f2.parameters(), lr=1e-3)
scheduler = CosineAnnealingWarmRestarts(
    optimizer, T_0=200, T_mult=2, eta_min=1e-6)

params_T1 = (list(model_f2.encoder_T1.parameters()) +
             list(model_f2.head_T1.parameters()))

history_f2      = []
best_val_global = float('inf')
best_epoch_f2   = 0
etapa_actual    = -1
stage_actual    = -1
t_inicio        = datetime.now()

for ep in range(EP_FASE2):
    lambdas, active = get_lambdas_and_active(ep)

    # ── Congelar/descongelar encoder_T1 ──────────────────────────────────
    if ep < 150:
        if etapa_actual != 0:
            etapa_actual = 0
            for p in params_T1:
                p.requires_grad = False
            print(f"\n  ── Etapa 0 (ep {ep}) encoder_T1 CONGELADO ──")
    else:
        if etapa_actual != 1:
            etapa_actual = 1
            for p in params_T1:
                p.requires_grad = True
            print(f"\n  ── Etapa 1 (ep {ep}) encoder_T1 DESCONGELADO ──")

    stage_nuevo = {150:1, 200:2, 350:3, 500:4}.get(ep, stage_actual)
    if stage_nuevo != stage_actual and ep >= 150:
        stage_actual = stage_nuevo
        print(f"\n  ── Stage {stage_actual} (ep {ep}) "
              f"T1λ={lambdas['T1']} "
              f"eig={lambdas['eig']} "
              f"lr={optimizer.param_groups[0]['lr']:.2e} ──")

    # ── Train ─────────────────────────────────────────────────────────────
    model_f2.train()
    tr_losses = {}
    for batch in dl_tr_f2:
        optimizer.zero_grad()
        out  = model_f2(batch['X'].to(device),
                        batch['mask'].to(device))
        loss, ld = compute_loss(batch, out, lambdas, scalers_f2, active)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_f2.parameters(), 1.0)
        optimizer.step()
        for k, v in ld.items():
            tr_losses[k] = tr_losses.get(k, 0) + v

    # ── Val ───────────────────────────────────────────────────────────────
    model_f2.eval()
    val_losses = {}
    with torch.no_grad():
        for batch in dl_val_f2:
            out  = model_f2(batch['X'].to(device),
                            batch['mask'].to(device))
            _, ld = compute_loss(batch, out, lambdas, scalers_f2, active)
            for k, v in ld.items():
                val_losses[k] = val_losses.get(k, 0) + v

    val_total = sum(lambdas.get(k, 0) * v
                    for k, v in val_losses.items())
    scheduler.step()

    if val_total < best_val_global and ep >= 250:
        best_val_global = val_total
        torch.save(model_f2.state_dict(), best_path)
        best_epoch_f2 = ep

    row = {'etapa': etapa_actual, 'stage': stage_actual,
           'epoch': ep, 'val_total': val_total,
           'lr': optimizer.param_groups[0]['lr']}
    row.update({f'tr_{k}': v/max(len(dl_tr_f2),1)
                for k,v in tr_losses.items()})
    row.update({f'val_{k}': v/max(len(dl_val_f2),1)
                for k,v in val_losses.items()})
    history_f2.append(row)

    if ep % 25 == 0:
        frozen_str = 'T1_frozen' if ep < 150 else 'T1_free'
        print(f"  ep{ep:4d}/{EP_FASE2} "
              f"val={val_total:.5f} "
              f"best={best_val_global:.5f} (ep{best_epoch_f2}) "
              f"lr={optimizer.param_groups[0]['lr']:.2e} "
              f"[{frozen_str}]")

t_total = (datetime.now() - t_inicio).total_seconds() / 60
print(f"\n  Completado en {t_total:.1f} min | best_epoch={best_epoch_f2}")

pd.DataFrame(history_f2).to_csv(
    DEFINITIVO_DIR / 'history_fase2_definitivo.csv', index=False)

# ── Evaluación ────────────────────────────────────────────────────────────
print(f"\n{'─'*60}")
print(f"Evaluando sobre Test OOD ({len(idx_test_f2)} casos)...")
print(f"{'─'*60}")

model_f2.load_state_dict(torch.load(best_path, map_location=device))
metrics_f2 = compute_metrics_ood(model_f2, dl_test_f2, scalers_f2)

with open(TRIALS_DIR / vfile('all_results','json'), encoding='utf-8') as f:
    r_f1 = json.load(f)[str(TRIAL_FASE2)]

print(f"\n{'='*70}")
print(f"COMPARACION FASE 1 vs FASE 2 — Trial {TRIAL_FASE2:02d} {NOMBRE_FASE2}")
print(f"{'='*70}")
print(f"  {'Metrica':<14} {'Fase 1':>12} {'Fase 2':>12} {'Mejora':>12}")
print(f"  {'-'*52}")
for k, label in [('MAC_avg',    'MAC_avg   '),
                 ('T1_MAPE',    'T1_MAPE % '),
                 ('Vb_MAPE',    'Vb_MAPE % '),
                 ('drift_MAPE', 'drift_MAPE%')]:
    v_f1 = r_f1['media'][k]
    v_f2 = metrics_f2[k]
    if k == 'MAC_avg':
        mejora = f"+{(v_f2-v_f1)*100:.3f}pp"
    else:
        pct    = ((v_f1 - v_f2) / v_f1) * 100
        mejora = f"{pct:+.1f}%{'↓' if pct>0 else '↑'}"
    print(f"  {label:<14} {v_f1:>12.4f} {v_f2:>12.4f} {mejora:>12}")

print(f"{'='*70}")
print(f"  Pool A Fase 1 : {r_f1['n_train']} casos")
print(f"  Pool A Fase 2 : {len(idx_pool_A)} casos")
print(f"  best_epoch    : {best_epoch_f2}")
print(f"{'='*70}")

# ── Guardar ───────────────────────────────────────────────────────────────
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer,)):    return int(obj)
        if isinstance(obj, (np.floating,)):   return float(obj)
        if isinstance(obj, np.ndarray):       return obj.tolist()
        if isinstance(obj, (np.bool_, bool)): return bool(obj)
        return super().default(obj)

resultado_f2 = {
    'trial'        : TRIAL_FASE2,
    'nombre'       : NOMBRE_FASE2,
    'arquitectura' : 'PINNModal_v4b',
    'loss'         : 'compute_loss_v16',
    'curriculum'   : 'v20_staged_training',
    'dataset'      : 'v6 nuevo — B4-INF dividido por hall (20260523)',
    'h5_usado'     : H5_MODAL.name,
    'n_train_fase2': len(idx_pool_A),
    'n_test_fase2' : len(idx_test_f2),
    'ep_fase2'     : EP_FASE2,
    'best_epoch'   : best_epoch_f2,
    'tiempo_min'   : round(t_total, 2),
    'metricas_f2'  : metrics_f2,
    'metricas_f1'  : {k: r_f1['media'][k]
                      for k in ['MAC_avg','T1_MAPE',
                                'Vb_MAPE','drift_MAPE']},
    'referencias'  : [
        'Wang et al. (2022) — gradient pathologies PINNs',
        'Bengio et al. (2009) — Curriculum Learning',
        'Howard & Ruder (2018) — ULMFiT',
        'Loshchilov & Hutter (2017) — CosineAnnealingWarmRestarts',
        'Raissi et al. (2019) — Journal Comp. Physics 378',
        'NCh433 Mod.2012 + DS61',
    ],
}

with open(DEFINITIVO_DIR / 'resultado_fase2_definitivo.json',
          'w', encoding='utf-8') as f:
    json.dump(resultado_f2, f, indent=4,
              ensure_ascii=False, cls=NpEncoder)

print(f"\nArchivos guardados en: {DEFINITIVO_DIR}")
print(f"  model_fase2_definitivo.pt")
print(f"  history_fase2_definitivo.csv")
print(f"  resultado_fase2_definitivo.json")


FASE 2 — Trial 16 — zona_1_3_vs_zona_2
  Pool A (TRAIN): 6623 casos
  Pool B (OOD)  : 3377 casos
  Val           : 400 casos
  Test          : 2977 casos
  Épocas        : 600
  Arquitectura  : PINNModal_v4b
  Loss          : compute_loss_v16
  Curriculum    : v20 staged training
  HDF5          : dataset_familyB_OPENSEES_v6_20260523_1226.h5
  Guardado en   : C:\Users\rodri\Documents\PINN\b2_proyecto\models\trials_31\V8\DEFINITIVO

────────────────────────────────────────────────────────────
Iniciando entrenamiento Fase 2...
────────────────────────────────────────────────────────────

  ── Etapa 0 (ep 0) encoder_T1 CONGELADO ──
  ep   0/600 val=11.79155 best=inf (ep0) lr=1.00e-03 [T1_frozen]
  ep  25/600 val=9.35754 best=inf (ep0) lr=9.59e-04 [T1_frozen]
  ep  50/600 val=8.78910 best=inf (ep0) lr=8.48e-04 [T1_frozen]
  ep  75/600 val=8.69098 best=inf (ep0) lr=6.84e-04 [T1_frozen]
  ep 100/600 val=8.55127 best=inf (ep0) lr=4.93e-04 [T1_frozen]
  ep 125/600 val=8.73460 best=inf (ep0) l

In [ ]:
# ============================================================
# CELDA 13.1 — FASE 2 MULTI-SEMILLA (media ± std)
# Misma lógica y mismos prints que la CELDA 13.
# Lo único nuevo: bucle sobre varias semillas + agregación final.
# ============================================================

from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# DEFINITIVO_DIR y TRIALS_DIR vienen del panel (CELDA 0)

TRIAL_FASE2  = 16
NOMBRE_FASE2 = 'zona_1_3_vs_zona_2'
EP_FASE2     = 600
SEEDS_FASE2  = [2026, 1234, 777, 99]   # ← una corrida por semilla (4 = simétrico a Fase 1)

variables_necesarias = ['X_clean', 'N_pisos_arr', 'COLS_BASE',
                        'T_r', 'Vb_x', 'Vb_y', 'drift_x', 'drift_y',
                        'build_model', 'compute_scalers',
                        'make_dataloaders', 'compute_metrics_ood',
                        'compute_loss', 'TRIALS_DEF',
                        'N_MODOS', 'N_MODOS_PHYS', 'N_MAX_PISOS',
                        'get_lambdas_and_active']
faltantes = [v for v in variables_necesarias if v not in globals()]
if faltantes:
    print(f"⚠️  Faltan: {faltantes}")
    raise SystemExit

# ── Datos (Pool A/B fijos; el split val/test depende de la semilla) ─────────
trial_def  = next(t for t in TRIALS_DEF if t['id'] == TRIAL_FASE2)
idx_pool_A = np.where(trial_def['mask_A'])[0]
idx_pool_B = np.where(trial_def['mask_B'])[0]

print(f"\n{'='*70}")
print(f"FASE 2 MULTI-SEMILLA — Trial {TRIAL_FASE2:02d} — {NOMBRE_FASE2}")
print(f"{'='*70}")
print(f"  Pool A (TRAIN): {len(idx_pool_A)} casos")
print(f"  Pool B (OOD)  : {len(idx_pool_B)} casos")
print(f"  Semillas      : {SEEDS_FASE2}  ({len(SEEDS_FASE2)} corridas)")
print(f"  Épocas/corrida: {EP_FASE2}")
print(f"  Arquitectura  : PINNModal_v4b")
print(f"  Loss          : compute_loss_v16")
print(f"  Curriculum    : v20 staged training")
print(f"  HDF5          : {H5_MODAL.name}")
print(f"  Guardado en   : {DEFINITIVO_DIR}")

corridas = []

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║                       BUCLE SOBRE SEMILLAS                                ║
# ╚══════════════════════════════════════════════════════════════════════════╝
for SEED_FASE2 in SEEDS_FASE2:

    torch.manual_seed(SEED_FASE2)
    np.random.seed(SEED_FASE2)

    rng_b       = np.random.default_rng(SEED_FASE2 + TRIAL_FASE2 * 200)
    idx_B_shuf  = rng_b.permutation(idx_pool_B)
    idx_val_f2  = idx_B_shuf[:400]
    idx_test_f2 = idx_B_shuf[400:]

    scalers_f2 = compute_scalers(idx_pool_A)
    dl_tr_f2, dl_val_f2, dl_test_f2 = make_dataloaders(
        idx_pool_A, idx_val_f2, idx_test_f2, scalers_f2)

    print(f"\n{'#'*70}")
    print(f"### SEMILLA {SEED_FASE2}")
    print(f"{'#'*70}")
    print(f"  Val           : {len(idx_val_f2)} casos")
    print(f"  Test          : {len(idx_test_f2)} casos")

    # ── Entrenamiento ──────────────────────────────────────────────────────
    print(f"\n{'─'*60}")
    print(f"Iniciando entrenamiento Fase 2 (semilla {SEED_FASE2})...")
    print(f"{'─'*60}")

    model_f2  = build_model()
    best_path = DEFINITIVO_DIR / f'model_fase2_seed{SEED_FASE2}.pt'

    optimizer = torch.optim.Adam(model_f2.parameters(), lr=1e-3)
    scheduler = CosineAnnealingWarmRestarts(
        optimizer, T_0=200, T_mult=2, eta_min=1e-6)

    params_T1 = (list(model_f2.encoder_T1.parameters()) +
                 list(model_f2.head_T1.parameters()))

    history_f2      = []
    best_val_global = float('inf')
    best_epoch_f2   = 0
    etapa_actual    = -1
    stage_actual    = -1
    t_inicio        = datetime.now()

    for ep in range(EP_FASE2):
        lambdas, active = get_lambdas_and_active(ep)

        # ── Congelar/descongelar encoder_T1 ──────────────────────────────
        if ep < 150:
            if etapa_actual != 0:
                etapa_actual = 0
                for p in params_T1:
                    p.requires_grad = False
                print(f"\n  ── Etapa 0 (ep {ep}) encoder_T1 CONGELADO ──")
        else:
            if etapa_actual != 1:
                etapa_actual = 1
                for p in params_T1:
                    p.requires_grad = True
                print(f"\n  ── Etapa 1 (ep {ep}) encoder_T1 DESCONGELADO ──")

        stage_nuevo = {150:1, 200:2, 350:3, 500:4}.get(ep, stage_actual)
        if stage_nuevo != stage_actual and ep >= 150:
            stage_actual = stage_nuevo
            print(f"\n  ── Stage {stage_actual} (ep {ep}) "
                  f"T1λ={lambdas['T1']} "
                  f"eig={lambdas['eig']} "
                  f"lr={optimizer.param_groups[0]['lr']:.2e} ──")

        # ── Train ─────────────────────────────────────────────────────────
        model_f2.train()
        tr_losses = {}
        for batch in dl_tr_f2:
            optimizer.zero_grad()
            out  = model_f2(batch['X'].to(device),
                            batch['mask'].to(device))
            loss, ld = compute_loss(batch, out, lambdas, scalers_f2, active)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_f2.parameters(), 1.0)
            optimizer.step()
            for k, v in ld.items():
                tr_losses[k] = tr_losses.get(k, 0) + v

        # ── Val ───────────────────────────────────────────────────────────
        model_f2.eval()
        val_losses = {}
        with torch.no_grad():
            for batch in dl_val_f2:
                out  = model_f2(batch['X'].to(device),
                                batch['mask'].to(device))
                _, ld = compute_loss(batch, out, lambdas, scalers_f2, active)
                for k, v in ld.items():
                    val_losses[k] = val_losses.get(k, 0) + v

        val_total = sum(lambdas.get(k, 0) * v
                        for k, v in val_losses.items())
        scheduler.step()

        if val_total < best_val_global and ep >= 250:
            best_val_global = val_total
            torch.save(model_f2.state_dict(), best_path)
            best_epoch_f2 = ep

        row = {'seed': SEED_FASE2, 'etapa': etapa_actual,
               'stage': stage_actual, 'epoch': ep,
               'val_total': val_total,
               'lr': optimizer.param_groups[0]['lr']}
        row.update({f'tr_{k}': v/max(len(dl_tr_f2),1)
                    for k,v in tr_losses.items()})
        row.update({f'val_{k}': v/max(len(dl_val_f2),1)
                    for k,v in val_losses.items()})
        history_f2.append(row)

        if ep % 25 == 0:
            frozen_str = 'T1_frozen' if ep < 150 else 'T1_free'
            print(f"  ep{ep:4d}/{EP_FASE2} "
                  f"val={val_total:.5f} "
                  f"best={best_val_global:.5f} (ep{best_epoch_f2}) "
                  f"lr={optimizer.param_groups[0]['lr']:.2e} "
                  f"[{frozen_str}]")

    t_total = (datetime.now() - t_inicio).total_seconds() / 60
    print(f"\n  Completado en {t_total:.1f} min | best_epoch={best_epoch_f2}")

    pd.DataFrame(history_f2).to_csv(
        DEFINITIVO_DIR / f'history_fase2_seed{SEED_FASE2}.csv', index=False)

    # ── Evaluación ─────────────────────────────────────────────────────────
    print(f"\n{'─'*60}")
    print(f"Evaluando sobre Test OOD ({len(idx_test_f2)} casos)...")
    print(f"{'─'*60}")

    model_f2.load_state_dict(torch.load(best_path, map_location=device))
    metrics_f2 = compute_metrics_ood(model_f2, dl_test_f2, scalers_f2)

    print(f"  seed {SEED_FASE2}: "
          f"MAC={metrics_f2['MAC_avg']:.4f} "
          f"T1={metrics_f2['T1_MAPE']:.2f}% "
          f"Vb={metrics_f2['Vb_MAPE']:.2f}% "
          f"drift={metrics_f2['drift_MAPE']:.2f}%")

    corridas.append({
        'seed'       : SEED_FASE2,
        'best_epoch' : best_epoch_f2,
        'tiempo_min' : round(t_total, 2),
        'n_test'     : int(len(idx_test_f2)),
        'metricas'   : {k: float(metrics_f2[k])
                        for k in ['MAC_avg','T1_MAPE',
                                  'Vb_MAPE','drift_MAPE']},
    })

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║              AGREGACIÓN media ± std + COMPARACIÓN SIMÉTRICA               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
METRICAS = ['MAC_avg', 'T1_MAPE', 'Vb_MAPE', 'drift_MAPE']
agg = {}
for k in METRICAS:
    vals = np.array([c['metricas'][k] for c in corridas], dtype=float)
    agg[k] = {'media': float(vals.mean()), 'std': float(vals.std(ddof=1)),
              'min': float(vals.min()), 'max': float(vals.max()),
              'valores': vals.tolist()}

print(f"\n{'='*70}")
print(f"FASE 2 — RESUMEN sobre {len(SEEDS_FASE2)} semillas (media ± std)")
print(f"{'='*70}")
print(f"  {'Metrica':<14} {'media':>10} {'std':>10} {'min':>10} {'max':>10}")
print(f"  {'-'*56}")
for k in METRICAS:
    a = agg[k]
    print(f"  {k:<14} {a['media']:>10.4f} {a['std']:>10.4f} "
          f"{a['min']:>10.4f} {a['max']:>10.4f}")

# ── Comparación simétrica Fase 1 (4 reps) vs Fase 2 (N semillas) ────────────
with open(TRIALS_DIR / vfile('all_results','json'), encoding='utf-8') as f:
    r_f1 = json.load(f)[str(TRIAL_FASE2)]

print(f"\n{'='*70}")
print(f"COMPARACION SIMÉTRICA — Fase 1 (4 reps) vs Fase 2 ({len(SEEDS_FASE2)} semillas)")
print(f"{'='*70}")
print(f"  {'Metrica':<12} {'Fase 1 (μ±σ)':>21} {'Fase 2 (μ±σ)':>21} {'Δ':>10} {'banda':>9}")
print(f"  {'-'*73}")
for k in METRICAS:
    m1, s1 = r_f1['media'][k], r_f1['std'][k]
    m2, s2 = agg[k]['media'], agg[k]['std']
    solapan = not (m1 + s1 < m2 - s2 or m2 + s2 < m1 - s1)
    if k == 'MAC_avg':
        delta = f"{(m2-m1)*100:+.3f}pp"
    else:
        delta = f"{((m1-m2)/m1)*100:+.1f}%{'↓' if m2<m1 else '↑'}"
    print(f"  {k:<12} {m1:>10.4f}±{s1:<10.4f} {m2:>10.4f}±{s2:<10.4f} "
          f"{delta:>10} {'solapan' if solapan else 'SEPARAD':>9}")

print(f"{'='*70}")
print("  'solapan'  → diferencia dentro del ruido entre semillas (no concluyente)")
print("  'SEPARAD'  → bandas μ±σ no se tocan → diferencia real")

# ── Guardar ─────────────────────────────────────────────────────────────────
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer,)):    return int(obj)
        if isinstance(obj, (np.floating,)):   return float(obj)
        if isinstance(obj, np.ndarray):       return obj.tolist()
        if isinstance(obj, (np.bool_, bool)): return bool(obj)
        return super().default(obj)

resultado_multi = {
    'trial'         : TRIAL_FASE2,
    'nombre'        : NOMBRE_FASE2,
    'seeds'         : SEEDS_FASE2,
    'ep_fase2'      : EP_FASE2,
    'n_train'       : int(len(idx_pool_A)),
    'arquitectura'  : 'PINNModal_v4b',
    'loss'          : 'compute_loss_v16',
    'curriculum'    : 'v20_staged_training',
    'h5_usado'      : H5_MODAL.name,
    'fase2_agg'     : agg,
    'fase2_corridas': corridas,
    'fase1_media'   : {k: r_f1['media'][k] for k in METRICAS},
    'fase1_std'     : {k: r_f1['std'][k]   for k in METRICAS},
}

out_json = DEFINITIVO_DIR / 'resultado_fase2_multisemilla.json'
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(resultado_multi, f, indent=4,
              ensure_ascii=False, cls=NpEncoder)

print(f"\nArchivos guardados en: {DEFINITIVO_DIR}")
print(f"  model_fase2_seed<semilla>.pt   ({len(SEEDS_FASE2)} archivos)")
print(f"  history_fase2_seed<semilla>.csv ({len(SEEDS_FASE2)} archivos)")
print(f"  resultado_fase2_multisemilla.json")

In [ ]:
import shutil
from pathlib import Path

DEF_DIR = Path(r'C:\Users\rodri\Documents\PINN\b2_proyecto\models\trials_31\V8\DEFINITIVO')

# Renombrar/copiar seed ganadora como modelo definitivo
shutil.copy(DEF_DIR / 'model_fase2_seed2026.pt',
            DEF_DIR / 'model_fase2_definitivo.pt')
shutil.copy(DEF_DIR / 'history_fase2_seed2026.csv',
            DEF_DIR / 'history_fase2_definitivo.csv')

print('✅ Modelo definitivo: seed 2026')
print(f'   MAC=0.9982  T1=9.35%  Vb=8.94%  drift=11.27%')
print(f'   Guardado en: {DEF_DIR}')


In [ ]:
# ============================================================
# CELDA 13.2 — FASE 2: AÑADIR SEMILLAS (acumula, no re-corre)
# Corre SOLO las semillas nuevas, las fusiona con las que ya
# están en resultado_fase2_multisemilla.json, y recalcula
# media±std sobre el total. NO repite las corridas previas.
# ============================================================

from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# DEFINITIVO_DIR y TRIALS_DIR vienen del panel (CELDA 0)

TRIAL_FASE2   = 16
NOMBRE_FASE2  = 'zona_1_3_vs_zona_2'
EP_FASE2      = 600

# ── Semillas NUEVAS a agregar (las ya corridas se detectan solas) ───────────
SEEDS_NUEVAS  = [314, 555, 808, 1001, 4242, 2718]   # ← edita esta lista

OUT_JSON      = DEFINITIVO_DIR / 'resultado_fase2_multisemilla.json'

variables_necesarias = ['X_clean', 'N_pisos_arr', 'COLS_BASE',
                        'T_r', 'Vb_x', 'Vb_y', 'drift_x', 'drift_y',
                        'build_model', 'compute_scalers',
                        'make_dataloaders', 'compute_metrics_ood',
                        'compute_loss', 'TRIALS_DEF',
                        'N_MODOS', 'N_MODOS_PHYS', 'N_MAX_PISOS',
                        'get_lambdas_and_active']
faltantes = [v for v in variables_necesarias if v not in globals()]
if faltantes:
    print(f"⚠️  Faltan: {faltantes}")
    raise SystemExit

# ── Cargar progreso previo ──────────────────────────────────────────────────
if OUT_JSON.exists():
    with open(OUT_JSON, encoding='utf-8') as f:
        prev = json.load(f)
    corridas = list(prev['fase2_corridas'])
    seeds_hechas = {c['seed'] for c in corridas}
    print(f"Cargadas {len(corridas)} corridas previas | semillas: {sorted(seeds_hechas)}")
else:
    corridas = []
    seeds_hechas = set()
    print("No hay JSON previo — se parte desde cero.")

# Solo correr las semillas que NO estén ya hechas
seeds_a_correr = [s for s in SEEDS_NUEVAS if s not in seeds_hechas]
ya_estaban     = [s for s in SEEDS_NUEVAS if s in seeds_hechas]
if ya_estaban:
    print(f"Omitidas (ya estaban): {ya_estaban}")
print(f"A correr ahora: {seeds_a_correr}")

# ── Datos (Pool A/B fijos) ──────────────────────────────────────────────────
trial_def  = next(t for t in TRIALS_DEF if t['id'] == TRIAL_FASE2)
idx_pool_A = np.where(trial_def['mask_A'])[0]
idx_pool_B = np.where(trial_def['mask_B'])[0]

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║                  BUCLE SOLO SOBRE SEMILLAS NUEVAS                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
for SEED_FASE2 in seeds_a_correr:

    torch.manual_seed(SEED_FASE2)
    np.random.seed(SEED_FASE2)

    rng_b       = np.random.default_rng(SEED_FASE2 + TRIAL_FASE2 * 200)
    idx_B_shuf  = rng_b.permutation(idx_pool_B)
    idx_val_f2  = idx_B_shuf[:400]
    idx_test_f2 = idx_B_shuf[400:]

    scalers_f2 = compute_scalers(idx_pool_A)
    dl_tr_f2, dl_val_f2, dl_test_f2 = make_dataloaders(
        idx_pool_A, idx_val_f2, idx_test_f2, scalers_f2)

    print(f"\n{'#'*70}")
    print(f"### SEMILLA {SEED_FASE2}  (nueva)")
    print(f"{'#'*70}")
    print(f"  Val : {len(idx_val_f2)} | Test : {len(idx_test_f2)}")

    model_f2  = build_model()
    best_path = DEFINITIVO_DIR / f'model_fase2_seed{SEED_FASE2}.pt'

    optimizer = torch.optim.Adam(model_f2.parameters(), lr=1e-3)
    scheduler = CosineAnnealingWarmRestarts(
        optimizer, T_0=200, T_mult=2, eta_min=1e-6)
    params_T1 = (list(model_f2.encoder_T1.parameters()) +
                 list(model_f2.head_T1.parameters()))

    history_f2      = []
    best_val_global = float('inf')
    best_epoch_f2   = 0
    etapa_actual    = -1
    stage_actual    = -1
    t_inicio        = datetime.now()

    for ep in range(EP_FASE2):
        lambdas, active = get_lambdas_and_active(ep)

        if ep < 150:
            if etapa_actual != 0:
                etapa_actual = 0
                for p in params_T1:
                    p.requires_grad = False
                print(f"\n  ── Etapa 0 (ep {ep}) encoder_T1 CONGELADO ──")
        else:
            if etapa_actual != 1:
                etapa_actual = 1
                for p in params_T1:
                    p.requires_grad = True
                print(f"\n  ── Etapa 1 (ep {ep}) encoder_T1 DESCONGELADO ──")

        stage_nuevo = {150:1, 200:2, 350:3, 500:4}.get(ep, stage_actual)
        if stage_nuevo != stage_actual and ep >= 150:
            stage_actual = stage_nuevo
            print(f"\n  ── Stage {stage_actual} (ep {ep}) "
                  f"T1λ={lambdas['T1']} eig={lambdas['eig']} "
                  f"lr={optimizer.param_groups[0]['lr']:.2e} ──")

        model_f2.train()
        tr_losses = {}
        for batch in dl_tr_f2:
            optimizer.zero_grad()
            out  = model_f2(batch['X'].to(device), batch['mask'].to(device))
            loss, ld = compute_loss(batch, out, lambdas, scalers_f2, active)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_f2.parameters(), 1.0)
            optimizer.step()
            for k, v in ld.items():
                tr_losses[k] = tr_losses.get(k, 0) + v

        model_f2.eval()
        val_losses = {}
        with torch.no_grad():
            for batch in dl_val_f2:
                out  = model_f2(batch['X'].to(device), batch['mask'].to(device))
                _, ld = compute_loss(batch, out, lambdas, scalers_f2, active)
                for k, v in ld.items():
                    val_losses[k] = val_losses.get(k, 0) + v

        val_total = sum(lambdas.get(k, 0) * v for k, v in val_losses.items())
        scheduler.step()

        if val_total < best_val_global and ep >= 250:
            best_val_global = val_total
            torch.save(model_f2.state_dict(), best_path)
            best_epoch_f2 = ep

        row = {'seed': SEED_FASE2, 'etapa': etapa_actual,
               'stage': stage_actual, 'epoch': ep,
               'val_total': val_total,
               'lr': optimizer.param_groups[0]['lr']}
        row.update({f'tr_{k}': v/max(len(dl_tr_f2),1) for k,v in tr_losses.items()})
        row.update({f'val_{k}': v/max(len(dl_val_f2),1) for k,v in val_losses.items()})
        history_f2.append(row)

        if ep % 25 == 0:
            frozen_str = 'T1_frozen' if ep < 150 else 'T1_free'
            print(f"  ep{ep:4d}/{EP_FASE2} val={val_total:.5f} "
                  f"best={best_val_global:.5f} (ep{best_epoch_f2}) "
                  f"lr={optimizer.param_groups[0]['lr']:.2e} [{frozen_str}]")

    t_total = (datetime.now() - t_inicio).total_seconds() / 60
    print(f"\n  Completado en {t_total:.1f} min | best_epoch={best_epoch_f2}")

    pd.DataFrame(history_f2).to_csv(
        DEFINITIVO_DIR / f'history_fase2_seed{SEED_FASE2}.csv', index=False)

    model_f2.load_state_dict(torch.load(best_path, map_location=device))
    metrics_f2 = compute_metrics_ood(model_f2, dl_test_f2, scalers_f2)
    print(f"  seed {SEED_FASE2}: MAC={metrics_f2['MAC_avg']:.4f} "
          f"T1={metrics_f2['T1_MAPE']:.2f}% Vb={metrics_f2['Vb_MAPE']:.2f}% "
          f"drift={metrics_f2['drift_MAPE']:.2f}%")

    corridas.append({
        'seed'       : SEED_FASE2,
        'best_epoch' : best_epoch_f2,
        'tiempo_min' : round(t_total, 2),
        'n_test'     : int(len(idx_test_f2)),
        'metricas'   : {k: float(metrics_f2[k])
                        for k in ['MAC_avg','T1_MAPE','Vb_MAPE','drift_MAPE']},
    })

    # ── Guardado INCREMENTAL tras cada semilla (a prueba de cortes) ─────────
    ckpt = {'trial': TRIAL_FASE2, 'nombre': NOMBRE_FASE2,
            'ep_fase2': EP_FASE2, 'seeds': [c['seed'] for c in corridas],
            'fase2_corridas': corridas, '_parcial': True}
    with open(OUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(ckpt, f, indent=4, ensure_ascii=False, default=float)
    print(f"  [checkpoint] JSON actualizado con {len(corridas)} corridas")

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║         AGREGACIÓN FINAL sobre TODAS las corridas (previas+nuevas)        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
METRICAS = ['MAC_avg', 'T1_MAPE', 'Vb_MAPE', 'drift_MAPE']

def agregar(lista):
    out = {}
    for k in METRICAS:
        v = np.array([c['metricas'][k] for c in lista], dtype=float)
        out[k] = {'media': float(v.mean()),
                  'std': float(v.std(ddof=1)) if len(v) > 1 else 0.0,
                  'min': float(v.min()), 'max': float(v.max()),
                  'n': int(len(v)), 'valores': v.tolist()}
    return out

agg = agregar(corridas)
n_tot = len(corridas)

print(f"\n{'='*70}")
print(f"FASE 2 — RESUMEN sobre {n_tot} semillas (TODAS)")
print(f"{'='*70}")
print(f"  {'Metrica':<14} {'media':>10} {'std':>10} {'min':>10} {'max':>10}")
print(f"  {'-'*56}")
for k in METRICAS:
    a = agg[k]
    print(f"  {k:<14} {a['media']:>10.4f} {a['std']:>10.4f} "
          f"{a['min']:>10.4f} {a['max']:>10.4f}")

# ── Detección de outliers por T1_MAPE (regla 1.5·IQR) ───────────────────────
t1 = np.array([c['metricas']['T1_MAPE'] for c in corridas])
q1, q3 = np.percentile(t1, [25, 75])
iqr = q3 - q1
hi  = q3 + 1.5 * iqr
out_mask = t1 > hi
print(f"\n  Detección outliers T1_MAPE (regla 1.5·IQR, umbral > {hi:.2f}):")
if out_mask.any():
    for c, isout in zip(corridas, out_mask):
        if isout:
            print(f"    seed {c['seed']:>5}: T1={c['metricas']['T1_MAPE']:.2f}% "
                  f"(best_epoch={c['best_epoch']}) ← OUTLIER")
    keep = [c for c, o in zip(corridas, out_mask) if not o]
    agg_clean = agregar(keep)
    print(f"\n  T1_MAPE SIN outliers ({len(keep)} corridas): "
          f"{agg_clean['T1_MAPE']['media']:.2f} ± {agg_clean['T1_MAPE']['std']:.2f}")
    print("  (Reportar AMBOS valores en la tesis; NO descartar el outlier sin justificar)")
else:
    print("    Ninguno. Distribución de T1_MAPE sin outliers por IQR.")

# ── Comparación simétrica con Fase 1 ────────────────────────────────────────
with open(TRIALS_DIR / vfile('all_results','json'), encoding='utf-8') as f:
    r_f1 = json.load(f)[str(TRIAL_FASE2)]

print(f"\n{'='*70}")
print(f"COMPARACION — Fase 1 (4 reps) vs Fase 2 ({n_tot} semillas)")
print(f"{'='*70}")
print(f"  {'Metrica':<12} {'Fase 1 (μ±σ)':>21} {'Fase 2 (μ±σ)':>21} {'Δ':>10} {'banda':>9}")
print(f"  {'-'*73}")
for k in METRICAS:
    m1, s1 = r_f1['media'][k], r_f1['std'][k]
    m2, s2 = agg[k]['media'], agg[k]['std']
    solapan = not (m1 + s1 < m2 - s2 or m2 + s2 < m1 - s1)
    if k == 'MAC_avg':
        delta = f"{(m2-m1)*100:+.3f}pp"
    else:
        delta = f"{((m1-m2)/m1)*100:+.1f}%{'↓' if m2<m1 else '↑'}"
    print(f"  {k:<12} {m1:>10.4f}±{s1:<10.4f} {m2:>10.4f}±{s2:<10.4f} "
          f"{delta:>10} {'solapan' if solapan else 'SEPARAD':>9}")
print(f"{'='*70}")

# ── Guardado final completo ─────────────────────────────────────────────────
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer,)):    return int(obj)
        if isinstance(obj, (np.floating,)):   return float(obj)
        if isinstance(obj, np.ndarray):       return obj.tolist()
        if isinstance(obj, (np.bool_, bool)): return bool(obj)
        return super().default(obj)

resultado_multi = {
    'trial'        : TRIAL_FASE2,
    'nombre'       : NOMBRE_FASE2,
    'seeds'        : [c['seed'] for c in corridas],
    'ep_fase2'     : EP_FASE2,
    'n_train'      : int(len(idx_pool_A)),
    'arquitectura' : 'PINNModal_v4b',
    'loss'         : 'compute_loss_v16',
    'curriculum'   : 'v20_staged_training',
    'h5_usado'     : H5_MODAL.name,
    'fase2_agg'    : agg,
    'fase2_corridas': corridas,
    'fase1_media'  : {k: r_f1['media'][k] for k in METRICAS},
    'fase1_std'    : {k: r_f1['std'][k]   for k in METRICAS},
}
with open(OUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(resultado_multi, f, indent=4, ensure_ascii=False, cls=NpEncoder)

print(f"\nGuardado final: {OUT_JSON}")
print(f"  Total acumulado: {n_tot} semillas → {[c['seed'] for c in corridas]}")

In [ ]:
# ============================================================
# CELDA — DIAGNÓSTICO DE CONFOUNDING DE LOS 31 TRIALS OOD
# Verifica que cada trial separe Pool A / Pool B SOLO por su
# factor nominal, y que las demás variables (en especial N_pisos)
# estén balanceadas. Muestra la figura inline y la guarda en 500 DPI.
# Rutas: DATA_DIR, TRIALS_DIR, vfile() vienen del panel (CELDA 0)
# ============================================================
%matplotlib inline
import h5py, numpy as np, pandas as pd, matplotlib.pyplot as plt

H5_PATH = sorted(DATA_DIR.glob('dataset_familyB_OPENSEES_v6_*.h5'))[-1]
print(f"HDF5        : {H5_PATH.name}")
print(f"Guardando en: {TRIALS_DIR}")

with h5py.File(H5_PATH, 'r') as f:
    Xc    = f['inputs/X'][:]
    feats = [s.decode() for s in f['inputs/input_features'][:]]
    Nv    = f['scalars/N_pisos'][:].astype(float)

def C(name): return Xc[:, feats.index(name)]
fc=C('fc_MPa'); h_s=C('h_story_m'); n_u=C('n_unid_lado'); L_m=C('L_mod_m')
t_n=C('t_muro_nucleo_m'); t_b=C('t_muro_borde_m'); t_m=C('t_muro_mid_m')
gk=C('gk_kN_m2'); Ln=C('L_nucleo_m'); Bn=C('B_nucleo_m'); pd_=C('prof_depto_m'); ac=C('ancho_corredor_m')
suelo=C('suelo'); zona=C('zona')                       # suelo:0=A,1=B,2=C,3=D ; zona:1,2,3
s_A,s_B,s_C,s_D = suelo==0,suelo==1,suelo==2,suelo==3
z_1,z_2,z_3 = zona==1,zona==2,zona==3
med = lambda v: float(np.median(v))
Ln_med,Bn_med,pd_med,tn_med,tb_med = map(med,[Ln,Bn,pd_,t_n,t_b])

# (id, etiqueta legible, mask_A, mask_B, factor_nominal)
TR = [
 (1,'N par vs impar',Nv%2==0,Nv%2!=0,'N_pisos'),
 (2,'N bajo vs alto (≤10 / ≥13)',Nv<=10,Nv>=13,'N_pisos'),
 (3,'N 6–9 vs 14–18',Nv<=9,Nv>=14,'N_pisos'),
 (4,'N 6–8 vs 15–18',Nv<=8,Nv>=15,'N_pisos'),
 (5,'gk 6.0–6.5 vs 7.0–7.5',gk<=6.5,gk>=7.0,'gk_kN_m2'),
 (6,'fc 25,35 vs 30,40',(fc==25)|(fc==35),(fc==30)|(fc==40),'fc_MPa'),
 (7,'t_núcleo fino vs grueso',t_n<=tn_med,t_n>tn_med,'t_muro_nucleo_m'),
 (8,'t_borde fino vs grueso',t_b<=tb_med,t_b>tb_med,'t_muro_borde_m'),
 (9,'gk bajo vs alto',gk<=6.5,gk>=7.0,'gk_kN_m2'),
 (10,'suelo A,B vs C,D',s_A|s_B,s_C|s_D,'suelo'),
 (11,'n_unid bajo vs alto',n_u<=3,n_u>=5,'n_unid_lado'),
 (12,'L_mod bajo vs alto',L_m<=3.4,L_m>=3.6,'L_mod_m'),
 (13,'L_núcleo corto vs largo',Ln<Ln_med,Ln>Ln_med,'L_nucleo_m'),
 (14,'B_núcleo pequeño vs grande',Bn<Bn_med,Bn>Bn_med,'B_nucleo_m'),
 (15,'prof_depto bajo vs alto',pd_<pd_med,pd_>pd_med,'prof_depto_m'),
 (16,'zona 1,3 vs zona 2',z_1|z_3,z_2,'zona'),
 (17,'suelo A,C vs B,D',s_A|s_C,s_B|s_D,'suelo'),
 (18,'t_núcleo p25 vs p75',t_n<=np.percentile(t_n,25),t_n>=np.percentile(t_n,75),'t_muro_nucleo_m'),
 (19,'zona 1 vs zona 3',z_1,z_3,'zona'),
 (20,'fc 25,30 vs 35,40',(fc==25)|(fc==30),(fc==35)|(fc==40),'fc_MPa'),
 (21,'n_unid pequeño vs grande',n_u<=2,n_u>=4,'n_unid_lado'),
 (22,'t_borde p25 vs p75',t_b<=np.percentile(t_b,25),t_b>=np.percentile(t_b,75),'t_muro_borde_m'),
 (23,'h_story bajo vs alto',h_s<=np.percentile(h_s,33),h_s>=np.percentile(h_s,67),'h_story_m'),
 (24,'t_mid p25 vs p75',t_m<=np.percentile(t_m,25),t_m>=np.percentile(t_m,75),'t_muro_mid_m'),
 (25,'corredor angosto vs ancho',ac<=np.percentile(ac,33),ac>=np.percentile(ac,67),'ancho_corredor_m'),
 (26,'t_núcleo 0.18–0.20 vs 0.28–0.30',t_n<=0.20,t_n>=0.28,'t_muro_nucleo_m'),
 (27,'n_unid 2,3 vs 4,5',(n_u==2)|(n_u==3),(n_u==4)|(n_u==5),'n_unid_lado'),
 (28,'corredor p33 vs p67',ac<=np.percentile(ac,33),ac>=np.percentile(ac,67),'ancho_corredor_m'),
 (29,'fc alto vs gk alto',fc>=35,gk>=7.0,'MIXTO(fc/gk)'),
]

VARS = {'N_pisos':Nv,'n_unid_lado':n_u,'L_mod_m':L_m,'prof_depto_m':pd_,
        'ancho_corredor_m':ac,'L_nucleo_m':Ln,'B_nucleo_m':Bn,'h_story_m':h_s,
        'fc_MPa':fc,'gk_kN_m2':gk,'t_muro_nucleo_m':t_n,'t_muro_borde_m':t_b,
        't_muro_mid_m':t_m,'suelo':suelo,'zona':zona}
SHORT = {'t_muro_nucleo_m':'t_núcleo','t_muro_borde_m':'t_borde','t_muro_mid_m':'t_mid',
         'B_nucleo_m':'B_núcleo','L_nucleo_m':'L_núcleo','n_unid_lado':'n_unid','L_mod_m':'L_mod',
         'prof_depto_m':'prof_depto','ancho_corredor_m':'corredor','h_story_m':'h_story',
         'gk_kN_m2':'gk','fc_MPa':'fc','N_pisos':'N','suelo':'suelo','zona':'zona'}

def cohen_d(a, b):
    """Tamaño de efecto |μB-μA|/σ entre dos grupos."""
    s = max(a.std(), b.std(), 1e-8)
    return (b.mean() - a.mean()) / s
def fmt_d(x):
    if x is None or (isinstance(x,float) and np.isnan(x)): return '—'
    return '≫ (total)' if abs(x)>50 else f'{abs(x):.2f}'

# ── Tabla exhaustiva (trial × variable) + resumen ───────────────────────────
filas, resumen = [], []
for tid, nom, A, B, fac in TR:
    nA, nB = int(A.sum()), int(B.sum()); inter = int((A & B).sum())
    fila = {'id':tid,'nombre':nom,'factor':fac,'n_A':nA,'n_B':nB,'A∩B':inter}
    for vn, v in VARS.items():
        fila[f'd_{vn}'] = round(float(cohen_d(v[A], v[B])), 3)
    filas.append(fila)
    fcol = fac if fac in VARS else None
    d_fac = abs(cohen_d(VARS[fcol][A], VARS[fcol][B])) if fcol else None
    colat = {vn:abs(cohen_d(v[A],v[B])) for vn,v in VARS.items() if vn!=fcol}
    topv = max(colat, key=colat.get); topval = colat[topv]
    dN = abs(cohen_d(Nv[A],Nv[B])) if fac!='N_pisos' else np.nan
    estado = 'FUGA' if inter>0 else ('CONF?' if topval>0.3 else 'OK')
    resumen.append({'id':tid,'nombre':nom,'nA':nA,'nB':nB,'A∩B':inter,
                    'd_fac':d_fac,'dN':dN,'top':f'{SHORT.get(topv,topv)} ({topval:.2f})','estado':estado})
df      = pd.DataFrame(filas)
df_res  = pd.DataFrame(resumen)

# ── CSVs (en la carpeta de la versión activa) ───────────────────────────────
df.to_csv(    TRIALS_DIR / vfile('confounding_full',    'csv'), index=False)
df_res.to_csv(TRIALS_DIR / vfile('confounding_resumen', 'csv'), index=False)

# ── Figura-tabla 500 DPI ────────────────────────────────────────────────────
plt.rcParams['font.family']='DejaVu Sans'
fig,ax=plt.subplots(figsize=(14.5,12)); ax.axis('off')
cols=['ID','Trial (factor nominal)','n_A','n_B','d_factor (σ)','d_N colat. (σ)','Top colateral (σ)','Estado']
cell=[]; colors=[]
for _,r in df_res.iterrows():
    dN_str = '(es el factor)' if pd.isna(r['dN']) else f"{r['dN']:.3f}"
    cell.append([f"T{int(r['id']):02d}", r['nombre'], f"{r['nA']}", f"{r['nB']}",
                 fmt_d(r['d_fac']), dN_str, r['top'], r['estado']])
    rc = '#f8d7da' if r['estado']=='FUGA' else ('#fff3cd' if r['estado']=='CONF?' else '#d4edda')
    colors.append([rc]*len(cols))
tab=ax.table(cellText=cell,colLabels=cols,cellColours=colors,cellLoc='center',loc='center',
             colWidths=[0.05,0.30,0.07,0.07,0.12,0.13,0.18,0.08])
tab.auto_set_font_size(False); tab.set_fontsize(9); tab.scale(1,1.5)
for j in range(len(cols)):
    c=tab[0,j]; c.set_facecolor('#2c3e50'); c.set_text_props(color='white',fontweight='bold')
for i in range(1,len(cell)+1):
    tab[i,1].set_text_props(ha='left')
ax.set_title('Diagnóstico de confounding — 31 trials OOD (dataset v6, N = 10 000)\n'
             'Tamaño de efecto (d de Cohen) entre Pool A y Pool B',
             fontsize=14,fontweight='bold',pad=20)
fig.text(0.5,0.055,
  'd_factor: separación A/B por el factor nominal (valor alto ⇒ partición efectiva).   '
  'd_N colat.: separación de N_pisos cuando N no es el factor (≈ 0 ⇒ sin confounding con N).\n'
  'Top colateral: variable no declarada con mayor separación.   '
  'Verde = OK  ·  Amarillo = colateral > 0.3σ  ·  Rojo = fuga (A∩B > 0).   '
  '«≫ (total)» = separación categórica completa (σ → 0).',
  ha='center',fontsize=8.5,style='italic',color='#444')

out_png = TRIALS_DIR / vfile('confounding_trials','png')
plt.savefig(out_png, dpi=500, bbox_inches='tight', facecolor='white')
plt.show()        # ← muestra la figura inline en el output (con %matplotlib inline no hay warning)

print(f"\n=== RESUMEN ===")
print(df_res[['id','nombre','estado']].to_string(index=False))
print(f"\nGuardado en {TRIALS_DIR}:")
print(f"  {vfile('confounding_trials','png')}   (figura 500 DPI)")
print(f"  {vfile('confounding_full','csv')}     (matriz completa trial × variable)")
print(f"  {vfile('confounding_resumen','csv')}  (resumen interpretativo)")